# DATA-DRIVEN 1D MECHANICAL EARTH MODEL
## INCREMENT 5 / 5.1 / 5.1.1 — FORMATION-TOP INGESTION, SOURCE RECONCILIATION, AND SURVEY-CORRECTED STRATIGRAPHIC DEPTH FRAMEWORK

**Author:** Mikael Elgo
**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D MEM. This notebook is a portfolio and educational workflow. It is **not** calibrated for operational drilling, well design, casing design, or real-world mud-weight decisions.

**Locked foundation:** Increment 4.1.2 (`p2mem` 0.4.1.1) — LAS ingestion, deviation-survey ingestion, minimum-curvature trajectory validation, MD-to-TVD/TVDSS depth mapping, checkshot ingestion, duplicate-tie conditioning, and the time-depth framework for the four approved wells — has passed independent technical review (482 combined tests target for this increment as patched to 5.1.1 — 476 prior to the Increment 5.1.1 corrective patch's added regression tests, and 439 prior to the Increment 5.1 corrective patch's; the actual count is read from Step 9's own output, never assumed) and is treated as **LOCKED**. `p2mem/units.py`, `p2mem/models.py`, `p2mem/deviation_models.py`, `p2mem/trajectory.py`, `p2mem/depth_mapping.py`, `p2mem/checkshot_models.py`, `p2mem/time_depth.py`, every file under `p2mem/io/` from prior increments, and every prior `config/*.yml` are **not modified** in this notebook. This increment builds strictly on that foundation: it never re-parses, re-derives, or replaces the locked `petrel_source_trace` survey trajectory, the locked canonical LAS arrays, or the locked checkshot/time-depth layer.

> **Increment 5.1 corrective patch applied.** After an independent technical/software-QA audit, four findings were corrected WITHOUT beginning Increment 6 and WITHOUT changing any previously verified real Poseidon 2/Boreas 1 formation-top result: (1) `reconcile_formation_top_sources` now correctly rejects two disjoint, non-empty marker sets as a fatal `NO_COMMON_MARKERS` ERROR (it previously tested the wrong condition — the UNION of both sources' names being empty — so two entirely unrelated marker sets were silently accepted); (2) `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path`, so a failure originating from the "selected readable" file can no longer leak that file's own absolute path into any exported CSV/JSON field (previously only the HRS path was ever recorded, regardless of which file actually failed); (3) `reconcile_formation_top_sources` now fully validates its numeric-array and `mdrt_agreement_tolerance_m` inputs (finiteness, non-negativity, dimensionality, length consistency, tolerance type/value) before any comparison, with deliberate typed exceptions rather than an incidental `IndexError`/`TypeError`; (4) `INCREMENT_05_MANIFEST.md`'s claim that no absolute path exists ANYWHERE in the package was overbroad (synthetic tests intentionally contain fake absolute-path strings) — the corrected, narrowly scoped claim is now stated in `INCREMENT_05_1_MANIFEST.md`. See `INCREMENT_05_1_MANIFEST.md` for the full audit and re-verification record.
>
> **Increment 5.1.1 corrective patch applied.** After a further independent technical/software-QA audit, one remaining reconciliation edge case and three documentation-accuracy gaps were corrected WITHOUT beginning Increment 6 and WITHOUT changing any scientific result, tolerance, depth-mapping method, or formation-top contract: (1) Increment 5.1's fix still silently accepted the one remaining zero-common-markers configuration — exactly one source entirely empty and the other non-empty — because its condition required BOTH sources to be non-empty before checking their intersection; `reconcile_formation_top_sources` now uses the single, strictly correct check `if not common_markers` (the canonical-name intersection), which is empty in every zero-common-markers case and never empty whenever at least one marker is genuinely shared; (2) `INCREMENT_05_1_MANIFEST.md` stated the real Increment 5 baseline ZIP's SHA-256 "matches" a governing-instruction-supplied hash that was actually a two-character truncation — a mathematical impossibility, now stated transparently as a documentation typo; (3) `INCREMENT_05_1_MANIFEST.md` Section 7 incorrectly stated `outputs/` is excluded from the delivered ZIP, when it is in fact packaged (as in every prior increment) — corrected; (4) the same manifest incorrectly stated no `*.las` file exists in the package, when small intentional synthetic LAS/checkshot/deviation/top fixtures are packaged under `tests/fixtures/` for portable testing — corrected, with fictional fixtures explicitly distinguished from excluded real/private source files. See `INCREMENT_05_1_1_MANIFEST.md` for the full audit and re-verification record.

### Technical Objective

Add a validated formation-top ingestion, HRS-versus-selected-readable source RECONCILIATION, and survey-corrected stratigraphic depth layer on top of the locked foundation, for exactly FOUR approved formation-top files (two independently supplied representations for each of two wells):

1. Strict, auditable, contract-driven ingestion of each admitted formation-top file, with an explicit per-file contract (`config/formation_top_contracts.yml`) keyed by exact filename and verified SHA-256 — raw files are never rewritten, renamed, or "cleaned".
2. Preservation of every raw marker name and raw source depth from BOTH file representations — nothing is ever silently overwritten, sorted, deduplicated, or repaired.
3. Explicit RECONCILIATION between each well's "HRS" file (`Top_Name`/`MDRT_m` only, no well name in the file body) and its "selected readable" file (`TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE`, `#`-comment headers, a dashed separator line, deliberately parsed rather than treated as data) — markers are matched by exact name, whitespace-normalized name, or an explicit human-authored alias contract; NO fuzzy/similarity matching of any kind. MDRT is cross-checked between the two sources within a documented 0.005 m tolerance; a disagreement beyond that tolerance is registered and excludes the marker from mapping, never silently resolved by picking one file.
4. Mapping of each well's reconciled MDRT through the LOCKED Increment 3.1.1 `petrel_source_trace` survey trajectory, using the existing, unmodified `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` (per marker, so one out-of-coverage marker never blocks the rest of the well) — computing survey-derived TVD and a corrected TVDSS, and rejecting (never extrapolating) any marker outside survey MD coverage.
5. An explicit, unambiguous source-versus-survey TVDSS residual comparison (`TVDSS_residual_source_minus_survey_m = TVDSS_source_m - TVDSS_survey_corrected_m`) for every mapped marker.
6. Formation-top availability/provenance records — Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as `NOT_AVAILABLE`, never substituted or depth-correlated from another well.

**Explicitly NOT implemented in this increment:** gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic properties, rock strength, horizontal stresses, or wellbore-stability analysis. Those belong to later, explicitly gated increments. **Increment 6 has not been started.**

### Approved Inputs and Well Roles

Only four formation-top files are admitted, by exact filename and independently verified SHA-256 (never a same-well file under a different name, and never Poseidon 1, Kronos 1, Torosa 1, or any other well's tops):

| File | Project well | Representation | Identity evidence |
|---|---|---|---|
| `Poseidon_2_HRS_tops_no_wellname_MDRT.txt` | Poseidon 2 | HRS (`Top_Name`/`MDRT_m` only) | **inferred_unverified** — no well name in the file body; filename-only association |
| `Poseidon_2_selected_well_tops.txt` | Poseidon 2 | selected readable (`TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE`) | **inferred_unverified** — well name only in a project-supplied `#` comment |
| `Boreas_1_HRS_tops_no_wellname_MDRT.txt` | Boreas 1 | HRS | **inferred_unverified** — filename-only association |
| `Boreas_1_selected_well_tops.txt` | Boreas 1 | selected readable | **inferred_unverified** — project-supplied comment only |

**Poseidon North 1 and Proteus 1ST2 have no approved formation-top file.** This is recorded as `formation_top_availability: NOT_AVAILABLE` — a factual data gap, never an ingestion failure, and never filled by substituting another well's tops or correlating them by depth alone.

### Theory and Physical Basis

**MDRT ≡ survey MD.** Both approved wells' real deviation-survey file headers state "MD AND TVD ARE REFERENCED (=0) AT WELL DATUM [RT, Rotary Table] AND INCREASE DOWNWARDS" — the IDENTICAL convention as each formation-top file's own `MDRT_m`/`MDRT_M` column. No datum shift or unit conversion is required before mapping a marker's MDRT through the locked survey trajectory, and the locked `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` function is reused UNMODIFIED for this purpose — this increment never reimplements minimum curvature or the TVD/TVDSS formula.

**Two source representations, reconciled — never silently preferred.** Every marker is looked up by canonical name in BOTH the HRS file and the selected-readable file. A marker present in only one source is recorded as `NOT_COMPARABLE` (never treated as if the other source agreed). A marker present in both, with MDRT agreeing within 0.005 m, is mapped using that agreed value (`mdrt_authority_basis = "hrs_and_readable_agree"`). A marker whose two sources DISAGREE beyond tolerance is registered as a `MISMATCH` and excluded from depth mapping entirely (`mdrt_authority_basis = "disagreement_unresolved"`, `mapping_status = "not_mapped_mdrt_unresolved"`) — this project never silently picks one file's value over the other.

**Supplied TVDSS is preserved, never treated as corrected.** The selected-readable file's own `TVDSS_M` column is preserved exactly as `TVDSS_source_m`, used ONLY for the residual/QC comparison against the survey-corrected value. The project's corrected stratigraphic depth is always `TVDSS_survey_corrected_m`, computed independently from the locked survey trajectory.

> **QUALITY-CONTROL NOTE:** this notebook reports the source-versus-survey TVDSS residual as an OBSERVED, diagnostic finding per well — Poseidon 2's residual pattern is disclosed as a confirmed vertical-well-assumption depth-reference DEFECT (not merely "an offset"), while Boreas 1's small residual is disclosed as evidence the file is survey-consistent, NOT corrected. Evidence is applied per file, per well — never by analogy from one well's finding to the other's. The Poseidon 2 – Boreas 1 marker-depth panel (Step 16, Fig 3) is explicitly labelled as a depth comparison only, never a geological correlation or lithology interpretation.

### Input Data and Contracts

#### Step 1 — Mount Google Drive

**Technical objective:** re-attach the persistent project folder from prior increments.

**Inputs/outputs:** no project inputs are read here; this only establishes the `/content/drive` mount point.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#### Step 2 — Verify the Increment 4.1.2 foundation is present and locked

**Technical objective:** confirm that the locked LAS-ingestion, deviation-survey, and checkshot/time-depth layers already exist in this Drive project folder, BEFORE this notebook adds anything on top. Increment 5 is explicitly instructed not to modify these files unless an actual blocking defect is demonstrated — none is claimed here.

**Failure behavior:** raises `RuntimeError` naming exactly which expected file is missing. This is a hard gate, not a warning.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/Poseidon_1D_MEM"

_required_locked_files = [
    "pyproject.toml",
    os.path.join("p2mem", "__init__.py"),
    os.path.join("p2mem", "units.py"),
    os.path.join("p2mem", "models.py"),
    os.path.join("p2mem", "deviation_models.py"),
    os.path.join("p2mem", "trajectory.py"),
    os.path.join("p2mem", "depth_mapping.py"),
    os.path.join("p2mem", "checkshot_models.py"),
    os.path.join("p2mem", "time_depth.py"),
    os.path.join("p2mem", "io", "las.py"),
    os.path.join("p2mem", "io", "inventory.py"),
    os.path.join("p2mem", "io", "deviation.py"),
    os.path.join("p2mem", "io", "deviation_inventory.py"),
    os.path.join("p2mem", "io", "checkshot.py"),
    os.path.join("p2mem", "io", "checkshot_inventory.py"),
    os.path.join("config", "las_curve_contracts.yml"),
    os.path.join("config", "deviation_survey_contracts.yml"),
    os.path.join("config", "checkshot_contracts.yml"),
]
_missing = [f for f in _required_locked_files if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
if _missing:
    raise RuntimeError(
        "Locked Increment 4.1.2 foundation is missing file(s): "
        + ", ".join(_missing)
        + ". Run the Increment 4.1.2 notebook first; Increment 5 does not "
        "reconstruct the locked foundation from memory."
    )
print("Locked Increment 4.1.2 foundation confirmed present:")
for f in _required_locked_files:
    print("  -", f)


#### Step 2b — Enter the project root before any relative file write

**Increment 3.1.1 lesson applied from the start:** a real Google Colab `Run all` from a fresh runtime previously exposed a working-directory execution-order defect — Colab starts in `/content`, not `PROJECT_ROOT`, and every `%%writefile` cell below writes a RELATIVE path. This cell performs the `chdir` immediately after `PROJECT_ROOT` is defined and the locked foundation is confirmed present (Step 2), and strictly before the first relative `%%writefile` cell (Step 6) — never deferred to a later `%cd` cell.

**Failure behavior:** raises `RuntimeError` if the working directory does not actually resolve to `PROJECT_ROOT`, or if `p2mem` is not a directory there. Never silently creates a substitute directory.

In [ ]:
os.chdir(PROJECT_ROOT)

if Path.cwd().resolve() != Path(PROJECT_ROOT).resolve():
    raise RuntimeError(
        f"Failed to enter the project root. "
        f"Expected {PROJECT_ROOT}, actual working directory: {Path.cwd()}"
    )

if not Path("p2mem").is_dir():
    raise RuntimeError(
        f"Required p2mem directory is missing under {PROJECT_ROOT}. "
        "Extract the approved Increment package before running this notebook."
    )

print("Working directory confirmed:", Path.cwd())


#### Step 3 — Create the Increment 5 directory additions

**Technical objective:** create the new directories this increment needs, without touching any existing directory.

**Expected result:** `data/raw/tops/`, `outputs/05_formation_tops/` and `outputs/05_formation_tops/figures/` exist.

In [ ]:
for d in ["data/raw/tops", "outputs/05_formation_tops", "outputs/05_formation_tops/figures"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print("Increment 5 directories ready.")


#### Step 4 — Verify the four approved formation-top filenames are present (exact names only), and their SHA-256

**Technical objective:** require the exact approved filenames under `data/raw/tops/`, and stop cleanly, naming every missing file, if any are absent. No substitute file (Poseidon 1, Kronos 1, Torosa 1, or any other well's tops) is ever accepted in place of these four, and Poseidon North 1 / Proteus 1ST2 are never given a substitute formation-top file.

**Assumptions:** the user has uploaded these four files to `/content/drive/MyDrive/Poseidon_1D_MEM/data/raw/tops/` before running this cell — this notebook cannot fetch them itself, they are private project inputs.

**Failure behavior:** raises `RuntimeError` listing every missing filename by its exact expected name.

In [ ]:
import hashlib

TOPS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "tops")
TOP_FILES = {
    "Poseidon_2": (
        "Poseidon_2_HRS_tops_no_wellname_MDRT.txt",
        "Poseidon_2_selected_well_tops.txt",
    ),
    "Boreas_1": (
        "Boreas_1_HRS_tops_no_wellname_MDRT.txt",
        "Boreas_1_selected_well_tops.txt",
    ),
}

_all_expected_files = [fn for pair in TOP_FILES.values() for fn in pair]
_missing_top = [fn for fn in _all_expected_files if not os.path.exists(os.path.join(TOPS_DIR, fn))]
if _missing_top:
    raise RuntimeError(
        "Missing required formation-top file(s) under "
        f"{TOPS_DIR}: {_missing_top}. Increment 5 requires exactly these four "
        "approved formation-top files, under these exact names - no other "
        "file is accepted as a substitute, and Poseidon North 1 / Proteus "
        "1ST2 have no approved formation-top file at all (recorded as "
        "NOT_AVAILABLE below)."
    )

print("All four approved formation-top files found. SHA-256:")
for key, (hrs_fn, readable_fn) in TOP_FILES.items():
    for fn in (hrs_fn, readable_fn):
        p = os.path.join(TOPS_DIR, fn)
        print(f"  {key} ({fn}): {hashlib.sha256(open(p, 'rb').read()).hexdigest()}")


#### Step 5 — Install dependencies

**Technical objective:** install exactly the packages this increment's code and notebook display/plotting cells need. No new runtime dependency is added — `p2mem` itself still depends only on NumPy and PyYAML (see `pyproject.toml`); `pandas`/`matplotlib` remain notebook-only, unchanged from prior increments.

In [ ]:
!pip install -q numpy pyyaml pytest pandas matplotlib


### Input Data and Contracts

#### Step 6 — Write the Increment 5 package files

**Technical objective:** write every new/updated source file for this increment, verbatim from the tested files on disk (this build script never hand-retypes code into notebook cells).

**New modules:** `p2mem/top_models.py` (typed dataclasses), `p2mem/io/tops.py` (formation-top parser, per-file contract resolver, HRS-versus-readable reconciliation, survey mapping), `p2mem/io/tops_inventory.py` (deterministic output-table builders).

**Updated (version/documentation only — package version bumped to 0.5.2 for the Increment 5.1.1 corrective patch; 0.5.1 was the Increment 5.1 corrective-patch version):** `pyproject.toml`, `p2mem/__init__.py`, `README.md`. Every locked file from prior increments is intentionally NOT rewritten here.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.5.2"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 / 2.1.1 are corrective patches to Increment 2, applied
  after independent technical audits, WITHOUT changing scope or the
  underlying LAS-parsing/curve-resolution architecture (which both audits
  found sound). 2.1 corrected: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. 2.1.1 corrected a packaging-only gap
  (three notebook ``%%writefile`` cells that had drifted from their
  packaged source files). See ``INCREMENT_02_v2.1_MANIFEST.md`` and
  ``INCREMENT_02_v2.1.1_MANIFEST.md`` for the full audits and
  corrected-file checksums.
* Increment 3 adds Petrel deviation-survey ingestion with explicit
  per-file contracts (``p2mem.io.deviation``), a standard minimum-
  curvature trajectory engine with a numerically stable ratio-factor
  limit (``p2mem.trajectory``), an explicit MD-referenced/TVD-referenced/
  TVDSS depth-reference framework and MD-to-TVD/TVDSS interpolation with
  no silent extrapolation (``p2mem.depth_mapping``), and typed dataclasses
  for all of the above (``p2mem.deviation_models``) - for the same four
  approved wells. It independently reproduces the Petrel-supplied
  trajectory to millimetre scale for three of the four wells and
  discloses (rather than resolves) a real, larger trajectory-
  reconstruction discrepancy found in Proteus 1ST2's deeper section - see
  ``INCREMENT_03_MANIFEST.md``. It performs deviation-survey ingestion,
  trajectory validation, and depth mapping ONLY - no checkshot
  processing, formation-top correction, petrophysical interpretation, or
  any later-phase geomechanical calculation.
* Increment 3.1 is a corrective patch to Increment 3, applied after an
  independent technical audit, WITHOUT changing scope, equations, real
  well data, or the locked LAS/Increment-2.1.1 foundation. It corrected
  four defects: (1) the four deviation-survey source filenames are the
  exact, literal names as they exist in Google Drive, which contain
  spaces (e.g. ``"Poseidon 2_dev.txt"``) - Increment 3 had incorrectly
  substituted underscores in the contract keys, notebook mapping, and
  tests, which would have failed to resolve against the real files;
  internal well keys (e.g. ``Poseidon_2``) remain underscored and are
  unaffected; (2) every exported CSV/JSON/manifest field is now
  guaranteed to carry a basename only, never a full environment-dependent
  build path (runtime-only diagnostic objects may still retain one);
  (3) the previously undisclosed inference that the supplied ``DLS``
  column is normalized as degrees per 30 metres is now explicitly flagged
  with a new, independently per-file-verified
  ``DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`` WARNING (mirroring the
  pre-existing MD-unit-inference warning); (4) the dogleg angle between
  successive stations is now computed with a numerically stable
  ``arctan2(||cross||, dot)`` vector formulation (``p2mem.trajectory``)
  instead of ``arccos``, which was ill-conditioned near a zero dogleg and
  previously reported a spurious ~1e-6-degree value for two stations with
  identical inclination/azimuth. The real four-well data, station counts,
  tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are
  all unchanged by this patch. See ``INCREMENT_03_1_MANIFEST.md`` for the
  full audit and re-verification record.

* Increment 4 adds checkshot (velocity survey) ingestion with explicit
  per-file contracts (``p2mem.io.checkshot``, ``config/checkshot_
  contracts.yml``), typed checkshot dataclasses (``p2mem.checkshot_
  models``), deterministic checkshot inventory/QC-table builders
  (``p2mem.io.checkshot_inventory``), and a numerical time-depth layer
  (``p2mem.time_depth``): duplicate-tie detection/conditioning, average/
  interval velocity diagnostics, checkshot-vs-locked-survey depth-
  reference comparison, forward/inverse piecewise-linear time-depth
  interpolation with explicit coverage masking (no extrapolation), LAS
  MD-to-checkshot-time mapping within validated checkshot coverage only,
  and a Poseidon-2-only sonic-checkshot drift diagnostic (trapezoidal
  integration of sonic slowness vs. the checkshot-interpolated OWT
  increment over the same MD/Depth interval). Three checkshot files are
  admitted: ``Poseidon2-Checkshot.txt`` (Poseidon 2 - the ONLY checkshot
  approved to define a primary time-depth relationship),
  ``Boreas1-Checkshot.txt`` and ``Proteus1-Checkshot.txt`` (Boreas 1 and
  Proteus 1ST2 - supporting QC data only, newly admitted in this
  increment, never transferred into Poseidon 2 as a substitute time-depth
  model). Proteus 1ST2's association with ``Proteus1-Checkshot.txt`` is
  explicitly disclosed as inferred/unverified (no embedded well
  identifier). Poseidon North 1 has no approved checkshot file
  (``checkshot_availability: NOT_AVAILABLE`` - a factual data gap, not an
  ingestion failure). This increment reuses the LOCKED Increment 1
  ``owt_to_twt``/``twt_to_owt`` unit functions and the LOCKED Increment
  3/3.1.1 ``petrel_source_trace`` survey trajectory unchanged; it performs
  checkshot QC and time-depth framework work ONLY - no formation-top
  correction, lithology interpretation, density modelling, pore-pressure
  prediction, elastic properties, rock strength, stress modelling, or
  wellbore-stability analysis. See ``INCREMENT_04_MANIFEST.md`` for the
  full technical detail and independently recomputed statistics. NOTE:
  ``INCREMENT_04_MANIFEST.md`` contained one identified defect, corrected
  by Increment 4.1 below - do not rely on its original, uncorrected
  statement that TVDSS is strictly increasing after Depth-tie conditioning
  for all three wells.

* Increment 4.1 is a narrowly scoped corrective patch to Increment 4,
  applied after an independent technical/numerical-method audit, WITHOUT
  beginning Increment 5 or any formation-top/petrophysics/pore-pressure/
  mechanical-properties/stress/wellbore-stability work. It corrected two
  defects and hardened two numerical contracts: (1) ``tvdss_to_owt``/
  ``owt_to_tvdss`` (and their TWT equivalents) previously resolved a
  repeated (tied) value on the axis being inverted by silently keeping
  whichever tied row appeared first in the Depth-conditioned table and
  discarding the other (``p2mem.time_depth._build_strictly_increasing_
  table``, REMOVED) - an ORDER-DEPENDENT tie-break with no audit trail
  beyond a bare count. This is replaced by ``build_axis_conditioned_
  lookup_table``/``build_axis_conditioned_tables_for_well``: an explicit,
  ORDER-INVARIANT policy that groups every tied value by exact equality
  regardless of parse order, registers every tied row in a new typed
  audit register (``AxisTimeDepthTieRegisterEntry`` /
  ``checkshot_time_axis_tie_register.csv`` - separate from, and never
  confused with, the pre-existing Depth-axis ``DuplicateTieRegisterEntry``
  register), and uses the tied group's dependent-value MEDIAN as the
  conditioned representative (order-invariant; disclosed as reducing to
  the arithmetic mean for the size-2 groups observed in this project's
  real data). A genuine reversal (not a tie) in the axis being inverted
  raises ``TimeDepthError`` rather than being sorted, discarded, or forced
  monotonic. (2) ``INCREMENT_04_MANIFEST.md``'s statement that TVDSS is
  strictly increasing after Depth-tie conditioning for all three wells was
  INCORRECT - independently reproduced counts (Poseidon 2: two TVDSS-axis
  and two OWT-axis ties; Boreas 1: one TVDSS-axis tie, zero OWT-axis ties;
  Proteus 1ST2: none of either) are now documented in
  ``INCREMENT_04_1_MANIFEST.md`` and reflected in this module's own
  docstrings. (3) ``trapezoidal_integrate`` and (4) ``compute_sonic_
  checkshot_drift`` are hardened to validate their numerical
  preconditions (finite, one-dimensional, equal-length, strictly
  increasing MD/x where required) rather than silently integrating
  invalid input - most notably, a decreasing or duplicate MD run can no
  longer silently produce a physically invalid NEGATIVE transit time; it
  now raises ``TimeDepthError``. None of this hardening changes the
  already-verified real Poseidon 2 sonic-drift result, which is
  bit-for-bit unchanged. See ``INCREMENT_04_1_MANIFEST.md`` for the full
  audit, corrected statistics, and re-verification record.

* Increment 4.1.1 is a narrowly scoped numerical-validation corrective
  patch to Increment 4.1, applied after an independent numerical-method/
  software-QA audit, WITHOUT beginning Increment 5 or any formation-top/
  petrophysics/pore-pressure/mechanical-properties/stress/wellbore-
  stability work. It corrected four blocking defects and one input-safety
  gap, none of which altered any previously verified REAL Poseidon
  2/Boreas 1/Proteus 1ST2 result: (1) ``build_axis_conditioned_lookup_
  table`` grouped ALL occurrences of an identical axis value together
  GLOBALLY before checking for a reversal, so a reversal that returned to
  an already-seen value (e.g. ``[100.0, 200.0, 100.0]``) was silently
  hidden rather than raising ``TimeDepthError`` - it now evaluates the
  ORIGINAL, ungrouped sequence's successive differences for negativity
  BEFORE any grouping is attempted, which is provably equivalent to the
  4.1 behavior for every legitimate adjacent tie and strictly stronger
  against a non-adjacent reversal. (2) ``compute_sonic_checkshot_drift``/
  ``find_longest_finite_positive_run`` validated MD monotonicity only
  within the selected finite-positive-VP run, so a decreasing or
  duplicate MD value outside that run (e.g. at a NaN-VP station) could
  pass silently - the COMPLETE canonical ``md_m`` array is now required
  finite and strictly increasing before run-selection (the real Poseidon
  2 MD array, 31,897 samples, was independently re-verified to already
  satisfy this). (3) ``compare_checkshot_to_survey`` reached an untyped
  NumPy ``ValueError`` ("zero-size array to reduction operation") if
  every checkshot Depth row fell outside the locked survey's own MD
  coverage - it now raises a typed ``TimeDepthError`` naming the well,
  the checkshot Depth range, and the survey MD coverage. (4)
  ``p2mem.io.checkshot.load_checkshot_surveys`` did not catch
  ``TimeDepthError`` raised during numerical conditioning, so a defect in
  one well's data could stop the entire batch - it is now caught per well
  (never via a blanket ``except Exception``) and recorded as a typed
  ``CheckshotIngestionFailure(error_type="numerical_conditioning_
  failure")``, isolated exactly like every other expected per-well
  failure. (5) ``seconds_to_milliseconds``/``milliseconds_to_seconds``
  coerced their input directly, unlike ``p2mem.units``'s Increment-1
  input-safety policy, so a boolean, numeric-looking string, or complex
  value would be silently reinterpreted rather than rejected - both now
  reject such input with ``TypeError`` via a local, documented copy of
  ``p2mem.units``'s identical private dtype check (``p2mem/units.py``
  itself remains LOCKED and unmodified). See
  ``INCREMENT_04_1_1_MANIFEST.md`` for the full audit, the regression-test
  list, and the re-verification record.

* Increment 5 adds contract-driven formation-top ingestion, HRS-versus-
  selected-readable source RECONCILIATION, and survey-corrected
  stratigraphic depth mapping (``p2mem.top_models``, ``p2mem.io.tops``,
  ``p2mem.io.tops_inventory``) for the two approved wells with formation-
  top data (Poseidon 2, Boreas 1). Each well has TWO independently
  supplied top files - an "HRS" file (``Top_Name``/``MDRT_m`` only, no
  well name in the file body) and a "selected readable" file
  (``TOP_NAME``/``MDRT_M``/``TVDSS_M``/``NOTE``, with ``#``-comment
  headers and a dashed separator line, deliberately parsed rather than
  treated as data) - and neither is silently preferred: markers are
  matched by exact/normalized name or an explicit human-authored alias
  contract (never fuzzy matching), and MDRT is cross-checked between the
  two sources within a documented 0.005 m tolerance; a marker on which
  the two sources disagree beyond that tolerance is excluded from depth
  mapping (``mapping_status == "not_mapped_mdrt_unresolved"``) rather
  than resolved by picking one file. Reconciled MDRT is mapped through
  the LOCKED Increment 3.1.1 ``petrel_source_trace`` survey trajectory
  using the existing, unmodified ``p2mem.depth_mapping.
  map_las_md_to_tvd_tvdss`` (per marker, so one out-of-coverage marker
  never blocks the rest of the well; extrapolation is never performed),
  producing ``TVD_survey_m``/``TVDSS_survey_corrected_m`` alongside an
  explicit, unambiguous ``TVDSS_residual_source_minus_survey_m =
  TVDSS_source_m - TVDSS_survey_corrected_m`` residual field. Both file
  representations for both wells are classified
  ``well_identity_evidence_status = "inferred_unverified"`` (filename-only
  or in-file-comment association, never independently content-verified -
  a more conservative classification than Increment 4's checkshot files).
  Poseidon North 1 and Proteus 1ST2 have no approved formation-top file
  and are recorded as ``FormationTopAvailabilityRecord(..., "NOT_
  AVAILABLE")`` - never substituted or depth-correlated from another
  well. Independently recomputing (never hardcoding) this increment's two
  required regression findings against the real approved files confirmed:
  Poseidon 2's "selected readable" file's supplied TVDSS equals
  ``MDRT - 21.8 m`` EXACTLY for every one of its 9 markers (21.8 m is the
  well's own rotary-table elevation) - a literal vertical-well-assumption
  depth-reference defect, corrected in the derived
  ``TVDSS_survey_corrected_m`` representation while the raw
  ``TVDSS_source_m`` column is preserved unmodified; survey-corrected
  residuals reproduce Sea Bed at ~0 m, Plover Fm (Top Reservoir) at
  +1.68 m, and TD at +2.49 m, exactly as the approved Rev 1 design
  anticipated. Boreas 1's supplied TVDSS does NOT follow that pattern and
  its maximum absolute source-versus-survey residual is 0.0416 m, below
  the approved 0.05 m tolerance - confirming it was generated from the
  well's real surveyed trajectory, not a vertical-well shortcut, and it is
  therefore NOT "corrected" the way Poseidon 2 is. See
  ``INCREMENT_05_MANIFEST.md`` for the full audit, the per-file contracts,
  and the complete reconciliation/mapping results.

* Increment 5.1 is a narrowly scoped corrective patch to Increment 5,
  applied after an independent technical/software-QA audit, WITHOUT
  beginning Increment 6 or any petrophysics/pore-pressure/mechanical-
  properties/stress/wellbore-stability work, and WITHOUT changing any
  previously verified real Poseidon 2/Boreas 1 formation-top result. It
  corrected three defects and one documentation-accuracy gap: (1)
  ``reconcile_formation_top_sources`` documented "zero canonical markers
  in common between the two sources" as a fatal ``NO_COMMON_MARKERS``
  ERROR, but actually tested the emptiness of the UNION of both sources'
  canonical names - so two entirely DISJOINT, non-empty marker sets (e.g.
  HRS names sharing nothing with the readable file's names) were silently
  accepted as one-sided ``NOT_COMPARABLE`` entries instead of being
  rejected; the check now explicitly evaluates the INTERSECTION of the two
  sources' canonical names and raises the documented ERROR (and
  ``load_formation_top_well`` consequently raises
  ``TopSourceReconciliationError``) whenever both sources are non-empty but
  share nothing, while a legitimate one-sided marker (with at least one
  marker genuinely shared) remains the pre-existing, non-fatal
  ``NOT_COMPARABLE`` case. (2) ``load_formation_top_surveys`` recorded only
  the HRS file's path in every ``TopIngestionFailure.source_path``,
  regardless of which file/stage actually failed - so a failure originating
  from the "selected readable" file (missing file, malformed content, or a
  contract mismatch) could leak that file's own absolute path, unsanitized,
  into exported issues/availability/manifest rows (the sanitizer only ever
  stripped the recorded, and in that case WRONG, HRS path).
  ``TopIngestionFailure`` now carries an explicit ``failure_origin``
  ("hrs"/"readable"/"reconciliation"/"mapping"/"unknown") and both
  ``hrs_path``/``readable_path`` fields, ``load_formation_top_well`` tags
  each raised exception with the stage that actually failed, and every
  exporting function in ``p2mem.io.tops_inventory`` now sanitizes BOTH
  candidate paths (literal substring replacement only, never a regex) and
  reports the correctly identified failing file's basename as context. (3)
  ``reconcile_formation_top_sources`` - the public in-memory API, as
  distinct from the file parsers, which already enforced this - did not
  validate its own documented contract before any numerical comparison:
  non-finite (NaN/Inf) or negative MDRT/TVDSS values, a shorter
  ``NOTE_source`` tuple (previously reaching an untyped ``IndexError``), a
  non-1-dimensional array, and an unvalidated ``mdrt_agreement_tolerance_m``
  keyword (previously accepting ``NaN``, a negative value, or a boolean,
  reaching an incidental ``TypeError`` only for a string) were all silently
  accepted or reached an undocumented incidental error. All of these are
  now rejected before any comparison, with deliberate, documented
  exceptions (``TopParsingError`` for a value/structural defect,
  ``TypeError`` for a type-class defect - matching this module's existing
  split), while valid Python ``int``/``float`` and NumPy integer/floating
  scalars (including 0-d/size-1 arrays) continue to work. (4)
  ``INCREMENT_05_MANIFEST.md`` stated that no ``/home/``, ``/root/``,
  ``/content/``, or absolute path exists ANYWHERE in the package - this was
  inaccurate, since existing synthetic tests and historical documentation
  intentionally contain fake absolute-path strings as test inputs; the
  precise, narrowly scoped claim ("no environment-dependent build path
  appears in exported CSV/JSON outputs") is now stated explicitly in
  ``INCREMENT_05_1_MANIFEST.md``, which also acknowledges the prior
  wording was overbroad. ``INCREMENT_05_MANIFEST.md`` itself is a locked
  historical record and is NOT rewritten. See ``INCREMENT_05_1_MANIFEST.md``
  for the full audit, the regression-test list, and the real-data
  non-regression verification record.

* Increment 5.1.1 is a further narrowly scoped corrective patch to
  Increment 5.1, applied after an independent technical/software-QA audit,
  WITHOUT beginning Increment 6 and WITHOUT changing any scientific result,
  tolerance, depth-mapping method, or formation-top contract. It corrected
  one remaining reconciliation edge case and three documentation-accuracy
  gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint
  marker sets as a fatal ``NO_COMMON_MARKERS`` ERROR, but its condition
  (``both sources non-empty AND intersection empty``, plus a separate
  both-empty case) still silently accepted the remaining zero-common-
  markers configuration: exactly ONE source entirely empty and the other
  non-empty - the intersection of an empty set with anything is itself
  empty, so this was still a zero-common-markers condition, but the
  ``hrs_by_canon and readable_by_canon`` non-empty guard skipped it.
  ``reconcile_formation_top_sources`` now uses the single, strictly
  correct check ``if not common_markers`` (the intersection of the two
  sources' canonical names), which is empty in every zero-common-markers
  configuration - both empty, either side alone empty, or both non-empty
  and disjoint - and is never empty whenever at least one canonical marker
  is genuinely shared, so a legitimate one-sided marker alongside at least
  one shared marker remains the pre-existing, non-fatal ``NOT_COMPARABLE``
  case. (2) ``INCREMENT_05_1_MANIFEST.md`` stated that the real Increment
  5 baseline ZIP's SHA-256 "matches" a governing-prompt-supplied hash that
  was, in fact, a two-character truncation of the real 64-character value
  - a mathematical impossibility that is now stated transparently in
  ``INCREMENT_05_1_1_MANIFEST.md`` as a documentation/input typo, never as
  baseline corruption or uncertainty. (3) ``INCREMENT_05_1_MANIFEST.md``
  Section 7 stated that ``outputs/`` is excluded from the delivered ZIP;
  this was false - the delivered Increment 5.1 ZIP packages the
  ``outputs/`` tree (including the byte-identical Increment 5 formation-
  top outputs/figures) exactly as every prior increment's ZIP has; only
  the dev-only regenerated-comparison directories and private raw source
  files are excluded, and this is now stated accurately. (4) the same
  manifest's clean-room section stated that no ``*.las`` file exists in
  the package; this was false because small, intentionally packaged
  synthetic LAS/checkshot/deviation/top fixtures exist under
  ``tests/fixtures/`` for portable testing - the corrected wording
  distinguishes these fictional fixtures (never leaked private data) from
  the genuinely excluded real/private project LAS, deviation, checkshot,
  and formation-top source files. See ``INCREMENT_05_1_1_MANIFEST.md`` for
  the full audit, the regression-test list, and the real-data
  non-regression verification record.

Subsequent increments (petrophysics, pore pressure, elastic properties,
strength, stress, and wellbore-stability screening) are added one
validated phase at a time and are intentionally absent from this version -
importing them will fail until they exist.
"""

__version__ = "0.5.2"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 5 / 5.1 / 5.1.1 (this release, v0.5.2): Formation-Top Ingestion, Source Reconciliation, and Survey-Corrected Stratigraphic Depth Framework.** Builds on the LOCKED Increment 4.1.2 checkshot/time-depth layer, the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer, and the LOCKED Increment 2.1.1 LAS-ingestion layer (all unmodified) by adding contract-driven formation-top file ingestion for the two approved wells with formation-top data (Poseidon 2, Boreas 1), explicit reconciliation between each well's two independently supplied top-file representations, and mapping of reconciled marker depths through the locked survey trajectory to produce a corrected, auditable stratigraphic marker table. See `INCREMENT_05_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; `INCREMENT_05_1_MANIFEST.md` for the Increment 5.1 corrective patch (a zero-common-marker reconciliation defect, readable-file absolute-path leakage, incomplete in-memory numerical validation, and a manifest-language correction — see "Increment 5.1 update" below); and `INCREMENT_05_1_1_MANIFEST.md` for the Increment 5.1.1 corrective patch (a remaining one-empty-source zero-common-marker edge case, plus three documentation-accuracy corrections — see "Increment 5.1.1 update" below). Formation-top ingestion/reconciliation/depth-correction only — no gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic properties, rock strength, horizontal stresses, or wellbore-stability analysis is performed in this increment.

New in Increment 5:
- `p2mem/top_models.py` — typed, frozen dataclasses for every formation-top header/contract/raw-row/reconciliation/corrected-marker result object, mirroring the checkshot/deviation layers' design philosophy. Two source representations exist per well — an "HRS" file (`Top_Name`/`MDRT_m` only, no well name in the file body) and a "selected readable" file (`TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE`, `#`-comment headers, a dashed separator line) — and every raw (`_source_`) value from both is preserved separately; nothing is ever silently overwritten, sorted, deduplicated, or repaired.
- `p2mem/io/tops.py` — an auditable formation-top parser, per-file contract resolver, and HRS-versus-readable source reconciler for the four approved files, keyed by their exact literal source filenames. Markers are matched between the two files by exact name, whitespace-normalized name, or an explicit human-authored alias contract (`config/formation_top_contracts.yml`'s `marker_name_aliases` — currently empty; no fuzzy/similarity matching of any kind is ever performed). MDRT is cross-checked between the two sources within a documented 0.005 m tolerance; a marker on which the two sources disagree beyond that tolerance is registered (`mdrt_status = "MISMATCH"`) and excluded from depth mapping (`mdrt_authority_basis = "disagreement_unresolved"`, `mapping_status = "not_mapped_mdrt_unresolved"`) rather than resolved by silently picking one file's value. Reconciled MDRT is mapped through the LOCKED Increment 3.1.1 `petrel_source_trace` survey trajectory using the existing, unmodified `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` — reused unmodified, never reimplemented — called once per marker so that one out-of-coverage marker never blocks mapping of the rest of the well, and extrapolation is never performed (a marker outside survey MD coverage is reported as `mapping_status = "rejected_outside_coverage"`, never extrapolated).
- `p2mem/io/tops_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 5 outputs (file inventory, marker/source register, HRS-versus-readable reconciliation table, survey-corrected marker table, ingestion issues, formation-top availability, JSON manifest) — never raw per-marker arrays beyond a single scalar per field, and never a full environment-dependent build path, only a basename.
- `config/formation_top_contracts.yml` — the human-authored, human-reviewable per-file formation-top contract for each of the four approved files, including each file's `well_identity_evidence_status` (both representations for both wells are `inferred_unverified` — the HRS files carry no well name in their body at all, association resting on the filename alone, and the readable files' well name appears only in a project-supplied `#` comment, not independently verified content; this is a deliberately MORE conservative classification than Increment 4's checkshot files, explicitly justified in `INCREMENT_05_MANIFEST.md`).
- **Key real-data findings (disclosed):** every one of Poseidon 2's 9 "selected readable" TVDSS values equals `MDRT_M - 21.8 m` EXACTLY (21.8 m is the well's own rotary-table elevation) — a literal vertical-well-assumption depth-reference defect, since a real TVDSS should differ from MD by more than a constant datum shift once a well deviates. Survey-corrected residuals (`TVDSS_source_m - TVDSS_survey_corrected_m`) reproduce Sea Bed at ≈0 m, Plover Fm (Top Reservoir) at ≈+1.68 m, and TD at ≈+2.49 m — confirming the anticipated defect and correcting it in the derived `TVDSS_survey_corrected_m` representation while `TVDSS_source_m` itself is preserved unmodified. Boreas 1's supplied TVDSS does NOT follow the `MDRT - 21.8` pattern and its maximum absolute source-versus-survey residual across all 10 markers is 0.0416 m, below the approved 0.05 m tolerance — confirming it was generated from the well's real surveyed trajectory, and it is therefore NOT "corrected" the way Poseidon 2 is; evidence is applied per file, per well, never by analogy. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as `formation_top_availability: NOT_AVAILABLE`, a factual data gap, never substituted with another well's tops or correlated by depth alone.
- Still not implemented: gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

New in the Increment 5.1 corrective patch (see "Increment 5.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now correctly rejects two disjoint, non-empty marker sets as a fatal `NO_COMMON_MARKERS` ERROR (previously silently accepted); `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path`, so a failure originating from the "selected readable" file can no longer leak that file's own absolute path into any exported CSV/JSON field; and `reconcile_formation_top_sources` now fully validates its numeric-array and `mdrt_agreement_tolerance_m` inputs before any comparison, with deliberate typed exceptions rather than an incidental `IndexError`/`TypeError`. None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

New in the Increment 5.1.1 corrective patch (see "Increment 5.1.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now rejects the one remaining zero-common-markers configuration Increment 5.1 missed — exactly one source entirely empty, the other non-empty — via a single, strictly correct intersection check (`if not common_markers`) that covers every zero-common-markers case at once, while a legitimate one-sided marker alongside at least one genuinely shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. `INCREMENT_05_1_MANIFEST.md`'s baseline-hash, packaged-outputs, and packaged-LAS-fixture wording is also corrected (documentation-only; see "Increment 5.1.1 update"). None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

**Increment 4 / 4.1 / 4.1.1 / 4.1.2 (v0.4.1.1, LOCKED as of Increment 5): Checkshot (Velocity Survey) Ingestion, Duplicate-Tie Conditioning, and Time–Depth Framework.** Builds on the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer and the LOCKED Increment 2.1.1 LAS-ingestion layer (both unmodified - see below) by adding auditable checkshot parsing with explicit per-file contracts, raw-vs-conditioned duplicate-tie handling, average/interval velocity diagnostics, checkshot-vs-locked-survey depth-reference comparison, a coverage-masked forward/inverse piecewise-linear time-depth interpolation layer, LAS MD-to-checkshot-time mapping within validated coverage only, and a Poseidon-2-only sonic-checkshot drift diagnostic. See `INCREMENT_04_MANIFEST.md` for the full technical design and the independently recomputed real-data statistics, `INCREMENT_04_1_MANIFEST.md` for the Increment 4.1 corrective patch (order-invariant axis-tie conditioning for TVDSS↔OWT/TWT inversion, replacing an order-dependent defect; numerical-validation hardening — see "Increment 4.1 update" below), `INCREMENT_04_1_1_MANIFEST.md` for the Increment 4.1.1 numerical-validation corrective patch (a hidden-reversal grouping defect, incomplete full-MD validation, an untyped zero-coverage crash, missing batch isolation for numerical failures, and a unit-helper input-safety gap — see "Increment 4.1.1 update" below), and `INCREMENT_04_1_2_MANIFEST.md` for the Increment 4.1.2 packaging-only corrective patch (Colab line-ending reproducibility and a truthful notebook completion gate; no scientific, numerical, or version change). Checkshot data QC, duplicate-tie conditioning, and time-depth interpolation only — no formation-top correction, lithology interpretation, density modelling, pore-pressure prediction, elastic properties, rock strength, stress modelling, or wellbore-stability analysis is performed in this increment.

New in Increment 4:
- `p2mem/checkshot_models.py` — typed, frozen dataclasses for every checkshot header/contract/raw-row/duplicate-tie/velocity-diagnostic/depth-comparison/time-mapping/sonic-drift result object, mirroring the deviation-survey layer's design philosophy. Raw (`_source_`) values are always kept explicitly separate from conditioned (`_conditioned_`) values — never overwritten, never mixed.
- `p2mem/io/checkshot.py` — an auditable checkshot (velocity-survey) parser and per-file contract resolver for the three approved checkshot files (`Poseidon2-Checkshot.txt`, `Boreas1-Checkshot.txt`, `Proteus1-Checkshot.txt`), keyed by their exact literal source filenames. File-identity checks (filename, SHA-256, header/survey-statement text, column order, row width, numeric structure) are enforced as blocking `ERROR`s before any time-depth computation is attempted. Raises disclosure `WARNING`s including `DEPTH_BASIS_NOT_EXPLICITLY_DECLARED` (the source file's first column is labelled only `Depth`, never assumed to be MD without evidence) and, for Proteus 1ST2, `WELL_IDENTITY_INFERRED_UNVERIFIED` (the file carries no embedded well identifier tying it to Proteus 1ST2).
- `p2mem/time_depth.py` — the numerical time-depth layer: duplicate/repeated-tie detection and deterministic, disclosed median-based conditioning (raw rows always preserved and separately registered; ties never silently averaged, never force-monotonized with artificial epsilon increments); average velocity (`Vavg = TVDSS / OWT`) and interval velocity (`Vint = ΔTVDSS / ΔOWT`, NaN — never infinite or negative — for any invalid, zero, or non-increasing interval); checkshot-vs-locked-survey depth-reference comparison (residual = survey-interpolated TVDSS − checkshot-supplied TVDSS); forward/inverse piecewise-linear time-depth interpolation with explicit coverage masking (points outside validated checkshot coverage are reported as not-mapped, never extrapolated) via an order-invariant, median-based axis-tie-conditioned lookup table for TVDSS↔OWT/TWT inversion (Increment 4.1 — every tied value on the axis being inverted is grouped and registered, never resolved by an order-dependent "first wins" tie-break); LAS MD-to-checkshot-time mapping for Poseidon 2 within validated coverage only; and a Poseidon-2-only sonic-checkshot drift diagnostic (a local, version-independent trapezoidal integration of sonic slowness over the longest valid continuous MD interval, compared against the checkshot-interpolated OWT increment over the same interval — this diagnostic never modifies `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve itself).
- `p2mem/io/checkshot_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 4 / 4.1 outputs (file inventory, ingestion issues, Depth-axis duplicate-tie register, the Increment 4.1 TVDSS/OWT-axis tie register, depth-tie QC, velocity summary, sonic-checkshot drift summary, time-depth mapping summary, JSON manifest) — never raw per-sample checkshot or LAS arrays, and never a full environment-dependent build path, only a basename.
- `config/checkshot_contracts.yml` — the human-authored, human-reviewable per-file checkshot contract for each of the three approved files, including each well's `model_use_status` (`primary_model` for Poseidon 2 only; `qc_only` for Boreas 1 and Proteus 1ST2) and `identity_evidence_status` (`verified` for Poseidon 2 and Boreas 1; `inferred_unverified` for Proteus 1ST2).
- **Key real-data findings (disclosed):** Poseidon 2's raw checkshot file contains 5 repeated Depth ties, 6 non-increasing TVDSS steps, and 4 non-increasing OWT steps, all independently detected and conditioned (never silently smoothed); its checkshot-vs-survey TVDSS comparison shows a maximum absolute residual of ≈0.089 m with no systematic offset pattern. Boreas 1 shows 3 repeated Depth ties and 4 non-increasing TVDSS steps (OWT strictly increasing throughout) and a checkshot-vs-survey comparison with a near-constant offset of ≈−0.69 m, reported as an observed datum-like offset pattern — not a proven datum error. Proteus 1ST2's file is strictly increasing in all three columns with no repeated ties, and shows a near-constant offset of ≈+0.30 m against the locked survey trajectory; its association with Proteus 1ST2 remains `inferred_unverified` throughout every output. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval (≈MD 2449–4064 m) is ≈+16.7 ms (≈+4.4% of the checkshot-interpolated one-way transit time (OWT) increment over that interval — NOT two-way time; the diagnostic compares the integrated sonic transit time directly against the checkshot's OWT increment over the identical interval), reported as a diagnostic only. Poseidon North 1 has no approved checkshot file; this is recorded as `checkshot_availability: NOT_AVAILABLE`, a factual data gap, not an ingestion failure. Depth-tie conditioning alone does NOT guarantee TVDSS or OWT is itself strictly increasing (required for TVDSS↔OWT/TWT inversion) — see the Increment 4.1 update below for the corrected, order-invariant handling of this. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures.
- Still not implemented: formation-top correction, lithology interpretation, density modelling, sonic/checkshot drift *correction*, synthetic extension of the time-depth relationship beyond measured checkshot coverage, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

**Increment 3 / 3.1 / 3.1.1: Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD–TVD–TVDSS Depth Framework.** Builds on the LOCKED Increment 2.1.1 LAS-ingestion layer (unmodified - see below) by adding Petrel deviation-survey parsing, an explicit per-file survey contract, a standard minimum-curvature trajectory engine, an explicit depth-reference (MD/TVD/TVDSS) framework, and MD-to-TVD/TVDSS mapping of the existing LAS `MD_m` arrays. See `INCREMENT_03_MANIFEST.md` for the Increment 3 technical design and real four-well integration results, `INCREMENT_03_1_MANIFEST.md` for the Increment 3.1 corrective patch (four audit findings: source-filenames-with-spaces, absolute-path leakage, DLS-normalization disclosure, dogleg numerical stability — see "Increment 3.1 update" below), and `INCREMENT_03_1_1_MANIFEST.md` for the Increment 3.1.1 packaging-only corrective patch (notebook/source `%%writefile` synchronization; no scientific or numerical change). The Proteus 1ST2 trajectory-discrepancy finding is disclosed, not resolved, in any of these releases. This layer, and the Increment 2.1.1 LAS-ingestion layer beneath it, are LOCKED as of Increment 4 and reused unmodified.

Locked from Increment 2.1.1 (unmodified in Increment 3 unless a blocking defect is documented - none was found):
- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged since Increment 1.1. See the module docstring and `tests/test_units.py`.
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml` — the auditable LAS 2.0 parser, per-file curve-contract resolver, and inventory builders for the four approved wells, corrected and independently re-verified through Increment 2.1.1 (158 tests passing, 4/4 real wells loading with zero ingestion errors). See `INCREMENT_02_v2.1.1_MANIFEST.md`.

New in Increment 3:
- `p2mem/deviation_models.py` — typed, frozen dataclasses for every deviation-survey/trajectory/depth-mapping result object (header info, per-file contract, raw station data, minimum-curvature result, trajectory-validation residuals, depth-basis selection, typed batch failures, LAS depth-mapping result), mirroring the LAS layer's design philosophy. Every source (`_source_`) array is kept explicitly separate from every independently computed (`_mc_`) array — never overwritten, never mixed.
- `p2mem/io/deviation.py` — an auditable Petrel deviation-survey (well-trace) parser and per-file contract resolver for the same four wells, keyed by their exact, literal source filenames (which contain spaces, e.g. `"Poseidon 2_dev.txt"` — corrected in Increment 3.1; see below). Extracts and preserves the full header block (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system text, declared angle/depth/coordinate conventions) and the exact 11-column station table, with file-identity checks (filename, SHA-256, well/survey identifier, wellhead/datum values, coordinate system, column order, station count, MD coverage) all enforced as blocking `ERROR`s before any trajectory computation is attempted. Also raises two disclosure `WARNING`s for every successfully loaded file: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED`, and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (the supplied `DLS` column's degrees-per-30-m normalization is inferred, not header-declared, and is independently verified per file against a recomputation from that file's own inclination/azimuth).
- `p2mem/trajectory.py` — the standard minimum-curvature method (a numerically stable `arctan2(||cross||, dot)` dogleg-angle formulation — Increment 3.1 correction, see below — a ratio factor with an explicit Taylor-series limit as the dogleg approaches zero, TVD/northing/easting displacement, dogleg severity in degrees per 30 m), implemented explicitly and transparently with no third-party survey-computation library.
- `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation of the locked LAS `MD_m` array against the explicitly selected depth-trajectory basis, using a documented, deterministic piecewise-linear station interpolation (never a per-sample minimum-curvature recomputation) that never extrapolates silently.
- `p2mem/io/deviation_inventory.py` — deterministic, metadata-only inventory-table builders for the Increment 3 outputs (file inventory, trajectory-validation summary, depth-reference register, LAS depth-mapping summary, ingestion issues, JSON manifest) — never raw per-sample station or LAS arrays, and (Increment 3.1 correction) never a full environment-dependent build path, only a basename.
- `config/deviation_survey_contracts.yml` — the human-authored, human-reviewable per-file deviation-survey contract for each of the four wells, keyed by the exact literal source filename (`"Poseidon 2_dev.txt"`, `"Boreas 1_dev.txt"`, `"Poseidon North 1_dev.txt"`, `"Proteus 1ST2_dev.txt"` — corrected in Increment 3.1), including an explicit, uniformly applied `depth_basis_policy` (`petrel_source_trace`, the conservative default given the Proteus 1ST2 finding below) and residual-comparison tolerances declared once and applied identically to every well (never tuned per well to force a pass/fail outcome).
- **Key real-data finding (disclosed, not resolved):** independent minimum-curvature reconstruction of Poseidon 2, Boreas 1, and Poseidon North 1 agrees with their Petrel-supplied TVD to approximately millimetre scale. Proteus 1ST2 shows a materially larger discrepancy (~0.19 m TVD, ~1.6 m easting at maximum) concentrated in its deeper section (below ~MD 4200 m), even though its own supplied dogleg-severity column is internally consistent with an independent recomputation from its own inclination/azimuth at every station. This is reported as a visible trajectory-validation `WARNING`, not corrected, hidden, or used to justify loosening every well's tolerance — see `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation and the evidence pattern observed. Unaffected by the Increment 3.1 patch.
- Still not implemented (as of Increment 3/3.1/3.1.1): checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those were explicitly out of scope for this increment. Checkshot ingestion and the time-depth framework were subsequently added in Increment 4 (see above); the remainder are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — used for parsing the human-authored curve contracts in `config/las_curve_contracts.yml`, `config/deviation_survey_contracts.yml` (Increment 3), `config/checkshot_contracts.yml` (Increment 4), and (new in Increment 5) `config/formation_top_contracts.yml`. No new runtime dependency was added in Increment 3, 4, or 5: the minimum-curvature engine, depth-mapping interpolation, duplicate-tie conditioning, velocity diagnostics, time-depth interpolation, and formation-top reconciliation/mapping all use only NumPy (including a local, version-independent trapezoidal-integration helper in `p2mem/time_depth.py`, added because `numpy.trapz`/`numpy.trapezoid` are not consistently available across supported NumPy versions). Matplotlib and pandas are used only for notebook display and QC-figure generation (`run_integration_03.py`/`run_integration_04.py`/`run_integration_05.py`, the Increment 3/4/5 notebooks) — never imported by the installable `p2mem` package itself. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` (locked since Increment 2.1.1) against small synthetic LAS fixtures. The suites in `tests/test_trajectory.py`, `tests/test_deviation.py`, and `tests/test_depth_mapping.py` (new in Increment 3; extended in Increment 3.1 with dogleg numerical-stability, DLS-normalization-disclosure, and real-filename-with-spaces regression/negative tests) validate the minimum-curvature engine, the Petrel deviation-survey parser/contract resolver, and the MD-to-TVD/TVDSS mapping respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data (this layer is locked, unmodified, as of Increment 4). `tests/test_deviation_inventory.py` (new in Increment 3.1) validates that no exported inventory/issues/manifest row, for a successful or a failed well, ever embeds a full environment-dependent build path. The suites in `tests/test_checkshot.py`, `tests/test_time_depth.py`, and `tests/test_checkshot_inventory.py` (new in Increment 4) validate the checkshot parser/contract resolver, the duplicate-tie conditioning/velocity-diagnostic/time-depth-interpolation/sonic-drift numerical layer, and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` (including a CRLF fixture used to verify line-ending detection) and in-memory synthetic/analytic data — including an analytic constant-velocity case used to independently verify the trapezoidal-integration helper. The suites in `tests/test_tops.py` and `tests/test_tops_inventory.py` (new in Increment 5) validate the formation-top parser/contract resolver/HRS-versus-readable reconciliation logic and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data, including synthetic analogs of both real regression findings (a vertical-assumption depth-reference defect with a growing residual, and a survey-consistent well with a near-zero residual). None of these suites require the private/raw project LAS, deviation, checkshot, or formation-top files, so the full suite runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real integration validation (which DOES require the raw LAS/deviation/checkshot/formation-top files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, and `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   ├── deviation_models.py
│   ├── trajectory.py
│   ├── depth_mapping.py
│   ├── checkshot_models.py
│   ├── time_depth.py
│   ├── top_models.py
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       ├── inventory.py
│       ├── deviation.py
│       ├── deviation_inventory.py
│       ├── checkshot.py
│       ├── checkshot_inventory.py
│       ├── tops.py
│       └── tops_inventory.py
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   ├── test_trajectory.py
│   ├── test_deviation.py
│   ├── test_depth_mapping.py
│   ├── test_deviation_inventory.py
│   ├── test_checkshot.py
│   ├── test_time_depth.py
│   ├── test_checkshot_inventory.py
│   ├── test_tops.py
│   ├── test_tops_inventory.py
│   └── fixtures/         (small synthetic LAS + deviation-survey + checkshot + formation-top files; no project raw data)
├── config/
│   ├── las_curve_contracts.yml
│   ├── deviation_survey_contracts.yml
│   ├── checkshot_contracts.yml
│   └── formation_top_contracts.yml
├── data/
│   └── raw/
│       ├── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
│       ├── deviation/    (the four raw deviation-survey files, exact filenames contain spaces, e.g. "Poseidon 2_dev.txt" - NOT included in this repository; immutable inputs)
│       ├── checkshot/    (the three raw checkshot files, exact filenames e.g. "Poseidon2-Checkshot.txt" - NOT included in this repository; immutable inputs, never rewritten/renamed/"cleaned")
│       └── tops/         (the four raw formation-top files, e.g. "Poseidon_2_HRS_tops_no_wellname_MDRT.txt" - NOT included in this repository; immutable inputs)
├── notebooks/  (reserved for later increments)
└── outputs/
    ├── 02_las_inventory/            (Increment 2.1.1 real four-well run: CSV/JSON metadata only, no raw log samples)
    ├── 03_deviation_depth/          (Increment 3 real four-well run: CSV/JSON metadata + QC figures, no raw station/log samples)
    ├── 04_checkshot_time_depth/     (Increment 4 real three-file checkshot run: CSV/JSON metadata + QC figures, no raw per-sample checkshot/LAS arrays)
    └── 05_formation_tops/           (Increment 5 real four-file formation-top run: CSV/JSON metadata + QC figures, no raw per-marker arrays beyond scalar fields)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 / 2.1.1 update:** corrective patches addressing independent audits' naming, reporting, contract-validation, and notebook/source-synchronization findings — see `INCREMENT_02_v2.1_MANIFEST.md` and `INCREMENT_02_v2.1.1_MANIFEST.md`. No new scientific finding was made in either patch; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.
- **Increment 3 update:** deviation-survey ingestion independently reconfirms the Petrel-supplied trajectory for Poseidon 2, Boreas 1, and Poseidon North 1 to approximately millimetre scale via minimum curvature, and additionally DISCOVERS (not merely reconfirms) a real trajectory-reconstruction discrepancy in Proteus 1ST2's deeper section (~0.19 m TVD, ~1.6 m easting at maximum, concentrated below ~MD 4200 m) — see "Current implementation status" above and `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation. This is disclosed as an open item, not corrected or hidden; downstream MD-to-TVD/TVDSS mapping for Proteus 1ST2 conservatively uses the Petrel-supplied source trajectory (not the disagreeing minimum-curvature trajectory) as a result.
- **Increment 3.1 update:** a corrective patch addressing an independent audit's findings on source-filename handling, output environment-independence, an undisclosed normalization inference, and dogleg-angle numerical conditioning — see `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. No new scientific finding was made in this patch; the Proteus 1ST2 discrepancy above is unaffected, remains disclosed exactly as before, and was neither corrected nor concealed. The only numerical changes are at the floating-point noise floor of the diagnostic `dogleg_deg`/`dls_deg_per_30m`/TVD-and-offset-residual fields (at most ~9×10⁻¹³ m for TVD, ~7×10⁻¹⁵ m for easting/northing, across all four real wells), with zero change to any well's PASS/WARNING status.
- **Increment 3.1.1 update:** a packaging-only corrective patch (three notebook `%%writefile` cells that had drifted from their packaged source files, mirroring the earlier Increment 2.1.1 finding). No scientific, numerical, or real-data change of any kind — see `INCREMENT_03_1_1_MANIFEST.md`.
- **Increment 4 update:** checkshot ingestion and the time-depth framework independently reproduce every raw-data anomaly the project design anticipated (Poseidon 2: 5 repeated Depth ties, 6 non-increasing TVDSS steps, 4 non-increasing OWT steps, and a ≈257 m gap in Depth coverage between 1313.1 m and 1570.1 m; Boreas 1: 3 repeated Depth ties and 4 non-increasing TVDSS steps with OWT strictly increasing; Proteus 1ST2: strictly increasing in all three raw columns) and DISCLOSES (not resolves) two further items: (1) Boreas 1's and Proteus 1ST2's checkshot-vs-locked-survey TVDSS comparisons each show a near-constant offset (≈−0.69 m and ≈+0.30 m respectively) — reported as an observed datum-like offset pattern, not a proven datum error, with both source references preserved unmodified; (2) `Proteus1-Checkshot.txt` carries no embedded well identifier, so its association with Proteus 1ST2 is recorded as `identity_status: inferred_unverified` and used for QC only, never as a substitute time-depth model for any other well. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval is a diagnostic finding only (≈+16.7 ms, ≈+4.4%) and does not trigger any correction of `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve. Poseidon North 1 has no approved checkshot file and is recorded as a factual data gap (`checkshot_availability: NOT_AVAILABLE`), not substituted with another well's data. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures. **NOTE:** `INCREMENT_04_MANIFEST.md` incorrectly stated TVDSS is strictly increasing after Depth-tie conditioning for all three wells; this was corrected by Increment 4.1 (see below) — do not rely on that original statement.
- **Increment 4.1 update:** a narrowly scoped corrective patch to Increment 4, applied after an independent numerical-method audit, that does NOT begin Increment 5 or any later-phase work. It corrected an order-dependent tie-break: `tvdss_to_owt`/`owt_to_tvdss` (and their TWT equivalents) previously resolved a repeated value on the axis being inverted by silently keeping whichever tied row was encountered first in the Depth-conditioned table and discarding the other, with no audit trail beyond a bare count. This is replaced by an explicit, order-invariant policy (`p2mem.time_depth.build_axis_conditioned_lookup_table`/`build_axis_conditioned_tables_for_well`): every tied value is grouped by exact equality regardless of parse order, every tied row is registered in a new audit register (`checkshot_time_axis_tie_register.csv` — separate from, and never confused with, the pre-existing Depth-axis `checkshot_duplicate_tie_register.csv`), and the group's dependent-value MEDIAN becomes the conditioned representative (order-invariant; a screening-level choice, not proof the original TVDSS↔OWT relationship was single-valued at that tied value). A genuine reversal (not a tie) raises a typed error rather than being sorted or forced monotonic. Independently reproduced real-data counts: Poseidon 2 has two TVDSS-axis and two OWT-axis tie groups after Depth-tie conditioning; Boreas 1 has one TVDSS-axis tie group and zero OWT-axis tie groups; Proteus 1ST2 has none of either — correcting `INCREMENT_04_MANIFEST.md`'s original, incorrect "strictly increasing for all three wells" statement. This patch also hardens `trapezoidal_integrate` and `compute_sonic_checkshot_drift` to validate their numerical preconditions (finite, one-dimensional, equal-length, strictly increasing MD/x where required) — most notably, a decreasing or duplicate MD run can no longer silently produce a physically invalid negative transit time; it now raises a typed error instead. None of this hardening changes the already-verified real Poseidon 2 sonic-drift result, which is bit-for-bit unchanged. See `INCREMENT_04_1_MANIFEST.md` for the full audit, corrected statistics, and re-verification record.

- **Increment 4.1.1 update:** a narrowly scoped numerical-validation corrective patch to Increment 4.1, applied after an independent numerical-method/software-QA audit, that does NOT begin Increment 5 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1/Proteus 1ST2 result. It corrected four blocking defects and one input-safety gap. (1) `build_axis_conditioned_lookup_table` grouped ALL occurrences of an identical axis value together GLOBALLY before checking for a reversal, so a reversal that returned to an already-seen value (e.g. `[100.0, 200.0, 100.0]`) was silently hidden rather than raising a typed error; it now evaluates the ORIGINAL, ungrouped sequence's successive differences for negativity BEFORE any grouping is attempted — provably equivalent to the 4.1 behavior for every legitimate adjacent tie, and strictly stronger against a non-adjacent reversal. (2) `compute_sonic_checkshot_drift`/`find_longest_finite_positive_run` validated MD monotonicity only within the selected finite-positive-VP run, so a decreasing or duplicate MD value outside that run (e.g. at a NaN-VP station) could pass silently; the COMPLETE canonical `md_m` array is now required finite and strictly increasing before run-selection (the real Poseidon 2 MD array, 31,897 samples, was independently re-verified to already satisfy this — the real sonic-drift result is bit-for-bit unchanged). (3) `compare_checkshot_to_survey` reached an untyped NumPy `ValueError` ("zero-size array to reduction operation") if every checkshot Depth row fell outside the locked survey's own MD coverage; it now raises a typed error naming the well, the checkshot Depth range, and the survey MD coverage. (4) `p2mem.io.checkshot.load_checkshot_surveys` did not catch the typed numerical-conditioning error, so a defect in one well's data could stop the entire batch; it is now caught per well (never via a blanket exception handler) and recorded as a typed, isolated per-well failure, exactly like every other expected failure mode. (5) `seconds_to_milliseconds`/`milliseconds_to_seconds` coerced their input directly, unlike `p2mem.units`'s Increment-1 input-safety policy, so a boolean, numeric-looking string, or complex value would be silently reinterpreted rather than rejected; both now reject such input with a typed error via a local, documented copy of `p2mem.units`'s identical private dtype check (`p2mem/units.py` itself remains LOCKED and unmodified). See `INCREMENT_04_1_1_MANIFEST.md` for the full audit, the regression-test list, and the re-verification record.

- **Increment 5 update:** formation-top ingestion independently reproduces both required regression findings from the real approved files. Poseidon 2's "selected readable" file's supplied TVDSS equals `MDRT - 21.8 m` EXACTLY for every one of its 9 markers (21.8 m is the well's own rotary-table elevation) — a vertical-well-assumption depth-reference defect that is corrected in the derived `TVDSS_survey_corrected_m` representation, reproducing residuals of ≈0 m at Sea Bed, ≈+1.68 m at Plover Fm (Top Reservoir), and ≈+2.49 m at TD. Boreas 1's supplied TVDSS does not follow that pattern (maximum absolute residual 0.0416 m, below the 0.05 m tolerance) and is therefore NOT corrected — this is applied per file, per well, never by analogy from Poseidon 2's defect. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as a factual data gap (`formation_top_availability: NOT_AVAILABLE`), never substituted with another well's tops or correlated by depth alone. Both formation-top file representations for both wells are classified `well_identity_evidence_status: inferred_unverified` — a deliberately more conservative classification than Increment 4's checkshot files, since neither the HRS files (no well name in the body) nor the readable files (well name only in a project-supplied comment) constitute independently verified file content; see `INCREMENT_05_MANIFEST.md` Section 2 for the full rationale. No lithology interpretation, petrophysical calculation, or geological correlation is performed on these markers — the Increment 5 marker-depth comparison figures are explicitly labelled as depth comparisons only.

- **Increment 5.1 update:** a narrowly scoped corrective patch to Increment 5, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1 formation-top result above. It corrected three defects and one documentation-accuracy gap. (1) `reconcile_formation_top_sources` documented "zero canonical markers in common between the two sources" as a fatal `NO_COMMON_MARKERS` ERROR, but actually tested the emptiness of the UNION of both sources' canonical names, so two entirely disjoint, non-empty marker sets were silently accepted as one-sided `NOT_COMPARABLE` entries instead of being rejected; the check now explicitly evaluates the INTERSECTION of the two sources' canonical names, and `load_formation_top_well` consequently raises `TopSourceReconciliationError` for this condition, while a legitimate one-sided marker (with at least one marker genuinely shared) remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `load_formation_top_surveys` recorded only the HRS file's path in every `TopIngestionFailure.source_path`, regardless of which file/stage actually failed, so a failure originating from the "selected readable" file (a missing file, malformed content, or a contract mismatch) could leak that file's own absolute path, unsanitized, into exported issues/availability/manifest rows; `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path` fields, and every exporting function in `p2mem.io.tops_inventory` now sanitizes both candidate paths (literal substring replacement only, never a regex). (3) `reconcile_formation_top_sources` — the public in-memory API, as distinct from the file parsers, which already enforced this — did not validate its own documented contract before any numerical comparison: non-finite or negative MDRT/TVDSS values, a shorter `NOTE_source` tuple (previously reaching an untyped `IndexError`), a non-1-dimensional array, and an unvalidated `mdrt_agreement_tolerance_m` keyword (previously accepting NaN, a negative value, or a boolean) were all silently accepted or reached an undocumented incidental error; all are now rejected before any comparison, with deliberate, documented exceptions. (4) `INCREMENT_05_MANIFEST.md`'s statement that no absolute path exists ANYWHERE in the package was overbroad and inaccurate, since existing synthetic tests and historical documentation intentionally contain fake absolute-path strings as test inputs; the precise, narrowly scoped claim ("no environment-dependent build path appears in exported CSV/JSON outputs") is now stated explicitly in `INCREMENT_05_1_MANIFEST.md`, which also acknowledges the prior wording was overbroad — `INCREMENT_05_MANIFEST.md` itself is a locked historical record and is NOT rewritten. See `INCREMENT_05_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 5.1.1 update:** a further narrowly scoped corrective patch to Increment 5.1, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any scientific result, tolerance, depth-mapping method, formation-top contract, or real-data output. It corrected one remaining reconciliation edge case and three documentation-accuracy gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint marker sets, but its condition ("both sources non-empty AND intersection empty", plus a separate both-empty case) still silently accepted the one remaining zero-common-markers configuration: exactly one source entirely empty and the other non-empty (the intersection of an empty set with anything is itself empty, so this was still a zero-common-markers condition). `reconcile_formation_top_sources` now uses the single, strictly correct check `if not common_markers` — empty in every zero-common-markers case (both empty, either side alone empty, or both non-empty and disjoint) and never empty whenever at least one canonical marker is genuinely shared, so a legitimate one-sided marker alongside at least one shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `INCREMENT_05_1_MANIFEST.md` stated that the real Increment 5 baseline ZIP's SHA-256 "matches" a governing-instruction-supplied hash that was, in fact, a two-character truncation of the real 64-character value — a mathematical impossibility, now stated transparently as a documentation/input typo, never as baseline corruption or uncertainty. (3) `INCREMENT_05_1_MANIFEST.md` Section 7 stated `outputs/` is excluded from the delivered ZIP; this was false — the delivered Increment 5.1 ZIP packages the `outputs/` tree (including the byte-identical Increment 5 formation-top outputs/figures), exactly as every prior increment's ZIP has; only dev-only regenerated-comparison directories and private raw source files are excluded, now stated accurately. (4) the same manifest's clean-room section stated no `*.las` file exists in the package; this was false because small, intentionally packaged synthetic LAS/checkshot/deviation/top fixtures exist under `tests/fixtures/` for portable testing — the corrected wording distinguishes these fictional fixtures from the genuinely excluded real/private project source files. See `INCREMENT_05_1_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


#### `p2mem/top_models.py` — typed dataclasses for the formation-top layer

In [ ]:
%%writefile p2mem/top_models.py
"""
p2mem.top_models - Typed, documented result/schema objects for the
Increment 5 formation-top ingestion, source-reconciliation, and
survey-corrected stratigraphic depth layer.

Design rationale
-----------------
Mirrors `p2mem.deviation_models` (Increment 3) and `p2mem.checkshot_models`
(Increment 4): every object here is a frozen `dataclass`, nothing here
performs I/O or numerical computation, and every array/field name is
explicit about what it holds (raw vs. reconciled vs. survey-derived - see
"Naming discipline" below). This module collects every typed object used
by `p2mem.io.tops` and `p2mem.io.tops_inventory`.

Two source representations, reconciled - never silently preferred
--------------------------------------------------------------------
Each approved well (Poseidon 2, Boreas 1) has TWO independently supplied
formation-top files:

* an "HRS" file (`Top_Name`, `MDRT_m` only - no TVDSS, no well name
  embedded in the file body; the well association rests on the filename
  alone);
* a "selected readable" file (`TOP_NAME`, `MDRT_M`, `TVDSS_M`, `NOTE`,
  preceded by `#`-prefixed comment lines that NAME the well - project-
  supplied metadata, not independently verified file content).

Neither file is treated as more authoritative than the other for MDRT
placement: `p2mem.io.tops` explicitly RECONCILES the two (exact/
normalized/aliased name matching, MDRT cross-check within a documented
tolerance) and registers every match, mismatch, and single-source marker
in `TopReconciliationEntry` - never silently picking one file's value.
Supplied TVDSS (only ever present in the "selected readable" file) is
preserved under its own `_source_` name and is NEVER treated as the
corrected depth; the project's corrected stratigraphic depth is always
`TVDSS_survey_corrected_m`, computed by mapping the well's reconciled
MDRT through the LOCKED Increment 3.1.1 `petrel_source_trace` survey
trajectory (`p2mem.depth_mapping.map_las_md_to_tvd_tvdss` - reused
unmodified; this module never reimplements minimum curvature or the
TVD/TVDSS formula). See `TopMarkerRecord` and the module-level residual-
sign-convention note below.

Naming discipline (explicit, unit-suffixed names - project-wide policy)
-------------------------------------------------------------------------
* `MDRT_source_hrs_m` / `MDRT_source_readable_m` - each source file's own
  literal MDRT value for a marker, UNMODIFIED. Never overwritten.
* `MDRT_reconciled_m` - the single MDRT value actually used for depth
  mapping, chosen ONLY when both sources agree within tolerance (or only
  one source supplies the marker); see `mdrt_authority_basis`. Never
  silently chosen when the two sources disagree beyond tolerance - such a
  marker is excluded from mapping and reported (`mapping_status ==
  "not_mapped_mdrt_unresolved"`).
* `TVDSS_source_m` - the "selected readable" file's own supplied TVDSS
  column, UNMODIFIED, preserved for residual/QC comparison only.
* `TVD_survey_m` / `TVDSS_survey_corrected_m` - the project's corrected
  depth representation, computed by mapping `MDRT_reconciled_m` through
  the locked survey trajectory (`TVDSS_survey_corrected_m = TVD_survey_m
  - datum_elevation_m`, identical convention to
  `p2mem.depth_mapping`/`p2mem.checkshot_models`).
* `TVDSS_residual_source_minus_survey_m` - the ONE explicit, named
  residual field used everywhere in this layer (never a bare `error_m` or
  similarly ambiguous name):

      TVDSS_residual_source_minus_survey_m = TVDSS_source_m - TVDSS_survey_corrected_m

  Positive means the source-supplied TVDSS is DEEPER (more positive) than
  the survey-corrected value at the same reconciled MDRT.

Evidence-status vocabulary (reused from `p2mem.checkshot_models`)
----------------------------------------------------------------------
`well_identity_evidence_status` reuses the project's established two-value
vocabulary: "verified" (the file's own content, or an independently
verified cross-reference, proves which well it belongs to) or
"inferred_unverified" (association rests on filename and/or project-
supplied in-file comments only - see `p2mem.io.tops` module docstring for
why NEITHER of this increment's two file representations is ever
described as "verified" by content alone). `model_use_status` is likewise
reused: every approved well's formation tops here are declared
"primary_model" (both Poseidon 2 and Boreas 1 formation-top sets feed the
project's stratigraphic marker framework equally - there is no
primary/qc_only distinction analogous to Increment 4's checkshot layer,
because reconciliation, not selection-of-one-well's-curve, is this
increment's authority mechanism; this is declared explicitly in the
contract, not left implicit). `formation_top_availability` mirrors
`CheckshotAvailabilityRecord.checkshot_availability` exactly
("AVAILABLE" / "NOT_AVAILABLE").
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np

__all__ = [
    "STATUS_PASS",
    "STATUS_WARNING",
    "STATUS_FAIL",
    "VALID_IDENTITY_EVIDENCE_STATUSES",
    "VALID_MODEL_USE_STATUSES",
    "VALID_TOP_AVAILABILITY_STATUSES",
    "VALID_REPRESENTATION_TYPES",
    "VALID_NAME_MATCH_STATUSES",
    "VALID_MDRT_STATUSES",
    "VALID_MAPPING_STATUSES",
    "VALID_TOP_FAILURE_ORIGINS",
    "HRSTopHeaderInfo",
    "ReadableTopHeaderInfo",
    "HRSTopFileContract",
    "ReadableTopFileContract",
    "TopIngestionIssue",
    "HRSTopStationData",
    "ReadableTopStationData",
    "TopReconciliationEntry",
    "TopMarkerRecord",
    "TopIngestionFailure",
    "FormationTopWellResult",
    "FormationTopAvailabilityRecord",
]

# ---------------------------------------------------------------------------
# Shared status vocabulary (mirrors p2mem.deviation_models / checkshot_models)
# ---------------------------------------------------------------------------
STATUS_PASS = "PASS"
STATUS_WARNING = "WARNING"
STATUS_FAIL = "FAIL"

VALID_IDENTITY_EVIDENCE_STATUSES = ("verified", "inferred_unverified")
VALID_MODEL_USE_STATUSES = ("primary_model", "qc_only")
VALID_TOP_AVAILABILITY_STATUSES = ("AVAILABLE", "NOT_AVAILABLE")
VALID_REPRESENTATION_TYPES = ("HRS_MDRT_only", "selected_readable_MDRT_TVDSS")
VALID_NAME_MATCH_STATUSES = (
    "exact",
    "normalized_match",
    "aliased_match",
    "missing_in_hrs",
    "missing_in_readable",
)
VALID_MDRT_STATUSES = ("MATCHED", "MISMATCH", "NOT_COMPARABLE")
VALID_MAPPING_STATUSES = (
    "mapped_within_coverage",
    "rejected_outside_coverage",
    "not_mapped_mdrt_unresolved",
)

# Increment 5.1 (Finding 2 fix) - identifies which stage of ingestion a
# `TopIngestionFailure` actually originated from, so the correct source
# path(s) can be sanitized wherever the failure is exported. "unknown" is
# reserved for a `TopIngestionFailure` built without this classification
# (e.g. directly by a caller/test predating this field) - never assumed to
# be "hrs" by default.
VALID_TOP_FAILURE_ORIGINS = ("hrs", "readable", "reconciliation", "mapping", "unknown")


# ---------------------------------------------------------------------------
# Raw header / provenance - one dataclass per file representation, since the
# two formats are structurally different (mirrors p2mem.io.checkshot vs
# p2mem.io.deviation each having their own header dataclass).
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class HRSTopHeaderInfo:
    """
    The literal, structurally parsed header of one "HRS" formation-top
    file: a single tab-separated column-header line (`Top_Name`,
    `MDRT_m`), no comment lines, no embedded well name anywhere in the
    file body. `well_identity_source` is always the literal string
    `"filename_only"` - this is stated here as a factual, structural
    property of the file format, never as a claim of verification.
    """

    source_filename: str
    sha256: str
    column_header_line: str
    column_names: Tuple[str, ...]
    well_identity_source: str  # always "filename_only" for this format
    line_ending_convention: str
    header_line_count: int
    data_line_offset: int


@dataclass(frozen=True)
class ReadableTopHeaderInfo:
    """
    The literal, structurally parsed header of one "selected readable"
    formation-top file: one or more leading `#`-prefixed comment lines
    (preserved verbatim in `comment_lines`), a blank line, a whitespace-
    padded column-header line (`TOP_NAME`, `MDRT_M`, `TVDSS_M`, `NOTE`),
    and a dashed separator line (preserved verbatim in
    `separator_line_raw`, structurally recognized and skipped, never
    accidentally treated as a data row).

    `well_name_from_comment` is whatever well name string this file's own
    comment block states (e.g. parsed from "# Selected readable well tops
    for Boreas 1") - this is PROJECT-SUPPLIED METADATA embedded in the
    file, not independently verified file content, and is never described
    as "verified" identity evidence (see `p2mem.io.tops` module
    docstring). `None` if no such comment line is present.
    """

    source_filename: str
    sha256: str
    comment_lines: Tuple[str, ...]
    well_name_from_comment: Optional[str]
    column_header_line: str
    column_names: Tuple[str, ...]
    separator_line_raw: str
    well_identity_source: str  # always "in_file_comment_project_supplied" for this format
    line_ending_convention: str
    header_line_count: int
    data_line_offset: int


# ---------------------------------------------------------------------------
# Per-file contract (config/formation_top_contracts.yml)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class HRSTopFileContract:
    """
    One HRS-format file's complete, human-authored expectation set.
    `expected_tvdss_min_m` / `expected_tvdss_max_m` do not apply to this
    format (no TVDSS column) and are intentionally absent from this
    dataclass - see `ReadableTopFileContract` for those fields.
    """

    source_filename: str
    expected_sha256: str
    project_well_key: str
    representation_type: str  # always "HRS_MDRT_only"
    well_identity_evidence_status: str
    well_identity_evidence_notes: str
    model_use_status: str
    expected_column_header_line: str
    expected_column_order: Tuple[str, ...]
    expected_column_count: int
    expected_marker_count: int
    expected_mdrt_min_m: float
    expected_mdrt_max_m: float
    numeric_range_tolerance: float
    datum_depth_column_interpretation: str
    notes: str


@dataclass(frozen=True)
class ReadableTopFileContract:
    """One "selected readable"-format file's complete, human-authored
    expectation set."""

    source_filename: str
    expected_sha256: str
    project_well_key: str
    representation_type: str  # always "selected_readable_MDRT_TVDSS"
    well_identity_evidence_status: str
    well_identity_evidence_notes: str
    model_use_status: str
    expected_well_name_from_comment: str
    expected_column_header_line: str
    expected_column_order: Tuple[str, ...]
    expected_column_count: int
    expected_marker_count: int
    expected_mdrt_min_m: float
    expected_mdrt_max_m: float
    expected_tvdss_min_m: float
    expected_tvdss_max_m: float
    numeric_range_tolerance: float
    datum_depth_column_interpretation: str
    notes: str


# ---------------------------------------------------------------------------
# Issues
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class TopIngestionIssue:
    """One ERROR (blocking) or WARNING (non-blocking, disclosed) fact."""

    severity: str  # "ERROR" or "WARNING"
    code: str
    message: str
    context: str


# ---------------------------------------------------------------------------
# Raw station data - every raw row preserved exactly, in file order
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class HRSTopStationData:
    """
    Exact, raw parsed HRS-file rows, in original file order. Nothing here
    is deduplicated, reordered, renamed, or repaired.
    """

    Top_Name_source: Tuple[str, ...]
    MDRT_source_m: np.ndarray


@dataclass(frozen=True)
class ReadableTopStationData:
    """
    Exact, raw parsed "selected readable"-file rows, in original file
    order. `NOTE_source` preserves the file's own free-text note column
    exactly (empty string where the file's own NOTE field is blank -
    never fabricated).
    """

    TOP_NAME_source: Tuple[str, ...]
    MDRT_source_m: np.ndarray
    TVDSS_source_m: np.ndarray
    NOTE_source: Tuple[str, ...]


# ---------------------------------------------------------------------------
# HRS-versus-readable source reconciliation (one row per canonical marker)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class TopReconciliationEntry:
    """
    One auditable row reconciling ONE canonical marker between this well's
    HRS file and its selected-readable file. Both raw row numbers
    (0-indexed, into that source's own original file order) are preserved
    so a reader can trace this entry back to the exact source line.

    `name_match_status` records how the two sources' own marker-name
    spellings were matched (see `p2mem.io.tops.MARKER_NAME_ALIAS_CONTRACT`
    for the human-authored alias table; "aliased_match" is the ONLY status
    that consulted it - "normalized_match" means whitespace/padding
    differed but the stripped, whitespace-collapsed names were identical,
    with no alias table involved).

    `mdrt_status` is "NOT_COMPARABLE" whenever the marker is present in
    only one source (there is nothing to compare); it is never reported
    as "MATCHED" in that case.
    """

    well_key: str
    canonical_marker_name: str
    hrs_marker_name_raw: Optional[str]
    readable_marker_name_raw: Optional[str]
    hrs_row_number: Optional[int]
    readable_row_number: Optional[int]
    present_in_hrs: bool
    present_in_readable: bool
    name_match_status: str
    MDRT_source_hrs_m: Optional[float]
    MDRT_source_readable_m: Optional[float]
    MDRT_agreement_readable_minus_hrs_m: Optional[float]
    mdrt_status: str
    TVDSS_source_readable_m: Optional[float]
    note_readable: str
    reconciliation_notes: str


# ---------------------------------------------------------------------------
# Corrected, auditable stratigraphic marker record (one per canonical
# marker per well - the project's corrected depth representation)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class TopMarkerRecord:
    """
    The complete, typed, auditable record for one canonical formation-top
    marker: every raw source value preserved separately, plus the
    project's corrected survey-derived depth and the explicit residual
    against the source-supplied TVDSS (where supplied).

    `mdrt_authority_basis` documents exactly how `MDRT_reconciled_m` (or
    its absence) was decided:
      * "hrs_and_readable_agree" - both sources supplied this marker and
        agreed within `TopReconciliationEntry` tolerance; the (identical,
        within tolerance) HRS value is used.
      * "hrs_only" / "readable_only" - only one source supplied this
        marker; that source's value is used, flagged as not cross-
        validated.
      * "disagreement_unresolved" - both sources supplied this marker but
        disagreed beyond tolerance; `MDRT_reconciled_m` is `None` and
        `mapping_status == "not_mapped_mdrt_unresolved"` - this project
        never silently picks one source's value in this case.

    `mapping_status` is one of `p2mem.top_models.VALID_MAPPING_STATUSES`;
    every field from `TVD_survey_m` onward is `None` unless
    `mapping_status == "mapped_within_coverage"`.
    """

    well_key: str
    canonical_marker_name: str
    MDRT_source_hrs_m: Optional[float]
    MDRT_source_readable_m: Optional[float]
    MDRT_reconciled_m: Optional[float]
    mdrt_authority_basis: str
    TVDSS_source_m: Optional[float]
    depth_basis_used: Optional[str]
    interpolation_method: Optional[str]
    TVD_survey_m: Optional[float]
    TVDSS_survey_corrected_m: Optional[float]
    TVDSS_residual_source_minus_survey_m: Optional[float]
    mapping_status: str
    well_identity_evidence_status: str
    notes: str


# ---------------------------------------------------------------------------
# Ingestion failure (typed, mirrors CheckshotIngestionFailure)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class TopIngestionFailure:
    """
    One well's failed formation-top ingestion attempt.

    `source_path` is retained for backward compatibility and always holds
    the single path judged most representative of the failure (the HRS
    path for an "hrs"-origin failure, the readable path for a
    "readable"-origin failure, the HRS path for a "reconciliation"-origin
    failure since both files parsed successfully in that case). It is
    never assumed to be the HRS path by construction - see `failure_origin`.

    `failure_origin` (Increment 5.1 Finding 2 fix) is one of
    `VALID_TOP_FAILURE_ORIGINS` and records which stage actually failed:
    "hrs" (HRS file parsing/reading/contract resolution), "readable" (the
    selected-readable file's own parsing/reading/contract resolution),
    "reconciliation" (both files parsed and contract-resolved, but could
    not be reconciled - e.g. `NO_COMMON_MARKERS`), or "mapping" (reserved;
    not currently raised as a fatal condition). "unknown" only appears for
    a `TopIngestionFailure` constructed without this classification.

    `hrs_path` / `readable_path` preserve BOTH source paths for this well,
    independent of which one actually failed, so every consumer that
    exports a failure message/context can sanitize both - never only the
    one path recorded in `source_path`.
    """

    well_key: str
    source_path: str
    error_type: str
    message: str
    exception: BaseException
    failure_origin: str = "unknown"
    hrs_path: Optional[str] = None
    readable_path: Optional[str] = None


# ---------------------------------------------------------------------------
# Combined per-well result
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class FormationTopWellResult:
    """
    The complete, typed result of successfully loading, contract-
    resolving, reconciling, and survey-mapping one well's pair of
    formation-top files.
    """

    well_key: str
    hrs_header: HRSTopHeaderInfo
    hrs_contract: HRSTopFileContract
    hrs_raw: HRSTopStationData
    readable_header: ReadableTopHeaderInfo
    readable_contract: ReadableTopFileContract
    readable_raw: ReadableTopStationData
    reconciliation: Tuple[TopReconciliationEntry, ...]
    markers: Tuple[TopMarkerRecord, ...]
    issues: Tuple[TopIngestionIssue, ...] = field(default_factory=tuple)
    contract_status: str = "PASSED"


# ---------------------------------------------------------------------------
# Data-availability record (e.g. Poseidon North 1, Proteus 1ST2 - a factual gap)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class FormationTopAvailabilityRecord:
    """
    Records that a project well has NO approved formation-top file, as a
    factual data gap - never as, or alongside, an ingestion failure, and
    never filled by substituting another well's tops or correlating them
    by depth alone.
    """

    well_key: str
    formation_top_availability: str  # "NOT_AVAILABLE"
    notes: str


#### `p2mem/io/tops.py` — formation-top parser, per-file contract resolver, HRS-versus-readable reconciliation, and survey-corrected depth mapping

In [ ]:
%%writefile p2mem/io/tops.py
"""
p2mem.io.tops - Auditable formation-top file ingestion, HRS-versus-readable
source reconciliation, and survey-corrected stratigraphic depth mapping
(Increment 5).

Why this module exists
------------------------
Each approved well (Poseidon 2, Boreas 1) has TWO independently supplied
formation-top files, in two structurally different formats:

* an "HRS" file (`Top_Name`, `MDRT_m` - tab-separated, no comments, no
  TVDSS, no well name anywhere in the file BODY - the well association
  rests on the filename alone, e.g.
  "Poseidon_2_HRS_tops_no_wellname_MDRT.txt". This is recorded honestly as
  `well_identity_source = "filename_only"`, never as content-verified
  identity.);
* a "selected readable" file (`#`-prefixed comment lines - including one
  naming the well - a blank line, a whitespace-padded
  `TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE` header, a dashed separator line,
  then whitespace-padded data rows). The comment naming the well is
  PROJECT-SUPPLIED METADATA embedded in the file, not independently
  verified file content - recorded as `well_identity_source =
  "in_file_comment_project_supplied"`, never as "verified".

STRUCTURAL PARSING (can the file be tokenized at all: is the expected
header/separator present, does every data row have the expected number of
numeric tokens, are all numeric tokens finite and non-negative) is kept
separate from CONTRACT RESOLUTION (does this specific file's actual
identity/schema/range agree with `config/formation_top_contracts.yml`) is
kept separate from SOURCE RECONCILIATION (do the two independently
supplied files for the SAME well agree with each other). A structural
defect raises `TopParsingError`; a contract mismatch raises
`TopContractError`; an unresolvable cross-file reconciliation problem
raises `TopSourceReconciliationError`. All three are always blocking.

Neither source file is ever treated as unconditionally authoritative. See
`reconcile_formation_top_sources` for the explicit name/MDRT reconciliation
policy, and `build_formation_top_markers` for the explicit MDRT-authority
and survey-mapping policy: the reconciled MDRT is mapped through the
LOCKED Increment 3.1.1 `petrel_source_trace` deviation-survey trajectory
using the UNMODIFIED, already-locked `p2mem.depth_mapping
.map_las_md_to_tvd_tvdss` function (this module never reimplements minimum
curvature or the TVD/TVDSS formula - that function is generic over ANY
1-D array of measured-depth values referenced to the same rotary-table
datum as the survey, not LAS-specific despite its parameter name; every
approved deviation-survey file's own header states "MD AND TVD ARE
REFERENCED (=0) AT WELL DATUM [RT]" - the identical zero/RT/downward-
increasing convention as this project's `MDRT_m` formation-top columns,
independently confirmed for both approved wells in
`INCREMENT_05_MANIFEST.md` Section 1).

Marker-name matching discipline (no silent fuzzy matching)
-------------------------------------------------------------
Two marker names are treated as the SAME canonical marker only if:

1. their raw strings are IDENTICAL (`name_match_status = "exact"`), or
2. they are identical after stripping leading/trailing whitespace and
   collapsing internal whitespace runs to a single space
   (`name_match_status = "normalized_match"` - this is the ONLY
   normalization applied, and it never changes letter case or
   punctuation), or
3. one of them appears, verbatim, as a key in the human-authored
   `marker_name_aliases` table in `config/formation_top_contracts.yml`,
   mapping it to the SAME canonical name the other side resolves to
   (`name_match_status = "aliased_match"`).

No similarity/fuzzy-distance matching of any kind is used. A marker name
that matches by none of the three rules above is recorded as present in
only one source (`name_match_status` = "missing_in_hrs" /
"missing_in_readable") - never silently paired with the nearest-looking
name from the other file.
"""

from __future__ import annotations

import hashlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import yaml

from p2mem.depth_mapping import DepthMappingError, ExtrapolationRejectedError, map_las_md_to_tvd_tvdss
from p2mem.deviation_models import DeviationWellResult
from p2mem.top_models import (
    VALID_IDENTITY_EVIDENCE_STATUSES,
    VALID_MODEL_USE_STATUSES,
    VALID_REPRESENTATION_TYPES,
    FormationTopWellResult,
    HRSTopFileContract,
    HRSTopHeaderInfo,
    HRSTopStationData,
    ReadableTopFileContract,
    ReadableTopHeaderInfo,
    ReadableTopStationData,
    TopIngestionFailure,
    TopIngestionIssue,
    TopMarkerRecord,
    TopReconciliationEntry,
)

__all__ = [
    "TopFileNotFoundError",
    "TopParsingError",
    "TopContractDefinitionError",
    "TopContractError",
    "TopSourceReconciliationError",
    "MDRT_AGREEMENT_TOLERANCE_M",
    "load_formation_top_contract_config",
    "parse_hrs_top_header",
    "read_hrs_top_rows",
    "parse_readable_top_header",
    "read_readable_top_rows",
    "resolve_hrs_top_contract",
    "resolve_readable_top_contract",
    "normalize_marker_name",
    "reconcile_formation_top_sources",
    "build_formation_top_markers",
    "load_formation_top_well",
    "load_formation_top_surveys",
]

_HRS_REQUIRED_COLUMN_COUNT = 2
_READABLE_REQUIRED_COLUMN_COUNT = 4

# The MDRT cross-check tolerance used when reconciling the two source
# files (Section 4 of the Increment 5 specification: "MDRT agreement").
# Both approved wells' HRS and readable files agree EXACTLY (0.00 m
# difference on every marker - independently confirmed in
# INCREMENT_05_MANIFEST.md Section 1); this tolerance absorbs only
# ordinary floating-point text round-trip noise (the files carry 2
# decimal places), never a genuinely different MDRT placement.
MDRT_AGREEMENT_TOLERANCE_M = 0.005


# ---------------------------------------------------------------------------
# Exceptions
# ---------------------------------------------------------------------------
class TopFileNotFoundError(FileNotFoundError):
    """A formation-top file path given to `load_formation_top_well` does not exist."""


class TopParsingError(ValueError):
    """
    A structural defect in a formation-top file, or in a caller-supplied
    in-memory formation-top data structure: a missing/malformed header or
    separator line, a data row without the expected number of tokens, a
    non-finite (NaN/Inf) or negative depth value, a duplicate canonical
    marker name within one source, a marker-order reversal within one
    source, an ambiguous-dtype (boolean/string/complex) numeric array, or
    a length mismatch between a station dataclass's name tuple and its
    numeric array(s). Raised before any contract or reconciliation step is
    reached (or, for a caller-supplied dataclass, before any downstream
    numerical use).
    """


class TopContractDefinitionError(ValueError):
    """
    `config/formation_top_contracts.yml` itself is malformed: a duplicate
    top-level key, a missing required field, an invalid type, an
    unsupported enum value, or an internally inconsistent expected-value
    set (e.g. min > max).
    """


class TopContractError(RuntimeError):
    """
    A specific file's actual parsed header/data does not agree with its
    contract: wrong filename, SHA-256 mismatch, header/schema mismatch,
    marker-count mismatch, or a numeric range outside the contract's
    declared tolerance. Carries `.issues` (the full tuple, ERROR and
    WARNING alike).
    """

    def __init__(self, message: str, issues: Tuple[TopIngestionIssue, ...]):
        super().__init__(message)
        self.issues = issues


class TopSourceReconciliationError(RuntimeError):
    """
    The HRS and selected-readable files for one well could not be
    reconciled at all: a duplicate canonical marker name within one
    source, a marker-order reversal within one source, or (a case never
    observed in either approved well's real data) zero canonical markers
    in common between the two sources. Carries `.issues`. A per-marker
    MDRT disagreement between otherwise-well-formed sources is NOT fatal -
    it is registered in the reconciliation table and that one marker is
    excluded from mapping (see `TopMarkerRecord.mdrt_authority_basis ==
    "disagreement_unresolved"`) without raising this exception.
    """

    def __init__(self, message: str, issues: Tuple[TopIngestionIssue, ...]):
        super().__init__(message)
        self.issues = issues


# ---------------------------------------------------------------------------
# Contract configuration (config/formation_top_contracts.yml)
# ---------------------------------------------------------------------------
class _NoDuplicateKeySafeLoader(yaml.SafeLoader):
    """
    A `yaml.SafeLoader` subclass that raises on a duplicate mapping key
    rather than silently keeping only the last occurrence. File-scoped to
    this module (mirrors, but does not import, the identical private
    pattern in `p2mem.io.checkshot` / `p2mem.io.deviation` - each of those
    classes is private to its own module and not exported for reuse).
    """


def _construct_mapping_no_duplicates(loader: yaml.SafeLoader, node, deep: bool = False):
    mapping: Dict = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise TopContractDefinitionError(
                f"Duplicate key {key!r} found while parsing formation-top contract YAML "
                f"(line {key_node.start_mark.line + 1}); duplicate contract keys are "
                f"rejected rather than silently keeping only the last one."
            )
        value = loader.construct_object(value_node, deep=deep)
        mapping[key] = value
    return mapping


_NoDuplicateKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _construct_mapping_no_duplicates
)

_COMMON_REQUIRED_FIELDS = (
    "expected_sha256",
    "project_well_key",
    "representation_type",
    "well_identity_evidence_status",
    "well_identity_evidence_notes",
    "model_use_status",
    "expected_column_header_line",
    "expected_column_order",
    "expected_marker_count",
    "expected_mdrt_min_m",
    "expected_mdrt_max_m",
    "numeric_range_tolerance",
    "datum_depth_column_interpretation",
    "notes",
)
_READABLE_ONLY_REQUIRED_FIELDS = (
    "expected_well_name_from_comment",
    "expected_tvdss_min_m",
    "expected_tvdss_max_m",
)


def _num(value, field_name: str, filename: str):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TopContractDefinitionError(f"{filename}: field {field_name!r} must be numeric, got {value!r}.")
    return float(value)


def _validate_common_fields(entry: dict, filename: str) -> None:
    missing = [f for f in _COMMON_REQUIRED_FIELDS if f not in entry]
    if missing:
        raise TopContractDefinitionError(f"{filename}: missing required field(s) {missing}.")
    if entry["representation_type"] not in VALID_REPRESENTATION_TYPES:
        raise TopContractDefinitionError(
            f"{filename}: representation_type must be one of {VALID_REPRESENTATION_TYPES}, "
            f"got {entry['representation_type']!r}."
        )
    if entry["well_identity_evidence_status"] not in VALID_IDENTITY_EVIDENCE_STATUSES:
        raise TopContractDefinitionError(
            f"{filename}: well_identity_evidence_status must be one of "
            f"{VALID_IDENTITY_EVIDENCE_STATUSES}, got {entry['well_identity_evidence_status']!r}."
        )
    if entry["model_use_status"] not in VALID_MODEL_USE_STATUSES:
        raise TopContractDefinitionError(
            f"{filename}: model_use_status must be one of {VALID_MODEL_USE_STATUSES}, "
            f"got {entry['model_use_status']!r}."
        )
    marker_count = entry["expected_marker_count"]
    if isinstance(marker_count, bool) or not isinstance(marker_count, int) or marker_count < 1:
        raise TopContractDefinitionError(f"{filename}: expected_marker_count must be a positive integer.")
    for f in (
        "expected_sha256", "project_well_key", "well_identity_evidence_notes",
        "expected_column_header_line", "datum_depth_column_interpretation", "notes",
    ):
        if not isinstance(entry[f], str) or not entry[f].strip():
            raise TopContractDefinitionError(f"{filename}: field {f!r} must be a non-empty string.")
    col_order = entry["expected_column_order"]
    if not isinstance(col_order, list) or not col_order or not all(isinstance(c, str) for c in col_order):
        raise TopContractDefinitionError(f"{filename}: expected_column_order must be a list of strings.")


def load_formation_top_contract_config(
    yaml_path: str,
) -> Tuple[Dict[str, HRSTopFileContract], Dict[str, ReadableTopFileContract], Dict[str, str]]:
    """
    Load and validate `config/formation_top_contracts.yml`, returning
    `(hrs_contracts, readable_contracts, marker_name_aliases)`:

    * `hrs_contracts` / `readable_contracts` are each keyed by exact
      source filename, containing only contracts of that representation
      type (a file listed under `representation_type:
      selected_readable_MDRT_TVDSS` never appears in `hrs_contracts`, and
      vice versa - this is enforced, not merely assumed).
    * `marker_name_aliases` is the flat, human-authored, project-wide
      alias table (raw marker-name string -> canonical marker-name
      string) from the YAML's top-level `marker_name_aliases` key
      (`{}` if absent - both approved wells' real data need zero aliases;
      see `reconcile_formation_top_sources`).

    Raises `TopContractDefinitionError` for any structural or internal-
    consistency problem, before any formation-top file is opened.
    """
    with open(yaml_path, "r", encoding="utf-8") as fh:
        raw = yaml.load(fh, Loader=_NoDuplicateKeySafeLoader)

    if not isinstance(raw, dict) or "files" not in raw:
        raise TopContractDefinitionError(f"{yaml_path}: top-level YAML must be a mapping with a 'files' key.")
    files_section = raw["files"]
    if not isinstance(files_section, dict) or not files_section:
        raise TopContractDefinitionError(f"{yaml_path}: 'files' must be a non-empty mapping.")

    aliases_section = raw.get("marker_name_aliases", {}) or {}
    if not isinstance(aliases_section, dict) or not all(
        isinstance(k, str) and isinstance(v, str) for k, v in aliases_section.items()
    ):
        raise TopContractDefinitionError(f"{yaml_path}: 'marker_name_aliases' must be a mapping of string to string.")

    hrs_contracts: Dict[str, HRSTopFileContract] = {}
    readable_contracts: Dict[str, ReadableTopFileContract] = {}

    for filename, entry in files_section.items():
        if not isinstance(entry, dict):
            raise TopContractDefinitionError(f"{filename}: contract entry must be a mapping.")
        _validate_common_fields(entry, filename)

        mdrt_min = _num(entry["expected_mdrt_min_m"], "expected_mdrt_min_m", filename)
        mdrt_max = _num(entry["expected_mdrt_max_m"], "expected_mdrt_max_m", filename)
        tolerance = _num(entry["numeric_range_tolerance"], "numeric_range_tolerance", filename)
        if tolerance < 0.0:
            raise TopContractDefinitionError(f"{filename}: numeric_range_tolerance must be >= 0.")
        if mdrt_min > mdrt_max:
            raise TopContractDefinitionError(f"{filename}: expected_mdrt_min_m > expected_mdrt_max_m.")

        if entry["representation_type"] == "HRS_MDRT_only":
            if len(entry["expected_column_order"]) != _HRS_REQUIRED_COLUMN_COUNT:
                raise TopContractDefinitionError(
                    f"{filename}: HRS expected_column_order must have exactly "
                    f"{_HRS_REQUIRED_COLUMN_COUNT} entries."
                )
            hrs_contracts[filename] = HRSTopFileContract(
                source_filename=filename,
                expected_sha256=entry["expected_sha256"],
                project_well_key=entry["project_well_key"],
                representation_type=entry["representation_type"],
                well_identity_evidence_status=entry["well_identity_evidence_status"],
                well_identity_evidence_notes=entry["well_identity_evidence_notes"],
                model_use_status=entry["model_use_status"],
                expected_column_header_line=entry["expected_column_header_line"],
                expected_column_order=tuple(entry["expected_column_order"]),
                expected_column_count=_HRS_REQUIRED_COLUMN_COUNT,
                expected_marker_count=entry["expected_marker_count"],
                expected_mdrt_min_m=mdrt_min,
                expected_mdrt_max_m=mdrt_max,
                numeric_range_tolerance=tolerance,
                datum_depth_column_interpretation=entry["datum_depth_column_interpretation"],
                notes=entry["notes"],
            )
        elif entry["representation_type"] == "selected_readable_MDRT_TVDSS":
            missing = [f for f in _READABLE_ONLY_REQUIRED_FIELDS if f not in entry]
            if missing:
                raise TopContractDefinitionError(f"{filename}: missing required field(s) {missing}.")
            if len(entry["expected_column_order"]) != _READABLE_REQUIRED_COLUMN_COUNT:
                raise TopContractDefinitionError(
                    f"{filename}: readable expected_column_order must have exactly "
                    f"{_READABLE_REQUIRED_COLUMN_COUNT} entries."
                )
            tvdss_min = _num(entry["expected_tvdss_min_m"], "expected_tvdss_min_m", filename)
            tvdss_max = _num(entry["expected_tvdss_max_m"], "expected_tvdss_max_m", filename)
            if tvdss_min > tvdss_max:
                raise TopContractDefinitionError(f"{filename}: expected_tvdss_min_m > expected_tvdss_max_m.")
            if not isinstance(entry["expected_well_name_from_comment"], str) or not entry[
                "expected_well_name_from_comment"
            ].strip():
                raise TopContractDefinitionError(
                    f"{filename}: expected_well_name_from_comment must be a non-empty string."
                )
            readable_contracts[filename] = ReadableTopFileContract(
                source_filename=filename,
                expected_sha256=entry["expected_sha256"],
                project_well_key=entry["project_well_key"],
                representation_type=entry["representation_type"],
                well_identity_evidence_status=entry["well_identity_evidence_status"],
                well_identity_evidence_notes=entry["well_identity_evidence_notes"],
                model_use_status=entry["model_use_status"],
                expected_well_name_from_comment=entry["expected_well_name_from_comment"],
                expected_column_header_line=entry["expected_column_header_line"],
                expected_column_order=tuple(entry["expected_column_order"]),
                expected_column_count=_READABLE_REQUIRED_COLUMN_COUNT,
                expected_marker_count=entry["expected_marker_count"],
                expected_mdrt_min_m=mdrt_min,
                expected_mdrt_max_m=mdrt_max,
                expected_tvdss_min_m=tvdss_min,
                expected_tvdss_max_m=tvdss_max,
                numeric_range_tolerance=tolerance,
                datum_depth_column_interpretation=entry["datum_depth_column_interpretation"],
                notes=entry["notes"],
            )
        else:  # pragma: no cover - already validated above
            raise TopContractDefinitionError(f"{filename}: unrecognized representation_type.")

    return hrs_contracts, readable_contracts, dict(aliases_section)


# ---------------------------------------------------------------------------
# Structural parsing - HRS format
# ---------------------------------------------------------------------------
def parse_hrs_top_header(path: str) -> HRSTopHeaderInfo:
    """
    Parse the single-line HRS header: one tab-separated column-header line
    (`Top_Name`, `MDRT_m`). Raises `TopFileNotFoundError` if the path does
    not exist, or `TopParsingError` if no header line is present.
    """
    p = Path(path)
    if not p.exists():
        raise TopFileNotFoundError(f"Formation-top (HRS) file not found: {path}")

    raw_bytes = p.read_bytes()
    sha256 = hashlib.sha256(raw_bytes).hexdigest()
    line_ending = "CRLF" if b"\r\n" in raw_bytes else ("LF" if b"\n" in raw_bytes else "NONE")

    text = raw_bytes.decode("utf-8")
    lines = text.splitlines()
    if len(lines) < 1 or not lines[0].strip():
        raise TopParsingError(f"{path}: expected a non-empty column-header line as the first line.")
    column_header_line = lines[0]
    column_names = tuple(column_header_line.split("\t"))
    if len(column_names) != _HRS_REQUIRED_COLUMN_COUNT:
        raise TopParsingError(
            f"{path}: column-header line must have exactly {_HRS_REQUIRED_COLUMN_COUNT} "
            f"tab-separated fields; found {len(column_names)} in {column_header_line!r}."
        )

    return HRSTopHeaderInfo(
        source_filename=p.name,
        sha256=sha256,
        column_header_line=column_header_line,
        column_names=column_names,
        well_identity_source="filename_only",
        line_ending_convention=line_ending,
        header_line_count=1,
        data_line_offset=1,
    )


def read_hrs_top_rows(path: str) -> HRSTopStationData:
    """
    Parse the data rows (all non-empty lines after the 1-line header) into
    raw `HRSTopStationData`, in original file order. A blank line is
    permitted only as a trailing end-of-file artifact; a blank line found
    before the last non-empty line is a structural defect. Every MDRT
    value must be finite and non-negative (a negative measured depth below
    rotary table has no physical meaning for a downhole marker).
    """
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    data_lines = lines[1:]

    last_nonblank = -1
    for i, ln in enumerate(data_lines):
        if ln.strip():
            last_nonblank = i
    if last_nonblank < 0:
        raise TopParsingError(f"{path}: no data rows found after the 1-line header.")

    names: List[str] = []
    mdrt_vals: List[float] = []
    for i, ln in enumerate(data_lines[: last_nonblank + 1]):
        if not ln.strip():
            raise TopParsingError(
                f"{path}: blank line found at data row {i + 1} before the last data row; "
                f"only a trailing blank line (end-of-file artifact) is permitted."
            )
        parts = ln.split("\t")
        if len(parts) != _HRS_REQUIRED_COLUMN_COUNT:
            raise TopParsingError(
                f"{path}: data row {i + 1} has {len(parts)} tab-separated field(s), "
                f"expected {_HRS_REQUIRED_COLUMN_COUNT}: {ln!r}"
            )
        name = parts[0]
        try:
            mdrt = float(parts[1])
        except ValueError as exc:
            raise TopParsingError(f"{path}: data row {i + 1} contains a non-numeric MDRT token: {ln!r} ({exc})") from exc
        if not np.isfinite(mdrt):
            raise TopParsingError(f"{path}: data row {i + 1} contains a non-finite MDRT value: {ln!r}")
        if mdrt < 0.0:
            raise TopParsingError(f"{path}: data row {i + 1} contains a negative MDRT value: {ln!r}")
        names.append(name)
        mdrt_vals.append(mdrt)

    return HRSTopStationData(
        Top_Name_source=tuple(names),
        MDRT_source_m=np.asarray(mdrt_vals, dtype=np.float64),
    )


# ---------------------------------------------------------------------------
# Structural parsing - selected-readable format
# ---------------------------------------------------------------------------
def parse_readable_top_header(path: str) -> ReadableTopHeaderInfo:
    """
    Parse the "selected readable" header: one or more leading `#`-prefixed
    comment lines, a blank line, a whitespace-padded column-header line
    (`TOP_NAME`, `MDRT_M`, `TVDSS_M`, `NOTE`), and a dashed separator
    line - each recognized structurally and deliberately, never
    accidentally treated as a data row. Raises `TopParsingError` if any of
    these structural elements is missing.
    """
    p = Path(path)
    if not p.exists():
        raise TopFileNotFoundError(f"Formation-top (selected readable) file not found: {path}")

    raw_bytes = p.read_bytes()
    sha256 = hashlib.sha256(raw_bytes).hexdigest()
    line_ending = "CRLF" if b"\r\n" in raw_bytes else ("LF" if b"\n" in raw_bytes else "NONE")

    text = raw_bytes.decode("utf-8")
    lines = text.splitlines()

    comment_lines: List[str] = []
    idx = 0
    while idx < len(lines) and lines[idx].startswith("#"):
        comment_lines.append(lines[idx])
        idx += 1
    if not comment_lines:
        raise TopParsingError(f"{path}: expected at least one leading '#'-prefixed comment line.")

    well_name_from_comment: Optional[str] = None
    for cl in comment_lines:
        marker = "Selected readable well tops for "
        if marker in cl:
            well_name_from_comment = cl.split(marker, 1)[1].strip()
            break

    # Skip exactly one blank line separating the comment block from the header.
    if idx >= len(lines) or lines[idx].strip():
        raise TopParsingError(f"{path}: expected a blank line after the comment block (line {idx + 1}).")
    idx += 1

    if idx >= len(lines) or not lines[idx].strip().startswith("TOP_NAME"):
        raise TopParsingError(f"{path}: expected the column-header line (starting 'TOP_NAME') at line {idx + 1}.")
    column_header_line = lines[idx]
    column_names = tuple(c.strip() for c in column_header_line.split() if c.strip())
    # The header line's own fields are whitespace-padded/aligned, not
    # tab-separated - split() on any whitespace run is the correct,
    # deliberate tokenization here (never accidentally merging TOP_NAME
    # tokens that themselves contain a single space, since the header only
    # contains the 4 declared column names).
    if len(column_names) != _READABLE_REQUIRED_COLUMN_COUNT:
        raise TopParsingError(
            f"{path}: column-header line must tokenize to exactly {_READABLE_REQUIRED_COLUMN_COUNT} "
            f"fields; found {len(column_names)} in {column_header_line!r}."
        )
    idx += 1

    if idx >= len(lines) or not lines[idx].strip() or set(lines[idx].strip()) != {"-"}:
        raise TopParsingError(f"{path}: expected a dashed separator line (all '-' characters) at line {idx + 1}.")
    separator_line_raw = lines[idx]
    idx += 1

    return ReadableTopHeaderInfo(
        source_filename=p.name,
        sha256=sha256,
        comment_lines=tuple(comment_lines),
        well_name_from_comment=well_name_from_comment,
        column_header_line=column_header_line,
        column_names=column_names,
        separator_line_raw=separator_line_raw,
        well_identity_source="in_file_comment_project_supplied",
        line_ending_convention=line_ending,
        header_line_count=idx,
        data_line_offset=idx,
    )


def read_readable_top_rows(path: str) -> ReadableTopStationData:
    """
    Parse the data rows (every non-empty line after the header/separator
    block) into raw `ReadableTopStationData`, in original file order.
    Fields are whitespace-delimited, but `TOP_NAME` may itself contain
    single internal spaces (e.g. "Sea Bed") - this function tokenizes each
    data line by splitting on 2-or-more-space runs / tabs (the file's own
    fixed-width column alignment), never on every single space, so a
    multi-word marker name is never accidentally split across fields.
    `NOTE` may be empty (preserved as `""`, never fabricated).
    """
    import re

    header = parse_readable_top_header(path)
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    data_lines = lines[header.data_line_offset :]

    last_nonblank = -1
    for i, ln in enumerate(data_lines):
        if ln.strip():
            last_nonblank = i
    if last_nonblank < 0:
        raise TopParsingError(f"{path}: no data rows found after the header/separator block.")

    field_splitter = re.compile(r"\t|  +")  # a tab, or 2-or-more consecutive spaces

    names: List[str] = []
    mdrt_vals: List[float] = []
    tvdss_vals: List[float] = []
    notes: List[str] = []
    for i, ln in enumerate(data_lines[: last_nonblank + 1]):
        if not ln.strip():
            raise TopParsingError(
                f"{path}: blank line found at data row {i + 1} before the last data row; "
                f"only a trailing blank line (end-of-file artifact) is permitted."
            )
        parts = [p.strip() for p in field_splitter.split(ln.rstrip("\n\r"))]
        parts = [p for p in parts if p != ""]
        if len(parts) < 3:
            raise TopParsingError(
                f"{path}: data row {i + 1} could not be tokenized into at least "
                f"TOP_NAME/MDRT_M/TVDSS_M fields: {ln!r}"
            )
        name = parts[0]
        try:
            mdrt = float(parts[1])
            tvdss = float(parts[2])
        except ValueError as exc:
            raise TopParsingError(f"{path}: data row {i + 1} contains a non-numeric MDRT/TVDSS token: {ln!r} ({exc})") from exc
        if not (np.isfinite(mdrt) and np.isfinite(tvdss)):
            raise TopParsingError(f"{path}: data row {i + 1} contains a non-finite MDRT/TVDSS value: {ln!r}")
        if mdrt < 0.0:
            raise TopParsingError(f"{path}: data row {i + 1} contains a negative MDRT value: {ln!r}")
        note = parts[3] if len(parts) >= 4 else ""
        names.append(name)
        mdrt_vals.append(mdrt)
        tvdss_vals.append(tvdss)
        notes.append(note)

    return ReadableTopStationData(
        TOP_NAME_source=tuple(names),
        MDRT_source_m=np.asarray(mdrt_vals, dtype=np.float64),
        TVDSS_source_m=np.asarray(tvdss_vals, dtype=np.float64),
        NOTE_source=tuple(notes),
    )


# ---------------------------------------------------------------------------
# Contract resolution
# ---------------------------------------------------------------------------
def resolve_hrs_top_contract(
    header: HRSTopHeaderInfo, stations: HRSTopStationData, contract: HRSTopFileContract
) -> Tuple[TopIngestionIssue, ...]:
    """Compare the actual parsed HRS header/data against `contract`."""
    issues: List[TopIngestionIssue] = []

    def err(code: str, message: str) -> None:
        issues.append(TopIngestionIssue("ERROR", code, message, header.source_filename))

    def warn(code: str, message: str) -> None:
        issues.append(TopIngestionIssue("WARNING", code, message, header.source_filename))

    if header.source_filename != contract.source_filename:
        err("FILENAME_MISMATCH", f"Actual filename {header.source_filename!r} does not match contract key {contract.source_filename!r}.")
    if header.sha256 != contract.expected_sha256:
        err("SHA256_MISMATCH", f"Actual SHA-256 {header.sha256!r} does not match expected {contract.expected_sha256!r}.")
    if header.column_header_line != contract.expected_column_header_line:
        err("COLUMN_HEADER_LINE_MISMATCH", f"Actual column-header line {header.column_header_line!r} does not match expected {contract.expected_column_header_line!r}.")
    if header.column_names != contract.expected_column_order:
        err("COLUMN_ORDER_MISMATCH", f"Actual column order {header.column_names!r} does not match expected {contract.expected_column_order!r}.")

    n_markers = len(stations.Top_Name_source)
    if n_markers != contract.expected_marker_count:
        err("MARKER_COUNT_MISMATCH", f"Actual marker count {n_markers} does not match expected {contract.expected_marker_count}.")

    if n_markers:
        tol = contract.numeric_range_tolerance
        actual_min, actual_max = float(stations.MDRT_source_m.min()), float(stations.MDRT_source_m.max())
        if abs(actual_min - contract.expected_mdrt_min_m) > tol or abs(actual_max - contract.expected_mdrt_max_m) > tol:
            err("NUMERIC_RANGE_MISMATCH", f"MDRT_source_m: actual range [{actual_min}, {actual_max}] does not match expected [{contract.expected_mdrt_min_m}, {contract.expected_mdrt_max_m}] within tolerance {tol}.")

    warn(
        "WELL_IDENTITY_FILENAME_ONLY",
        f"This HRS file carries NO well name in its own body; association with project well "
        f"{contract.project_well_key!r} rests on the filename alone. {contract.well_identity_evidence_notes} "
        f"Never described as content-verified identity.",
    )
    return tuple(issues)


def resolve_readable_top_contract(
    header: ReadableTopHeaderInfo, stations: ReadableTopStationData, contract: ReadableTopFileContract
) -> Tuple[TopIngestionIssue, ...]:
    """Compare the actual parsed selected-readable header/data against `contract`."""
    issues: List[TopIngestionIssue] = []

    def err(code: str, message: str) -> None:
        issues.append(TopIngestionIssue("ERROR", code, message, header.source_filename))

    def warn(code: str, message: str) -> None:
        issues.append(TopIngestionIssue("WARNING", code, message, header.source_filename))

    if header.source_filename != contract.source_filename:
        err("FILENAME_MISMATCH", f"Actual filename {header.source_filename!r} does not match contract key {contract.source_filename!r}.")
    if header.sha256 != contract.expected_sha256:
        err("SHA256_MISMATCH", f"Actual SHA-256 {header.sha256!r} does not match expected {contract.expected_sha256!r}.")
    if header.well_name_from_comment != contract.expected_well_name_from_comment:
        err("WELL_NAME_COMMENT_MISMATCH", f"Actual in-file comment well name {header.well_name_from_comment!r} does not match expected {contract.expected_well_name_from_comment!r}.")
    if header.column_names != contract.expected_column_order:
        err("COLUMN_ORDER_MISMATCH", f"Actual column order {header.column_names!r} does not match expected {contract.expected_column_order!r}.")

    n_markers = len(stations.TOP_NAME_source)
    if n_markers != contract.expected_marker_count:
        err("MARKER_COUNT_MISMATCH", f"Actual marker count {n_markers} does not match expected {contract.expected_marker_count}.")

    if n_markers:
        tol = contract.numeric_range_tolerance
        for label, arr, exp_min, exp_max in (
            ("MDRT_source_m", stations.MDRT_source_m, contract.expected_mdrt_min_m, contract.expected_mdrt_max_m),
            ("TVDSS_source_m", stations.TVDSS_source_m, contract.expected_tvdss_min_m, contract.expected_tvdss_max_m),
        ):
            actual_min, actual_max = float(arr.min()), float(arr.max())
            if abs(actual_min - exp_min) > tol or abs(actual_max - exp_max) > tol:
                err("NUMERIC_RANGE_MISMATCH", f"{label}: actual range [{actual_min}, {actual_max}] does not match expected [{exp_min}, {exp_max}] within tolerance {tol}.")

    warn(
        "WELL_IDENTITY_IN_FILE_COMMENT_PROJECT_SUPPLIED",
        f"This file's well name is stated in a leading '#' comment ({header.well_name_from_comment!r}) - "
        f"project-supplied metadata embedded in the file, NOT independently verified file content. "
        f"{contract.well_identity_evidence_notes} Never described as content-verified identity.",
    )
    warn(
        "SUPPLIED_TVDSS_NOT_CORRECTED_DEPTH",
        "This file's own TVDSS_M column is preserved as TVDSS_source_m for residual/QC comparison "
        "only - it is never treated as this project's corrected stratigraphic depth. The corrected "
        "depth is always TVDSS_survey_corrected_m, computed by mapping the reconciled MDRT through "
        "the locked petrel_source_trace survey (see p2mem.io.tops module docstring).",
    )
    return tuple(issues)


# ---------------------------------------------------------------------------
# Marker-name normalization (no fuzzy matching - see module docstring)
# ---------------------------------------------------------------------------
def normalize_marker_name(name: str) -> str:
    """Strip leading/trailing whitespace and collapse internal whitespace
    runs to a single space. Never changes letter case or punctuation."""
    return " ".join(name.split())


def _reject_ambiguous_dtype(raw: np.ndarray, context: str) -> None:
    """
    Local, documented copy of `p2mem.units`'s identical private dtype
    check (`p2mem/units.py` itself remains LOCKED and unmodified - see
    `p2mem.time_depth._reject_ambiguous_dtype` for the identical, already-
    established Increment 4.1.1 precedent for this exact pattern). Rejects
    boolean, string/bytes, and complex dtype input with `TypeError` before
    any numeric use; only integer- or floating-dtype arrays are accepted.
    """
    kind = raw.dtype.kind
    if kind == "b":
        raise TypeError(f"{context}: boolean input is not accepted as a numeric quantity.")
    if kind in ("U", "S"):
        raise TypeError(f"{context}: string/bytes input is not accepted as a numeric quantity.")
    if kind == "c":
        raise TypeError(f"{context}: complex input is not accepted as a numeric quantity.")
    if kind not in ("i", "u", "f"):
        raise TypeError(f"{context}: unsupported array dtype {raw.dtype!r} for a numeric quantity.")


def _validate_name_tuple(names: Tuple, label: str, well_key: str) -> None:
    """
    Increment 5.1 (Finding 3 fix): every element of a marker-name or note
    tuple must be a genuine `str` - never silently accepted as a bare
    length-N sequence of arbitrary objects. Raises `TopParsingError` (a
    structural/type defect in caller-supplied in-memory data, per this
    module's documented `TopParsingError` scope), never an incidental
    `TypeError` from a downstream string operation.
    """
    for i, n in enumerate(names):
        if not isinstance(n, str):
            raise TopParsingError(f"{well_key}: {label} element at index {i} must be a string, got {type(n).__name__!r}.")


def _validate_mdrt_agreement_tolerance(value, well_key: str) -> float:
    """
    Increment 5.1 (Finding 3 fix): explicitly validate the
    `mdrt_agreement_tolerance_m` keyword before it is used in any
    numerical comparison. Boolean, string/bytes, and complex values are
    rejected with `TypeError` (mirrors `_reject_ambiguous_dtype`'s
    type-class-versus-value-defect split); NaN, Inf, and negative values
    are rejected with `TopParsingError` (a value defect, not a type
    defect). Valid Python `int`/`float` and NumPy integer/floating
    scalars (including 0-d/size-1 NumPy arrays) are accepted and returned
    as a plain Python `float`.
    """
    if isinstance(value, bool):
        raise TypeError(f"{well_key}: mdrt_agreement_tolerance_m must not be boolean, got {value!r}.")
    if isinstance(value, (str, bytes)):
        raise TypeError(f"{well_key}: mdrt_agreement_tolerance_m must not be a string/bytes value, got {value!r}.")
    if isinstance(value, complex):
        raise TypeError(f"{well_key}: mdrt_agreement_tolerance_m must not be complex, got {value!r}.")
    if isinstance(value, np.ndarray):
        if value.size != 1:
            raise TypeError(
                f"{well_key}: mdrt_agreement_tolerance_m must be a scalar, got an array of size {value.size}."
            )
        _reject_ambiguous_dtype(value, f"{well_key}: mdrt_agreement_tolerance_m")
        value = float(value.reshape(()))
    elif isinstance(value, (np.integer, np.floating)):
        value = float(value)
    elif isinstance(value, (int, float)):
        value = float(value)
    else:
        raise TypeError(
            f"{well_key}: mdrt_agreement_tolerance_m must be a real numeric scalar, got {type(value).__name__!r}."
        )
    if not np.isfinite(value):
        raise TopParsingError(f"{well_key}: mdrt_agreement_tolerance_m must be finite, got {value!r}.")
    if value < 0.0:
        raise TopParsingError(f"{well_key}: mdrt_agreement_tolerance_m must be non-negative, got {value!r}.")
    return value


# ---------------------------------------------------------------------------
# HRS-versus-readable source reconciliation
# ---------------------------------------------------------------------------
def reconcile_formation_top_sources(
    well_key: str,
    hrs_raw: HRSTopStationData,
    readable_raw: ReadableTopStationData,
    marker_name_aliases: Dict[str, str],
    *,
    mdrt_agreement_tolerance_m: float = MDRT_AGREEMENT_TOLERANCE_M,
) -> Tuple[Tuple[TopIngestionIssue, ...], Tuple[TopReconciliationEntry, ...]]:
    """
    Reconcile one well's HRS and selected-readable formation-top sources.
    See the module docstring for the exact/normalized/aliased marker-name
    matching policy (no fuzzy matching). Returns `(issues, entries)` -
    never raises for an ordinary per-marker MDRT disagreement (that is
    registered in the returned entries); raises `TopParsingError` for a
    malformed/ambiguous input array, and lets the caller
    (`load_formation_top_well`) decide whether an ERROR-severity issue
    (duplicate canonical name, marker-order reversal, or zero markers in
    common) is fatal via `TopSourceReconciliationError`.
    """
    # ------------------------------------------------------------------
    # Increment 5.1 (Finding 3 fix): full structural/type/value
    # validation of caller-supplied in-memory data, enforced BEFORE any
    # reconciliation or numerical comparison. The file parsers
    # (`read_hrs_top_rows` / `read_readable_top_rows`) already enforce
    # every one of these checks on file-derived data - this closes the
    # gap for this public in-memory API, which a caller/test can invoke
    # directly with hand-built dataclasses. Deliberate, typed exceptions
    # only: never an incidental IndexError (mismatched NOTE_source
    # length), NumPy broadcasting error, or bare comparison TypeError.
    # ------------------------------------------------------------------
    hrs_mdrt = hrs_raw.MDRT_source_m
    readable_mdrt = readable_raw.MDRT_source_m
    readable_tvdss = readable_raw.TVDSS_source_m

    for arr, label in (
        (hrs_mdrt, "HRS MDRT_source_m"),
        (readable_mdrt, "readable MDRT_source_m"),
        (readable_tvdss, "readable TVDSS_source_m"),
    ):
        if not isinstance(arr, np.ndarray):
            raise TopParsingError(f"{well_key}: {label} must be a NumPy array, got {type(arr).__name__!r}.")
        if arr.ndim != 1:
            raise TopParsingError(f"{well_key}: {label} must be 1-dimensional, got ndim={arr.ndim}.")

    _reject_ambiguous_dtype(hrs_mdrt, f"{well_key}: HRS MDRT_source_m")
    _reject_ambiguous_dtype(readable_mdrt, f"{well_key}: readable MDRT_source_m")
    _reject_ambiguous_dtype(readable_tvdss, f"{well_key}: readable TVDSS_source_m")

    if len(hrs_raw.Top_Name_source) != hrs_mdrt.size:
        raise TopParsingError(f"{well_key}: HRS Top_Name_source length does not match MDRT_source_m length.")
    _readable_lengths = {
        len(readable_raw.TOP_NAME_source),
        int(readable_mdrt.size),
        int(readable_tvdss.size),
        len(readable_raw.NOTE_source),
    }
    if len(_readable_lengths) != 1:
        raise TopParsingError(
            f"{well_key}: readable TOP_NAME_source/MDRT_source_m/TVDSS_source_m/NOTE_source "
            f"lengths do not match."
        )

    _validate_name_tuple(hrs_raw.Top_Name_source, "HRS Top_Name_source", well_key)
    _validate_name_tuple(readable_raw.TOP_NAME_source, "readable TOP_NAME_source", well_key)
    _validate_name_tuple(readable_raw.NOTE_source, "readable NOTE_source", well_key)

    if hrs_mdrt.size and not np.all(np.isfinite(hrs_mdrt)):
        raise TopParsingError(f"{well_key}: HRS MDRT_source_m contains a non-finite (NaN/Inf) value.")
    if hrs_mdrt.size and np.any(hrs_mdrt < 0.0):
        raise TopParsingError(f"{well_key}: HRS MDRT_source_m contains a negative value.")
    if readable_mdrt.size and not np.all(np.isfinite(readable_mdrt)):
        raise TopParsingError(f"{well_key}: readable MDRT_source_m contains a non-finite (NaN/Inf) value.")
    if readable_mdrt.size and np.any(readable_mdrt < 0.0):
        raise TopParsingError(f"{well_key}: readable MDRT_source_m contains a negative value.")
    if readable_tvdss.size and not np.all(np.isfinite(readable_tvdss)):
        raise TopParsingError(f"{well_key}: readable TVDSS_source_m contains a non-finite (NaN/Inf) value.")

    mdrt_agreement_tolerance_m = _validate_mdrt_agreement_tolerance(mdrt_agreement_tolerance_m, well_key)

    issues: List[TopIngestionIssue] = []

    def err(code: str, message: str) -> None:
        issues.append(TopIngestionIssue("ERROR", code, message, well_key))

    def warn(code: str, message: str) -> None:
        issues.append(TopIngestionIssue("WARNING", code, message, well_key))

    # --- Canonical-name resolution: raw string as an alias key first,
    # then its whitespace-normalized form as an alias key, then the
    # normalized form itself. No fuzzy/similarity matching of any kind. ---
    def _canon(name: str) -> str:
        if name in marker_name_aliases:
            return marker_name_aliases[name]
        norm = normalize_marker_name(name)
        return marker_name_aliases.get(norm, norm)

    # --- Structural per-source QC: duplicate canonical names, ordering ---
    def _check_source(names: Tuple[str, ...], mdrt: np.ndarray, label: str) -> None:
        seen: Dict[str, int] = {}
        for i, n in enumerate(names):
            canon = _canon(n)
            if canon in seen:
                err(
                    "DUPLICATE_MARKER_NAME",
                    f"{label} row {i}: canonical marker name {canon!r} duplicates row {seen[canon]} "
                    f"in the same source - never silently merged.",
                )
            else:
                seen[canon] = i
        if mdrt.size >= 2 and not np.all(np.diff(mdrt) > 0.0):
            err(
                "MARKER_ORDER_REVERSAL",
                f"{label}: MDRT_m is not strictly increasing in file row order "
                f"(values: {mdrt.tolist()}) - a stratigraphic marker order reversal "
                f"is never silently sorted or accepted.",
            )

    _check_source(hrs_raw.Top_Name_source, hrs_raw.MDRT_source_m, "HRS")
    _check_source(readable_raw.TOP_NAME_source, readable_raw.MDRT_source_m, "readable")

    # --- Build canonical marker order: HRS order first, then any
    # readable-only markers appended in readable file order. Deterministic,
    # traceable to file order - never alphabetically re-sorted. ---
    hrs_by_canon: Dict[str, Tuple[int, str]] = {}
    for i, n in enumerate(hrs_raw.Top_Name_source):
        hrs_by_canon.setdefault(_canon(n), (i, n))
    readable_by_canon: Dict[str, Tuple[int, str]] = {}
    for i, n in enumerate(readable_raw.TOP_NAME_source):
        readable_by_canon.setdefault(_canon(n), (i, n))

    canonical_order: List[str] = list(hrs_by_canon.keys())
    for c in readable_by_canon.keys():
        if c not in hrs_by_canon:
            canonical_order.append(c)

    # Increment 5.1 (Finding 1 fix, further corrected in Increment 5.1.1):
    # the ERROR condition documented by `TopSourceReconciliationError`
    # ("zero canonical markers in common between the two sources") is the
    # INTERSECTION of the two sources' canonical marker names being empty
    # - never the emptiness of `canonical_order` above, which is a UNION.
    # Increment 5.1 gated this on "both sources non-empty AND intersection
    # empty" (plus a separate both-empty case), which left one remaining
    # gap: exactly ONE source empty, the other non-empty - the
    # intersection of an empty set with anything is itself empty, so this
    # is still a zero-common-markers condition, but the 5.1 condition's
    # `hrs_by_canon and readable_by_canon` guard (both dicts truthy/non-
    # empty) skipped it. Increment 5.1.1 replaces the whole condition with
    # the single, strictly equivalent-or-stronger check the intersection
    # itself: `not common_markers` is empty in EVERY zero-common-markers
    # case (both empty, either side alone empty, or both non-empty and
    # disjoint) and is never empty whenever at least one canonical marker
    # is genuinely shared - so a legitimate one-sided marker (HRS-only or
    # readable-only, alongside at least one shared marker) remains the
    # pre-existing, non-fatal NOT_COMPARABLE case, unaffected.
    common_markers = set(hrs_by_canon) & set(readable_by_canon)
    if not common_markers:
        err(
            "NO_COMMON_MARKERS",
            f"{well_key}: HRS and readable sources share zero canonical markers in common "
            f"(HRS canonical markers: {len(hrs_by_canon)}, readable canonical markers: "
            f"{len(readable_by_canon)}) - this includes the case where one source is "
            f"entirely empty and the other is not; a marker present in only one source is "
            f"legitimate ONLY when at least one other canonical marker is genuinely shared "
            f"between both sources, never merely tolerated as a bare one-sided set.",
        )

    entries: List[TopReconciliationEntry] = []
    for canon in canonical_order:
        in_hrs = canon in hrs_by_canon
        in_readable = canon in readable_by_canon
        hrs_row, hrs_name_raw = hrs_by_canon.get(canon, (None, None))
        readable_row, readable_name_raw = readable_by_canon.get(canon, (None, None))

        mdrt_hrs = float(hrs_raw.MDRT_source_m[hrs_row]) if in_hrs else None
        mdrt_readable = float(readable_raw.MDRT_source_m[readable_row]) if in_readable else None
        tvdss_readable = float(readable_raw.TVDSS_source_m[readable_row]) if in_readable else None
        note_readable = readable_raw.NOTE_source[readable_row] if in_readable else ""

        if in_hrs and in_readable:
            if hrs_name_raw == readable_name_raw:
                name_match_status = "exact"
            elif canon in marker_name_aliases.values() and (
                hrs_name_raw in marker_name_aliases or readable_name_raw in marker_name_aliases
            ):
                name_match_status = "aliased_match"
            else:
                name_match_status = "normalized_match"
            agreement = mdrt_readable - mdrt_hrs
            if abs(agreement) <= mdrt_agreement_tolerance_m:
                mdrt_status = "MATCHED"
            else:
                mdrt_status = "MISMATCH"
                warn(
                    "MDRT_MISMATCH",
                    f"{canon!r}: HRS MDRT={mdrt_hrs} m vs readable MDRT={mdrt_readable} m "
                    f"(difference {agreement:+.4f} m exceeds tolerance {mdrt_agreement_tolerance_m} m).",
                )
        elif in_hrs:
            name_match_status = "missing_in_readable"
            mdrt_status = "NOT_COMPARABLE"
            agreement = None
        else:
            name_match_status = "missing_in_hrs"
            mdrt_status = "NOT_COMPARABLE"
            agreement = None

        entries.append(
            TopReconciliationEntry(
                well_key=well_key,
                canonical_marker_name=canon,
                hrs_marker_name_raw=hrs_name_raw,
                readable_marker_name_raw=readable_name_raw,
                hrs_row_number=hrs_row,
                readable_row_number=readable_row,
                present_in_hrs=in_hrs,
                present_in_readable=in_readable,
                name_match_status=name_match_status,
                MDRT_source_hrs_m=mdrt_hrs,
                MDRT_source_readable_m=mdrt_readable,
                MDRT_agreement_readable_minus_hrs_m=agreement,
                mdrt_status=mdrt_status,
                TVDSS_source_readable_m=tvdss_readable,
                note_readable=note_readable,
                reconciliation_notes=(
                    "Present in both sources; MDRT agrees within tolerance." if mdrt_status == "MATCHED"
                    else "Present in both sources; MDRT disagreement registered above." if mdrt_status == "MISMATCH"
                    else f"Present only in {'HRS' if in_hrs else 'readable'} source."
                ),
            )
        )

    return tuple(issues), tuple(entries)


# ---------------------------------------------------------------------------
# Survey-corrected stratigraphic marker table
# ---------------------------------------------------------------------------
def build_formation_top_markers(
    well_key: str,
    reconciliation: Tuple[TopReconciliationEntry, ...],
    deviation_well_result: DeviationWellResult,
    well_identity_evidence_status: str,
) -> Tuple[Tuple[TopIngestionIssue, ...], Tuple[TopMarkerRecord, ...]]:
    """
    Build the corrected, auditable stratigraphic marker table for one
    well: for each reconciled canonical marker, decide the authoritative
    MDRT (see `TopMarkerRecord.mdrt_authority_basis`), then map it through
    the LOCKED `petrel_source_trace` survey trajectory (unmodified
    `p2mem.depth_mapping.map_las_md_to_tvd_tvdss`, called ONE MARKER AT A
    TIME so a single out-of-coverage marker is isolated and reported -
    `mapping_status = "rejected_outside_coverage"` - without blocking any
    other marker for the same well; see `p2mem.depth_mapping
    .ExtrapolationRejectedError`, never silently extrapolated).
    """
    issues: List[TopIngestionIssue] = []
    markers: List[TopMarkerRecord] = []

    for entry in reconciliation:
        if entry.mdrt_status == "MATCHED":
            mdrt_reconciled = entry.MDRT_source_hrs_m
            basis = "hrs_and_readable_agree"
        elif entry.mdrt_status == "MISMATCH":
            mdrt_reconciled = None
            basis = "disagreement_unresolved"
        elif entry.present_in_hrs:
            mdrt_reconciled = entry.MDRT_source_hrs_m
            basis = "hrs_only"
        else:
            mdrt_reconciled = entry.MDRT_source_readable_m
            basis = "readable_only"

        if mdrt_reconciled is None:
            issues.append(
                TopIngestionIssue(
                    "WARNING", "MDRT_UNRESOLVED_NOT_MAPPED",
                    f"{entry.canonical_marker_name!r}: MDRT disagreement between sources exceeds "
                    f"tolerance - this marker is EXCLUDED from survey mapping rather than silently "
                    f"choosing one source's value.",
                    well_key,
                )
            )
            markers.append(
                TopMarkerRecord(
                    well_key=well_key, canonical_marker_name=entry.canonical_marker_name,
                    MDRT_source_hrs_m=entry.MDRT_source_hrs_m, MDRT_source_readable_m=entry.MDRT_source_readable_m,
                    MDRT_reconciled_m=None, mdrt_authority_basis=basis,
                    TVDSS_source_m=entry.TVDSS_source_readable_m,
                    depth_basis_used=None, interpolation_method=None,
                    TVD_survey_m=None, TVDSS_survey_corrected_m=None,
                    TVDSS_residual_source_minus_survey_m=None,
                    mapping_status="not_mapped_mdrt_unresolved",
                    well_identity_evidence_status=well_identity_evidence_status,
                    notes=entry.reconciliation_notes,
                )
            )
            continue

        mdrt_array = np.asarray([mdrt_reconciled], dtype=np.float64)
        try:
            mapped = map_las_md_to_tvd_tvdss(well_key, mdrt_array, deviation_well_result)
        except ExtrapolationRejectedError as exc:
            issues.append(
                TopIngestionIssue(
                    "WARNING", "MARKER_OUTSIDE_SURVEY_COVERAGE",
                    f"{entry.canonical_marker_name!r} (MDRT={mdrt_reconciled} m): {exc}",
                    well_key,
                )
            )
            markers.append(
                TopMarkerRecord(
                    well_key=well_key, canonical_marker_name=entry.canonical_marker_name,
                    MDRT_source_hrs_m=entry.MDRT_source_hrs_m, MDRT_source_readable_m=entry.MDRT_source_readable_m,
                    MDRT_reconciled_m=mdrt_reconciled, mdrt_authority_basis=basis,
                    TVDSS_source_m=entry.TVDSS_source_readable_m,
                    depth_basis_used=None, interpolation_method=None,
                    TVD_survey_m=None, TVDSS_survey_corrected_m=None,
                    TVDSS_residual_source_minus_survey_m=None,
                    mapping_status="rejected_outside_coverage",
                    well_identity_evidence_status=well_identity_evidence_status,
                    notes=entry.reconciliation_notes,
                )
            )
            continue

        tvd_survey = float(mapped.tvd_mapped_m[0])
        tvdss_survey = float(mapped.tvdss_mapped_m[0])
        tvdss_source = entry.TVDSS_source_readable_m
        residual = (tvdss_source - tvdss_survey) if tvdss_source is not None else None

        markers.append(
            TopMarkerRecord(
                well_key=well_key, canonical_marker_name=entry.canonical_marker_name,
                MDRT_source_hrs_m=entry.MDRT_source_hrs_m, MDRT_source_readable_m=entry.MDRT_source_readable_m,
                MDRT_reconciled_m=mdrt_reconciled, mdrt_authority_basis=basis,
                TVDSS_source_m=tvdss_source,
                depth_basis_used=mapped.depth_basis_used, interpolation_method=mapped.interpolation_method,
                TVD_survey_m=tvd_survey, TVDSS_survey_corrected_m=tvdss_survey,
                TVDSS_residual_source_minus_survey_m=residual,
                mapping_status="mapped_within_coverage",
                well_identity_evidence_status=well_identity_evidence_status,
                notes=entry.reconciliation_notes,
            )
        )

    return tuple(issues), tuple(markers)


# ---------------------------------------------------------------------------
# High-level load functions
# ---------------------------------------------------------------------------
def load_formation_top_well(
    hrs_path: str,
    readable_path: str,
    hrs_contract: HRSTopFileContract,
    readable_contract: ReadableTopFileContract,
    marker_name_aliases: Dict[str, str],
    deviation_well_result: DeviationWellResult,
) -> FormationTopWellResult:
    """
    Parse, contract-resolve, reconcile, and survey-map one well's pair of
    formation-top files, returning a complete `FormationTopWellResult`.

    Raises `TopFileNotFoundError`, `TopParsingError`, `TopContractError`,
    or `TopSourceReconciliationError` (never returns a partially valid
    result).
    """
    well_key = hrs_contract.project_well_key

    def _tag(exc: BaseException, origin: str) -> BaseException:
        """
        Increment 5.1 (Finding 2 fix): attach which stage actually failed,
        and BOTH this well's source paths, to the exception before it
        propagates - so `load_formation_top_surveys` never has to guess
        (or default to the HRS path) when building a `TopIngestionFailure`.
        """
        exc.failure_origin = origin  # type: ignore[attr-defined]
        exc.hrs_path = hrs_path  # type: ignore[attr-defined]
        exc.readable_path = readable_path  # type: ignore[attr-defined]
        return exc

    try:
        hrs_header = parse_hrs_top_header(hrs_path)
        hrs_raw = read_hrs_top_rows(hrs_path)
    except (TopFileNotFoundError, TopParsingError) as exc:
        _tag(exc, "hrs")
        raise
    hrs_issues = resolve_hrs_top_contract(hrs_header, hrs_raw, hrs_contract)
    hrs_errors = tuple(i for i in hrs_issues if i.severity == "ERROR")
    if hrs_errors:
        raise _tag(
            TopContractError(
                f"{hrs_path}: {len(hrs_errors)} contract-resolution ERROR(s): "
                + "; ".join(f"[{i.code}] {i.message}" for i in hrs_errors),
                hrs_issues,
            ),
            "hrs",
        )

    try:
        readable_header = parse_readable_top_header(readable_path)
        readable_raw = read_readable_top_rows(readable_path)
    except (TopFileNotFoundError, TopParsingError) as exc:
        _tag(exc, "readable")
        raise
    readable_issues = resolve_readable_top_contract(readable_header, readable_raw, readable_contract)
    readable_errors = tuple(i for i in readable_issues if i.severity == "ERROR")
    if readable_errors:
        raise _tag(
            TopContractError(
                f"{readable_path}: {len(readable_errors)} contract-resolution ERROR(s): "
                + "; ".join(f"[{i.code}] {i.message}" for i in readable_errors),
                readable_issues,
            ),
            "readable",
        )

    recon_issues, reconciliation = reconcile_formation_top_sources(well_key, hrs_raw, readable_raw, marker_name_aliases)
    recon_errors = tuple(i for i in recon_issues if i.severity == "ERROR")
    if recon_errors:
        raise _tag(
            TopSourceReconciliationError(
                f"{well_key}: {len(recon_errors)} source-reconciliation ERROR(s): "
                + "; ".join(f"[{i.code}] {i.message}" for i in recon_errors),
                recon_issues,
            ),
            "reconciliation",
        )

    identity_status = (
        "verified"
        if hrs_contract.well_identity_evidence_status == "verified"
        and readable_contract.well_identity_evidence_status == "verified"
        else "inferred_unverified"
    )
    mapping_issues, markers = build_formation_top_markers(well_key, reconciliation, deviation_well_result, identity_status)

    all_issues = hrs_issues + readable_issues + recon_issues + mapping_issues
    return FormationTopWellResult(
        well_key=well_key,
        hrs_header=hrs_header, hrs_contract=hrs_contract, hrs_raw=hrs_raw,
        readable_header=readable_header, readable_contract=readable_contract, readable_raw=readable_raw,
        reconciliation=reconciliation, markers=markers,
        issues=all_issues, contract_status="PASSED",
    )


def load_formation_top_surveys(
    file_paths: Dict[str, Tuple[str, str]],
    hrs_contracts: Dict[str, HRSTopFileContract],
    readable_contracts: Dict[str, ReadableTopFileContract],
    marker_name_aliases: Dict[str, str],
    deviation_well_results: Dict[str, DeviationWellResult],
) -> Tuple[Dict[str, FormationTopWellResult], Dict[str, TopIngestionFailure]]:
    """
    Load a batch of formation-top file pairs keyed by well key, isolating
    expected per-well ingestion failures as typed `TopIngestionFailure`
    records. One well's failure never stops the others from loading.
    Never caught with a blanket `except Exception` - only the specific
    typed exceptions this module is documented to raise.

    `file_paths` maps well key -> `(hrs_path, readable_path)`.
    `deviation_well_results` maps well key -> the LOCKED
    `DeviationWellResult` for that well (required for every well passed
    here - a well with no locked deviation survey cannot have its markers
    survey-mapped and must not be included in this batch call).
    """
    results: Dict[str, FormationTopWellResult] = {}
    failures: Dict[str, TopIngestionFailure] = {}

    for well_key, (hrs_path, readable_path) in file_paths.items():
        hrs_basename = Path(hrs_path).name
        readable_basename = Path(readable_path).name
        hrs_contract = hrs_contracts.get(hrs_basename)
        readable_contract = readable_contracts.get(readable_basename)
        if hrs_contract is None or readable_contract is None:
            raise TopContractDefinitionError(
                f"No formation-top contract found for well key {well_key!r} "
                f"(hrs={hrs_basename!r}, readable={readable_basename!r}); contracts are keyed by "
                f"source filename and must be authored before ingestion."
            )
        deviation_result = deviation_well_results.get(well_key)
        if deviation_result is None:
            raise ValueError(
                f"No locked DeviationWellResult supplied for well key {well_key!r}; formation-top "
                f"markers cannot be survey-mapped without the well's locked deviation survey."
            )
        def _record_failure(exc: BaseException, error_type: str) -> None:
            """
            Increment 5.1 (Finding 2 fix): identify the actually-failing
            path from the exception's own `failure_origin` tag (set by
            `load_formation_top_well`), rather than unconditionally
            recording the HRS path regardless of which file/stage failed.
            Both source paths are always retained on the failure record so
            every consumer can sanitize both.
            """
            origin = getattr(exc, "failure_origin", "unknown")
            primary_path = readable_path if origin == "readable" else hrs_path
            failures[well_key] = TopIngestionFailure(
                well_key, primary_path, error_type, str(exc), exc,
                failure_origin=origin, hrs_path=hrs_path, readable_path=readable_path,
            )

        try:
            results[well_key] = load_formation_top_well(
                hrs_path, readable_path, hrs_contract, readable_contract, marker_name_aliases, deviation_result
            )
        except TopFileNotFoundError as exc:
            _record_failure(exc, "file_not_found")
        except TopParsingError as exc:
            _record_failure(exc, "parsing_failure")
        except TopContractError as exc:
            _record_failure(exc, "contract_failure")
        except TopSourceReconciliationError as exc:
            _record_failure(exc, "reconciliation_failure")

    return results, failures


#### `p2mem/io/tops_inventory.py` — deterministic inventory-table builders

In [ ]:
%%writefile p2mem/io/tops_inventory.py
"""
p2mem.io.tops_inventory - Deterministic, metadata-oriented output-table
builders for the Increment 5 formation-top layer.

Mirrors the design of `p2mem.io.checkshot_inventory` (Increment 4, LOCKED):
every function here returns a list of plain dicts (one per output row),
ready for `csv.DictWriter`, or a single JSON-serializable manifest dict -
never a full per-sample array. Every row uses only a file's BASENAME for
any path-shaped field, and every numeric field is a plain Python
float/int/`None` (never a NumPy scalar), so output is stable JSON/CSV
regardless of environment.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List

from p2mem.top_models import (
    FormationTopAvailabilityRecord,
    FormationTopWellResult,
    TopIngestionFailure,
)

__all__ = [
    "build_top_file_inventory_rows",
    "build_top_marker_register_rows",
    "build_top_reconciliation_rows",
    "build_top_corrected_marker_rows",
    "build_top_issues_rows",
    "build_top_availability_rows",
    "build_formation_top_manifest",
]


def _sanitize_message(message: str, *source_paths) -> str:
    """Replace every literal full path in `source_paths` inside `message`
    with its basename (mirrors `p2mem.io.checkshot_inventory
    ._sanitize_message`; extended in Increment 5.1 - Finding 2 fix - to
    accept more than one candidate path, since a formation-top ingestion
    failure may originate from either the HRS file or the readable file,
    and both must be sanitized wherever a failure message is exported -
    never only the single path recorded in `TopIngestionFailure
    .source_path`). A falsy/`None` path is skipped. Literal substring
    replacement only - never a broad regex, so unrelated scientific text
    is never corrupted."""
    for source_path in source_paths:
        if not source_path:
            continue
        message = message.replace(source_path, Path(source_path).name)
    return message


def _failure_context_basename(f: TopIngestionFailure) -> str:
    """
    Increment 5.1 (Finding 2 fix): the basename of the file that actually
    caused this failure, chosen from `f.failure_origin` - never assumed to
    be the HRS file. For a "reconciliation"/"mapping"-origin failure (both
    files parsed successfully) both source basenames are retained,
    separated by '+', since either or neither may be individually
    implicated. Falls back to `f.source_path`'s basename when no typed
    origin/path is available (e.g. a `TopIngestionFailure` built without
    this classification)."""
    if f.failure_origin == "hrs" and f.hrs_path:
        return Path(f.hrs_path).name
    if f.failure_origin == "readable" and f.readable_path:
        return Path(f.readable_path).name
    if f.hrs_path and f.readable_path:
        return f"{Path(f.hrs_path).name}+{Path(f.readable_path).name}"
    return Path(f.source_path).name


def build_top_file_inventory_rows(
    results: Dict[str, FormationTopWellResult],
    failures: Dict[str, TopIngestionFailure],
    availability: Dict[str, FormationTopAvailabilityRecord] = None,
) -> List[dict]:
    """One row per source FILE (two rows per successfully loaded well -
    HRS and readable - one row per failed/unavailable well)."""
    availability = availability or {}
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        rows.append(
            {
                "well_key": well_key,
                "representation_type": r.hrs_contract.representation_type,
                "source_filename": r.hrs_header.source_filename,
                "sha256": r.hrs_header.sha256,
                "well_identity_source": r.hrs_header.well_identity_source,
                "well_identity_evidence_status": r.hrs_contract.well_identity_evidence_status,
                "model_use_status": r.hrs_contract.model_use_status,
                "n_markers": len(r.hrs_raw.Top_Name_source),
                "mdrt_min_m": float(r.hrs_raw.MDRT_source_m.min()),
                "mdrt_max_m": float(r.hrs_raw.MDRT_source_m.max()),
                "tvdss_min_m": "",
                "tvdss_max_m": "",
                "contract_status": r.contract_status,
                "n_errors": sum(1 for i in r.issues if i.severity == "ERROR" and i.context == r.hrs_header.source_filename),
                "n_warnings": sum(1 for i in r.issues if i.severity == "WARNING" and i.context == r.hrs_header.source_filename),
            }
        )
        rows.append(
            {
                "well_key": well_key,
                "representation_type": r.readable_contract.representation_type,
                "source_filename": r.readable_header.source_filename,
                "sha256": r.readable_header.sha256,
                "well_identity_source": r.readable_header.well_identity_source,
                "well_identity_evidence_status": r.readable_contract.well_identity_evidence_status,
                "model_use_status": r.readable_contract.model_use_status,
                "n_markers": len(r.readable_raw.TOP_NAME_source),
                "mdrt_min_m": float(r.readable_raw.MDRT_source_m.min()),
                "mdrt_max_m": float(r.readable_raw.MDRT_source_m.max()),
                "tvdss_min_m": float(r.readable_raw.TVDSS_source_m.min()),
                "tvdss_max_m": float(r.readable_raw.TVDSS_source_m.max()),
                "contract_status": r.contract_status,
                "n_errors": sum(1 for i in r.issues if i.severity == "ERROR" and i.context == r.readable_header.source_filename),
                "n_warnings": sum(1 for i in r.issues if i.severity == "WARNING" and i.context == r.readable_header.source_filename),
            }
        )
    for well_key in sorted(failures):
        f = failures[well_key]
        rows.append(
            {
                "well_key": well_key, "representation_type": "", "source_filename": _failure_context_basename(f),
                "sha256": "", "well_identity_source": "", "well_identity_evidence_status": "",
                "model_use_status": "", "n_markers": "", "mdrt_min_m": "", "mdrt_max_m": "",
                "tvdss_min_m": "", "tvdss_max_m": "", "contract_status": "FAILED",
                "n_errors": "", "n_warnings": "",
            }
        )
    for well_key in sorted(availability):
        a = availability[well_key]
        rows.append(
            {
                "well_key": well_key, "representation_type": "", "source_filename": "",
                "sha256": "", "well_identity_source": "", "well_identity_evidence_status": "",
                "model_use_status": "", "n_markers": "", "mdrt_min_m": "", "mdrt_max_m": "",
                "tvdss_min_m": "", "tvdss_max_m": "", "contract_status": a.formation_top_availability,
                "n_errors": "", "n_warnings": "",
            }
        )
    return rows


def build_top_marker_register_rows(results: Dict[str, FormationTopWellResult]) -> List[dict]:
    """One row per raw marker per source file (the full source/provenance
    register: name, source depth, source filename/representation, row
    number, well-association evidence - see the Increment 5 specification,
    'preservation of source marker names and source depths')."""
    rows: List[dict] = []
    for well_key in sorted(results):
        r = results[well_key]
        for i, (name, mdrt) in enumerate(zip(r.hrs_raw.Top_Name_source, r.hrs_raw.MDRT_source_m)):
            rows.append(
                {
                    "well_key": well_key, "source_filename": r.hrs_header.source_filename,
                    "source_representation": r.hrs_contract.representation_type,
                    "source_row_number": i, "top_name_source": name,
                    "MDRT_source_m": float(mdrt), "TVDSS_source_m": "",
                    "note_source": "", "well_identity_source": r.hrs_header.well_identity_source,
                    "well_identity_evidence_status": r.hrs_contract.well_identity_evidence_status,
                }
            )
        for i, (name, mdrt, tvdss, note) in enumerate(
            zip(r.readable_raw.TOP_NAME_source, r.readable_raw.MDRT_source_m, r.readable_raw.TVDSS_source_m, r.readable_raw.NOTE_source)
        ):
            rows.append(
                {
                    "well_key": well_key, "source_filename": r.readable_header.source_filename,
                    "source_representation": r.readable_contract.representation_type,
                    "source_row_number": i, "top_name_source": name,
                    "MDRT_source_m": float(mdrt), "TVDSS_source_m": float(tvdss),
                    "note_source": note, "well_identity_source": r.readable_header.well_identity_source,
                    "well_identity_evidence_status": r.readable_contract.well_identity_evidence_status,
                }
            )
    return rows


def build_top_reconciliation_rows(results: Dict[str, FormationTopWellResult]) -> List[dict]:
    """One row per canonical marker per well - the HRS-versus-readable
    source reconciliation table."""
    rows: List[dict] = []
    for well_key in sorted(results):
        for e in results[well_key].reconciliation:
            rows.append(
                {
                    "well_key": e.well_key, "canonical_marker_name": e.canonical_marker_name,
                    "hrs_marker_name_raw": e.hrs_marker_name_raw or "",
                    "readable_marker_name_raw": e.readable_marker_name_raw or "",
                    "hrs_row_number": e.hrs_row_number if e.hrs_row_number is not None else "",
                    "readable_row_number": e.readable_row_number if e.readable_row_number is not None else "",
                    "present_in_hrs": e.present_in_hrs, "present_in_readable": e.present_in_readable,
                    "name_match_status": e.name_match_status,
                    "MDRT_source_hrs_m": e.MDRT_source_hrs_m if e.MDRT_source_hrs_m is not None else "",
                    "MDRT_source_readable_m": e.MDRT_source_readable_m if e.MDRT_source_readable_m is not None else "",
                    "MDRT_agreement_readable_minus_hrs_m": (
                        e.MDRT_agreement_readable_minus_hrs_m if e.MDRT_agreement_readable_minus_hrs_m is not None else ""
                    ),
                    "mdrt_status": e.mdrt_status,
                    "TVDSS_source_readable_m": e.TVDSS_source_readable_m if e.TVDSS_source_readable_m is not None else "",
                    "note_readable": e.note_readable, "reconciliation_notes": e.reconciliation_notes,
                }
            )
    return rows


def build_top_corrected_marker_rows(results: Dict[str, FormationTopWellResult]) -> List[dict]:
    """One row per canonical marker per well - the corrected, auditable
    survey-mapped stratigraphic marker table (the project's corrected
    depth representation; see `p2mem.top_models.TopMarkerRecord`)."""
    rows: List[dict] = []
    for well_key in sorted(results):
        for m in results[well_key].markers:
            rows.append(
                {
                    "well_key": m.well_key, "canonical_marker_name": m.canonical_marker_name,
                    "MDRT_source_hrs_m": m.MDRT_source_hrs_m if m.MDRT_source_hrs_m is not None else "",
                    "MDRT_source_readable_m": m.MDRT_source_readable_m if m.MDRT_source_readable_m is not None else "",
                    "MDRT_reconciled_m": m.MDRT_reconciled_m if m.MDRT_reconciled_m is not None else "",
                    "mdrt_authority_basis": m.mdrt_authority_basis,
                    "TVDSS_source_m": m.TVDSS_source_m if m.TVDSS_source_m is not None else "",
                    "depth_basis_used": m.depth_basis_used or "",
                    "interpolation_method": m.interpolation_method or "",
                    "TVD_survey_m": m.TVD_survey_m if m.TVD_survey_m is not None else "",
                    "TVDSS_survey_corrected_m": m.TVDSS_survey_corrected_m if m.TVDSS_survey_corrected_m is not None else "",
                    "TVDSS_residual_source_minus_survey_m": (
                        m.TVDSS_residual_source_minus_survey_m
                        if m.TVDSS_residual_source_minus_survey_m is not None else ""
                    ),
                    "mapping_status": m.mapping_status,
                    "well_identity_evidence_status": m.well_identity_evidence_status,
                    "notes": m.notes,
                }
            )
    return rows


def build_top_issues_rows(
    results: Dict[str, FormationTopWellResult], failures: Dict[str, TopIngestionFailure]
) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(results):
        for issue in results[well_key].issues:
            rows.append(
                {
                    "well_key": well_key, "severity": issue.severity, "code": issue.code,
                    "message": issue.message, "context": issue.context,
                }
            )
    for well_key in sorted(failures):
        f = failures[well_key]
        rows.append(
            {
                "well_key": well_key, "severity": "ERROR", "code": f.error_type.upper(),
                "message": _sanitize_message(f.message, f.hrs_path, f.readable_path, f.source_path),
                "context": _failure_context_basename(f),
            }
        )
    return rows


def build_top_availability_rows(
    results: Dict[str, FormationTopWellResult],
    failures: Dict[str, TopIngestionFailure],
    availability: Dict[str, FormationTopAvailabilityRecord],
) -> List[dict]:
    rows: List[dict] = []
    for well_key in sorted(set(results) | set(failures) | set(availability)):
        if well_key in results:
            status, notes = "AVAILABLE", ""
        elif well_key in failures:
            f = failures[well_key]
            status, notes = "INGESTION_FAILED", _sanitize_message(f.message, f.hrs_path, f.readable_path, f.source_path)
        else:
            status, notes = availability[well_key].formation_top_availability, availability[well_key].notes
        rows.append({"well_key": well_key, "formation_top_availability": status, "notes": notes})
    return rows


def build_formation_top_manifest(
    results: Dict[str, FormationTopWellResult],
    failures: Dict[str, TopIngestionFailure],
    availability: Dict[str, FormationTopAvailabilityRecord],
) -> dict:
    wells = {}
    for well_key in sorted(results):
        r = results[well_key]
        mapped = [m for m in r.markers if m.mapping_status == "mapped_within_coverage"]
        residuals = [m.TVDSS_residual_source_minus_survey_m for m in mapped if m.TVDSS_residual_source_minus_survey_m is not None]
        wells[well_key] = {
            "formation_top_availability": "AVAILABLE",
            "hrs_source_filename": r.hrs_header.source_filename,
            "hrs_sha256": r.hrs_header.sha256,
            "readable_source_filename": r.readable_header.source_filename,
            "readable_sha256": r.readable_header.sha256,
            "well_identity_evidence_status": (
                "verified"
                if r.hrs_contract.well_identity_evidence_status == "verified"
                and r.readable_contract.well_identity_evidence_status == "verified"
                else "inferred_unverified"
            ),
            "model_use_status": r.hrs_contract.model_use_status,
            "n_canonical_markers": len(r.reconciliation),
            "n_markers_matched": sum(1 for e in r.reconciliation if e.mdrt_status == "MATCHED"),
            "n_markers_mismatch": sum(1 for e in r.reconciliation if e.mdrt_status == "MISMATCH"),
            "n_markers_single_source": sum(1 for e in r.reconciliation if e.mdrt_status == "NOT_COMPARABLE"),
            "n_markers_mapped_within_coverage": len(mapped),
            "n_markers_rejected_outside_coverage": sum(1 for m in r.markers if m.mapping_status == "rejected_outside_coverage"),
            "n_markers_not_mapped_mdrt_unresolved": sum(1 for m in r.markers if m.mapping_status == "not_mapped_mdrt_unresolved"),
            "max_abs_tvdss_residual_source_minus_survey_m": max((abs(x) for x in residuals), default=None),
            "n_issue_errors": sum(1 for i in r.issues if i.severity == "ERROR"),
            "n_issue_warnings": sum(1 for i in r.issues if i.severity == "WARNING"),
        }
    for well_key in sorted(availability):
        a = availability[well_key]
        wells[well_key] = {"formation_top_availability": a.formation_top_availability, "notes": a.notes}

    failed = {
        well_key: {
            "error_type": f.error_type,
            "message": _sanitize_message(f.message, f.hrs_path, f.readable_path, f.source_path),
        }
        for well_key, f in failures.items()
    }

    return {
        "increment": "5",
        "corrective_patch_of": None,
        "tier_classification": "Tier C - Screening-Level / Uncalibrated Educational",
        "scope": (
            "Contract-driven formation-top file ingestion (two independently supplied source "
            "representations per well: 'HRS' MDRT-only, and 'selected readable' MDRT+TVDSS), "
            "explicit HRS-versus-readable source reconciliation (name matching with a human-"
            "authored alias contract, MDRT cross-check), mapping of reconciled MDRT through the "
            "LOCKED Increment 3.1.1 petrel_source_trace survey trajectory, survey-derived TVD/"
            "TVDSS, an explicit source-minus-survey TVDSS residual comparison, corrected auditable "
            "stratigraphic marker tables, formation-top availability/provenance records, and "
            "formation-top QC outputs/figures. No gamma-ray normalization, shale-volume "
            "calculation, named lithology classification, petrophysical interpretation, method-"
            "eligibility masks, shallow-density modelling, overburden-stress integration, NCT "
            "fitting, pore-pressure prediction, elastic properties, rock strength, horizontal "
            "stresses, or wellbore-stability analysis is performed in this increment."
        ),
        "n_wells_formation_top_available": len(results),
        "n_wells_formation_top_not_available": len(availability),
        "n_wells_failed": len(failures),
        "wells": wells,
        "failed_wells": failed,
    }


#### Step 7 — Write the per-file formation-top contracts (`config/formation_top_contracts.yml`)

**Technical objective:** the human-authored, human-reviewable per-file contract for each of the four approved formation-top files, with every `expected_*` value independently recomputed from the actual files (see `INCREMENT_05_MANIFEST.md`).

In [ ]:
%%writefile config/formation_top_contracts.yml
# config/formation_top_contracts.yml
#
# Increment 5 human-authored, human-reviewable per-file formation-top
# contract for each of the FOUR approved formation-top files:
#   - Poseidon_2_HRS_tops_no_wellname_MDRT.txt
#   - Poseidon_2_selected_well_tops.txt
#   - Boreas_1_HRS_tops_no_wellname_MDRT.txt
#   - Boreas_1_selected_well_tops.txt
#
# Every expected_* field below was independently recomputed from the
# actual four files (never assumed, never copied from prose) - see
# INCREMENT_05_MANIFEST.md Section 1. A mismatch in any expected_* field
# is a blocking ERROR at load time (p2mem.io.tops.resolve_hrs_top_contract /
# resolve_readable_top_contract) - ingestion never silently proceeds with
# "close enough" values.
#
# NOT APPROVED / MUST NOT BE USED (recorded here for traceability only -
# no contract entry exists for these, and none is created for them):
# Poseidon 1 tops, Kronos 1 tops, Torosa 1 tops, or any other well's tops.
#
# Poseidon North 1 and Proteus 1ST2 have NO approved formation-top file at
# all. This is recorded as a factual data gap
# (formation_top_availability: NOT_AVAILABLE - see
# p2mem.top_models.FormationTopAvailabilityRecord and the Increment 5
# integration script/notebook) - never as a missing/failed contract entry
# here, and never filled by substituting another well's tops or
# correlating them by depth alone.
#
# Well-identity evidence (Increment 5 vocabulary - see p2mem.top_models
# and p2mem.io.tops module docstrings): NEITHER file representation is
# ever described as content-verified identity for either well.
#   - The two "HRS" files carry NO well name anywhere in the file body;
#     association with the stated project well rests on the FILENAME
#     alone (well_identity_source = "filename_only").
#   - The two "selected readable" files DO name the well, but only in a
#     leading '#' comment line - project-supplied metadata embedded in
#     the file, not independently verified file content
#     (well_identity_source = "in_file_comment_project_supplied").
# Both are therefore declared well_identity_evidence_status:
# "inferred_unverified" below (never "verified" - unlike, e.g., Increment
# 4's Poseidon2-Checkshot.txt/Boreas1-Checkshot.txt, whose filenames were
# judged unambiguous because there is only one candidate well of each of
# those names in this project; the SAME reasoning is not extended here to
# claim "verified", per this increment's more conservative, explicitly
# audited evidence-classification decision - see INCREMENT_05_MANIFEST.md
# Section 2 for the rationale).
#
# model_use_status (Increment 4 vocabulary, reused): both approved wells'
# formation tops are declared "primary_model" - this increment's authority
# mechanism is explicit HRS-versus-readable RECONCILIATION per well, not a
# primary-vs-qc_only selection between competing wells (there is no
# Increment-4-style "one well's curve becomes the project's primary
# relationship" concept here), so there is no qc_only entry among the four
# approved files.
#
# datum_depth_column_interpretation (identical across all four files -
# independently confirmed against each approved well's own locked
# Increment 3 deviation-survey file header, which states "MD AND TVD ARE
# REFERENCED (=0) AT WELL DATUM [RT, Rotary Table] AND INCREASE
# DOWNWARDS"): this file's MDRT_m/MDRT_M column is measured depth
# referenced to zero at the rotary table, increasing downward - the IDENTICAL
# convention as the locked deviation survey's own MD_source_m column for
# the same well. No datum shift or unit conversion is required before
# mapping MDRT through the locked survey trajectory (see
# p2mem.depth_mapping.map_las_md_to_tvd_tvdss, reused unmodified).
#
# Numeric-range tolerance (0.005, identical across all four files): the
# expected min/max values below were read directly from the source
# file's own numeric text (2 decimal places); this tolerance absorbs
# ordinary floating-point round-trip noise without masking a genuinely
# different file.
files:
  "Poseidon_2_HRS_tops_no_wellname_MDRT.txt":
    expected_sha256: "c907e81dc16bd72c209ffd9d1cc1e6e8837868e465d0fa627756e8c773737b9a"
    project_well_key: "Poseidon_2"
    representation_type: "HRS_MDRT_only"
    well_identity_evidence_status: "inferred_unverified"
    well_identity_evidence_notes: >
      This file's body carries no well name at all (Top_Name/MDRT_m only);
      association with Poseidon 2 rests on the filename alone
      ("Poseidon_2_HRS_tops_no_wellname_MDRT.txt", whose own name states
      it carries no well name in its body). Never described as content-
      verified.
    model_use_status: "primary_model"
    expected_column_header_line: "Top_Name\tMDRT_m"
    expected_column_order: ["Top_Name", "MDRT_m"]
    expected_marker_count: 9
    expected_mdrt_min_m: 518.40
    expected_mdrt_max_m: 5356.00
    numeric_range_tolerance: 0.005
    datum_depth_column_interpretation: >
      MDRT_m referenced to zero at rotary table, increasing downward -
      identical convention to the locked Poseidon 2 deviation survey's own
      MD_source_m column.
    notes: >
      9 markers (Sea Bed through TD). No "Nome Fm" pick exists for
      Poseidon 2 in either approved source file - this is a genuine
      absence of that specific pick for this well, not a reconciliation
      defect (Boreas 1's marker set includes one).

  "Poseidon_2_selected_well_tops.txt":
    expected_sha256: "f4f057c44598efc7397029911521c10a0ec151f4b2387788b894bd99939c40a1"
    project_well_key: "Poseidon_2"
    representation_type: "selected_readable_MDRT_TVDSS"
    well_identity_evidence_status: "inferred_unverified"
    well_identity_evidence_notes: >
      This file's leading '#' comment states "Selected readable well tops
      for Poseidon 2" - project-supplied metadata embedded in the file,
      not independently verified file content. Never described as
      content-verified.
    model_use_status: "primary_model"
    expected_well_name_from_comment: "Poseidon 2"
    expected_column_header_line: "TOP_NAME                                     \t    MDRT_M\t   TVDSS_M\tNOTE"
    expected_column_order: ["TOP_NAME", "MDRT_M", "TVDSS_M", "NOTE"]
    expected_marker_count: 9
    expected_mdrt_min_m: 518.40
    expected_mdrt_max_m: 5356.00
    expected_tvdss_min_m: 496.60
    expected_tvdss_max_m: 5334.20
    numeric_range_tolerance: 0.005
    datum_depth_column_interpretation: >
      MDRT_M referenced to zero at rotary table, increasing downward -
      identical convention to the locked Poseidon 2 deviation survey's own
      MD_source_m column. TVDSS_M is this file's own SUPPLIED value,
      preserved as TVDSS_source_m for residual/QC comparison only - it is
      NOT this project's corrected depth (see p2mem.io.tops module
      docstring).
    notes: >
      Confirmed regression finding (independently recomputed, never
      hardcoded - see INCREMENT_05_MANIFEST.md Section 1): every one of
      this file's 9 TVDSS_M values equals MDRT_M minus 21.8 m EXACTLY (the
      well's own Kelly Bushing/RT elevation) - i.e. this file's TVDSS was
      generated assuming a perfectly vertical well (TVD = MD from datum),
      never accounting for the well's real, surveyed deviation. This is a
      confirmed formation-top depth-reference defect, corrected in the
      derived TVDSS_survey_corrected_m representation while this raw
      TVDSS_source_m column itself is preserved unmodified.

  "Boreas_1_HRS_tops_no_wellname_MDRT.txt":
    expected_sha256: "5504dfe9fdeb3e2b8e024f17d59aba345974d432bdce89f9c8e5b8fd156cc9b4"
    project_well_key: "Boreas_1"
    representation_type: "HRS_MDRT_only"
    well_identity_evidence_status: "inferred_unverified"
    well_identity_evidence_notes: >
      This file's body carries no well name at all (Top_Name/MDRT_m only);
      association with Boreas 1 rests on the filename alone. Never
      described as content-verified.
    model_use_status: "primary_model"
    expected_column_header_line: "Top_Name\tMDRT_m"
    expected_column_order: ["Top_Name", "MDRT_m"]
    expected_marker_count: 10
    expected_mdrt_min_m: 513.70
    expected_mdrt_max_m: 5210.00
    numeric_range_tolerance: 0.005
    datum_depth_column_interpretation: >
      MDRT_m referenced to zero at rotary table, increasing downward -
      identical convention to the locked Boreas 1 deviation survey's own
      MD_source_m column.
    notes: "10 markers (Sea Bed through TD), including Nome Fm."

  "Boreas_1_selected_well_tops.txt":
    expected_sha256: "bc677bc6a5b852de8e0e6deb1afd9ff7936870df95517e608cc6d771eed6327f"
    project_well_key: "Boreas_1"
    representation_type: "selected_readable_MDRT_TVDSS"
    well_identity_evidence_status: "inferred_unverified"
    well_identity_evidence_notes: >
      This file's leading '#' comment states "Selected readable well tops
      for Boreas 1" - project-supplied metadata embedded in the file, not
      independently verified file content. Never described as content-
      verified.
    model_use_status: "primary_model"
    expected_well_name_from_comment: "Boreas 1"
    expected_column_header_line: "TOP_NAME                                     \t    MDRT_M\t   TVDSS_M\tNOTE"
    expected_column_order: ["TOP_NAME", "MDRT_M", "TVDSS_M", "NOTE"]
    expected_marker_count: 10
    expected_mdrt_min_m: 513.70
    expected_mdrt_max_m: 5210.00
    expected_tvdss_min_m: 491.90
    expected_tvdss_max_m: 5184.10
    numeric_range_tolerance: 0.005
    datum_depth_column_interpretation: >
      MDRT_M referenced to zero at rotary table, increasing downward -
      identical convention to the locked Boreas 1 deviation survey's own
      MD_source_m column. TVDSS_M is this file's own SUPPLIED value,
      preserved as TVDSS_source_m for residual/QC comparison only - it is
      NOT this project's corrected depth.
    notes: >
      Confirmed regression finding (independently recomputed, never
      hardcoded - see INCREMENT_05_MANIFEST.md Section 1): unlike
      Poseidon 2, this file's TVDSS_M values do NOT equal MDRT_M minus
      21.8 m (e.g. Prion Fm: 2206.90 - 21.8 = 2185.10, but the file states
      2185.04) - they instead agree closely with the well's real, survey-
      corrected TVDSS (max absolute residual 0.0416 m across all 10
      markers, independently recomputed, below the approved Rev 1 design
      tolerance of 0.05 m). This file was evidently generated from the
      well's real, surveyed trajectory, not a vertical-well assumption.
      This distinct, per-file evidence is why Boreas 1 is NOT "corrected"
      merely because Poseidon 2 contains a defect - the two files' actual
      generation methods differ, and the evidence is applied per well.

# Human-authored marker-name alias contract (Increment 5 requirement:
# "If normalization is needed, implement a human-authored alias contract
# and preserve both original and canonical names... Do not use fuzzy
# marker matching silently"). Empty for the current approved data: an
# exhaustive comparison of all four files' marker names found every HRS
# marker name IDENTICAL, byte-for-byte, to its corresponding readable-file
# marker name (after only the readable file's own fixed-width space
# padding is stripped by p2mem.io.tops.normalize_marker_name - never an
# alias). No two markers currently require this table to be recognized as
# the same canonical marker. It exists, and is exercised by
# tests/test_tops.py's synthetic aliased-marker fixtures, so a genuinely
# differently-spelled marker name in a FUTURE approved file can be
# reconciled explicitly (by adding an entry here, reviewed by a human) -
# never by silent fuzzy/similarity matching.
marker_name_aliases: {}


#### Step 7a — Write the synthetic test fixtures and the new test suites

**Increment 5.1 addition:** `top_hrs_disjoint_markers.txt` (Finding 1 regression coverage — two entirely disjoint, non-empty marker sets) and `top_readable_malformed_numeric.txt` / `top_readable_nonfinite.txt` / `top_readable_negative_depth.txt` (readable-format-equivalent coverage for malformed/non-finite/negative-MDRT rows — only HRS-format fixtures existed for these three conditions prior to this patch).

In [ ]:
%%writefile tests/fixtures/top_hrs_valid.txt
Top_Name	MDRT_m
Marker A	100.00
Marker B	200.00
Marker C	300.00


In [ ]:
%%writefile tests/fixtures/top_hrs_duplicate_marker.txt
Top_Name	MDRT_m
Marker A	100.00
Marker A	150.00


In [ ]:
%%writefile tests/fixtures/top_hrs_order_reversal.txt
Top_Name	MDRT_m
Marker A	200.00
Marker B	100.00


In [ ]:
%%writefile tests/fixtures/top_hrs_malformed_numeric.txt
Top_Name	MDRT_m
Marker A	abc


In [ ]:
%%writefile tests/fixtures/top_hrs_nonfinite.txt
Top_Name	MDRT_m
Marker A	NaN


In [ ]:
%%writefile tests/fixtures/top_hrs_negative_depth.txt
Top_Name	MDRT_m
Marker A	-10.00


In [ ]:
%%writefile tests/fixtures/top_hrs_extra_marker.txt
Top_Name	MDRT_m
Marker A	100.00
Marker B	200.00
Marker D	400.00


In [ ]:
%%writefile tests/fixtures/top_hrs_disjoint_markers.txt
Top_Name	MDRT_m
Marker X	100.00
Marker Y	200.00


In [ ]:
%%writefile tests/fixtures/top_readable_valid.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    100.00	     95.00	Top marker
Marker B                                     	    200.00	    190.00	Middle marker
Marker C                                     	    300.00	    285.00	Bottom marker


In [ ]:
%%writefile tests/fixtures/top_readable_mdrt_mismatch.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    100.00	     95.00	Top marker
Marker B                                     	    205.00	    195.00	Middle marker, mismatched MDRT
Marker C                                     	    300.00	    285.00	Bottom marker


In [ ]:
%%writefile tests/fixtures/top_readable_missing_marker.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    100.00	     95.00	Top marker
Marker B                                     	    200.00	    190.00	Middle marker


In [ ]:
%%writefile tests/fixtures/top_readable_aliased_name.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    100.00	     95.00	Top marker
Marker B (Alt Spelling)                      	    200.00	    190.00	Middle marker, aliased name
Marker C                                     	    300.00	    285.00	Bottom marker


In [ ]:
%%writefile tests/fixtures/top_readable_missing_separator.txt
# Selected readable well tops for Test Well 1
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
Marker A                                     	    100.00	     95.00	Top marker


In [ ]:
%%writefile tests/fixtures/top_readable_extra_padding.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    100.00	     95.00	Top marker
Marker B                                     	    200.00	    190.00	Middle marker


In [ ]:
%%writefile tests/fixtures/top_readable_malformed_numeric.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    abc	     95.00	Top marker


In [ ]:
%%writefile tests/fixtures/top_readable_nonfinite.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    NaN	     95.00	Top marker


In [ ]:
%%writefile tests/fixtures/top_readable_negative_depth.txt
# Selected readable well tops for Test Well 1
# Source datum columns: MDRT = measured depth below rotary table; TVDSS = true vertical depth subsea
# Synthetic test fixture - not a real project well

TOP_NAME                                     	    MDRT_M	   TVDSS_M	NOTE
--------------------------------------------------------------------------------------------------------------
Marker A                                     	    -10.00	     95.00	Top marker


In [ ]:
%%writefile tests/test_tops.py
"""
tests/test_tops.py - Validation suite for p2mem.io.tops (Increment 5:
formation-top ingestion, HRS-versus-readable source reconciliation, and
survey-corrected stratigraphic depth mapping).

PORTABLE unit tests only: small synthetic fixtures under
tests/fixtures/top_*.txt, and hand-built typed `DeviationWellResult`
objects (mirroring `tests/test_depth_mapping.py`'s established pattern).
Real four-file integration (which requires the private/real project
formation-top files) is a separate notebook/script run - see
dev_scratch_inc5/run_integration_05.py (not packaged) and
INCREMENT_05_MANIFEST.md for the actual recomputed real-data results.
"""

import hashlib
from pathlib import Path

import numpy as np
import pytest

from p2mem.deviation_models import (
    DEPTH_BASIS_PETREL_SOURCE,
    DepthBasisSelection,
    DeviationFileContract,
    DeviationHeaderInfo,
    DeviationStationData,
    DeviationWellResult,
    TrajectoryValidationResult,
)
from p2mem.top_models import HRSTopFileContract, ReadableTopFileContract
from p2mem.io.tops import (
    MDRT_AGREEMENT_TOLERANCE_M,
    TopContractDefinitionError,
    TopContractError,
    TopFileNotFoundError,
    TopParsingError,
    TopSourceReconciliationError,
    build_formation_top_markers,
    load_formation_top_contract_config,
    load_formation_top_surveys,
    load_formation_top_well,
    normalize_marker_name,
    parse_hrs_top_header,
    parse_readable_top_header,
    read_hrs_top_rows,
    read_readable_top_rows,
    reconcile_formation_top_sources,
    resolve_hrs_top_contract,
    resolve_readable_top_contract,
)
from p2mem.trajectory import compute_minimum_curvature_trajectory

FIXTURES = Path(__file__).parent / "fixtures"


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


# ---------------------------------------------------------------------------
# Synthetic DeviationWellResult builder (mirrors tests/test_depth_mapping.py)
# ---------------------------------------------------------------------------
def _make_deviation_well_result(md, incl, azim_gn, datum_elevation_m=20.0) -> DeviationWellResult:
    md = np.asarray(md, dtype=np.float64)
    incl = np.asarray(incl, dtype=np.float64)
    azim_gn = np.asarray(azim_gn, dtype=np.float64)
    mc = compute_minimum_curvature_trajectory(
        md, incl, azim_gn, tvd_origin_m=float(md[0]), northing_origin_m=0.0, easting_origin_m=0.0
    )
    stations = DeviationStationData(
        MD_source_m=md, X_source_m=np.zeros_like(md), Y_source_m=np.zeros_like(md),
        Z_source_m=datum_elevation_m - mc.tvd_mc_m, TVD_source_m=mc.tvd_mc_m.copy(),
        DX_source_m=mc.easting_offset_mc_m.copy(), DY_source_m=mc.northing_offset_mc_m.copy(),
        AZIM_TN_source_deg=azim_gn.copy(), INCL_source_deg=incl,
        DLS_source_deg_per_30m=mc.dls_deg_per_30m, AZIM_GN_source_deg=azim_gn,
    )
    header = DeviationHeaderInfo(
        source_path="synthetic", source_filename="synthetic_dev.txt", sha256="0" * 64,
        well_name="Test_Well_1", survey_name="Synthetic survey", wellhead_x_m=0.0, wellhead_y_m=0.0,
        datum_elevation_m=datum_elevation_m, datum_reference="RT, Rotary table, from MSL",
        well_type="GAS", coordinate_reference_system="TEST", depth_reference_statement="test",
        angle_unit_statement="DEGREES", dx_dy_statement="m-UNITS", z_statement="m-UNITS",
        column_names=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        header_line_count=16, data_line_offset=17,
    )
    contract = DeviationFileContract(
        source_filename="synthetic_dev.txt", expected_sha256="0" * 64,
        expected_well_identifier="Test_Well_1", expected_survey_identifier="Synthetic survey",
        expected_coordinate_reference_system="TEST", expected_wellhead_x_m=0.0, expected_wellhead_y_m=0.0,
        expected_datum_m=datum_elevation_m, expected_datum_reference="RT, Rotary table, from MSL",
        expected_column_count=11,
        expected_column_order=("MD", "X", "Y", "Z", "TVD", "DX", "DY", "AZIM_TN", "INCL", "DLS", "AZIM_GN"),
        expected_units={}, expected_station_count=int(md.size),
        expected_md_min_m=float(md[0]), expected_md_max_m=float(md[-1]),
        azimuth_reference_for_grid_coordinates="AZIM_GN", source_depth_convention="test",
        header_tolerance_m=0.001, residual_tolerance_tvd_m=0.01, residual_tolerance_horizontal_m=0.01,
        residual_fail_threshold_m=5.0, depth_basis_policy=DEPTH_BASIS_PETREL_SOURCE, notes="synthetic",
    )
    validation = TrajectoryValidationResult(
        well_key="Test_Well_1", comparison_basis="synthetic self-consistent fixture",
        tvd_max_abs_residual_m=0.0, tvd_mean_residual_m=0.0, tvd_rmse_m=0.0,
        tvd_endpoint_residual_m=0.0, tvd_tolerance_m=0.01, tvd_status="PASS",
        easting_max_abs_residual_m=0.0, easting_mean_residual_m=0.0, easting_rmse_m=0.0,
        easting_endpoint_residual_m=0.0, easting_tolerance_m=0.01, easting_status="PASS",
        northing_max_abs_residual_m=0.0, northing_mean_residual_m=0.0, northing_rmse_m=0.0,
        northing_endpoint_residual_m=0.0, northing_tolerance_m=0.01, northing_status="PASS",
        x_consistency_max_abs_residual_m=0.0, x_consistency_status="PASS",
        y_consistency_max_abs_residual_m=0.0, y_consistency_status="PASS",
        z_consistency_max_abs_residual_m=0.0, z_consistency_status="PASS",
        overall_status="PASS", origin_initialization_note="synthetic",
    )
    depth_basis_sel = DepthBasisSelection(
        well_key="Test_Well_1", selected_basis=DEPTH_BASIS_PETREL_SOURCE, rationale="test"
    )
    return DeviationWellResult(
        header=header, contract=contract, raw=stations, mc=mc,
        validation=validation, depth_basis=depth_basis_sel,
    )


_VERTICAL_WELL = _make_deviation_well_result(
    md=[0.0, 100.0, 200.0, 300.0, 400.0], incl=[0.0, 0.0, 0.0, 0.0, 0.0],
    azim_gn=[0.0, 0.0, 0.0, 0.0, 0.0], datum_elevation_m=20.0,
)
_DEVIATED_WELL = _make_deviation_well_result(
    md=[0.0, 100.0, 200.0, 300.0, 400.0], incl=[0.0, 10.0, 25.0, 35.0, 40.0],
    azim_gn=[0.0, 30.0, 30.0, 30.0, 30.0], datum_elevation_m=20.0,
)


def _base_hrs_contract(filename: str, **overrides) -> HRSTopFileContract:
    fixture_path = FIXTURES / filename
    defaults = dict(
        source_filename=filename,
        expected_sha256=_sha256(fixture_path) if fixture_path.exists() else "0" * 64,
        project_well_key="Test_Well_1",
        representation_type="HRS_MDRT_only",
        well_identity_evidence_status="inferred_unverified",
        well_identity_evidence_notes="Test fixture; identity not a real project well.",
        model_use_status="primary_model",
        expected_column_header_line="Top_Name\tMDRT_m",
        expected_column_order=("Top_Name", "MDRT_m"),
        expected_column_count=2,
        expected_marker_count=3,
        expected_mdrt_min_m=100.0,
        expected_mdrt_max_m=300.0,
        numeric_range_tolerance=0.005,
        datum_depth_column_interpretation="test",
        notes="test",
    )
    defaults.update(overrides)
    return HRSTopFileContract(**defaults)


def _base_readable_contract(filename: str, **overrides) -> ReadableTopFileContract:
    fixture_path = FIXTURES / filename
    defaults = dict(
        source_filename=filename,
        expected_sha256=_sha256(fixture_path) if fixture_path.exists() else "0" * 64,
        project_well_key="Test_Well_1",
        representation_type="selected_readable_MDRT_TVDSS",
        well_identity_evidence_status="inferred_unverified",
        well_identity_evidence_notes="Test fixture; identity not a real project well.",
        model_use_status="primary_model",
        expected_well_name_from_comment="Test Well 1",
        expected_column_header_line="TOP_NAME                                     \t    MDRT_M\t   TVDSS_M\tNOTE",
        expected_column_order=("TOP_NAME", "MDRT_M", "TVDSS_M", "NOTE"),
        expected_column_count=4,
        expected_marker_count=3,
        expected_mdrt_min_m=100.0,
        expected_mdrt_max_m=300.0,
        expected_tvdss_min_m=95.0,
        expected_tvdss_max_m=285.0,
        numeric_range_tolerance=0.005,
        datum_depth_column_interpretation="test",
        notes="test",
    )
    defaults.update(overrides)
    return ReadableTopFileContract(**defaults)


# ---------------------------------------------------------------------------
# Structural parsing - both formats, comments/separator handling
# ---------------------------------------------------------------------------
def test_parse_hrs_header_and_rows():
    header = parse_hrs_top_header(str(FIXTURES / "top_hrs_valid.txt"))
    assert header.column_names == ("Top_Name", "MDRT_m")
    assert header.well_identity_source == "filename_only"
    rows = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    assert rows.Top_Name_source == ("Marker A", "Marker B", "Marker C")
    assert list(rows.MDRT_source_m) == [100.0, 200.0, 300.0]


def test_parse_readable_header_comments_and_separator_deliberately():
    header = parse_readable_top_header(str(FIXTURES / "top_readable_valid.txt"))
    assert header.well_name_from_comment == "Test Well 1"
    assert len(header.comment_lines) == 3
    assert set(header.separator_line_raw.strip()) == {"-"}
    assert header.well_identity_source == "in_file_comment_project_supplied"
    rows = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    assert rows.TOP_NAME_source == ("Marker A", "Marker B", "Marker C")
    assert list(rows.TVDSS_source_m) == [95.0, 190.0, 285.0]
    assert rows.NOTE_source == ("Top marker", "Middle marker", "Bottom marker")


def test_readable_missing_separator_line_is_parsing_error():
    with pytest.raises(TopParsingError, match="separator"):
        parse_readable_top_header(str(FIXTURES / "top_readable_missing_separator.txt"))


def test_hrs_file_not_found_raises_typed_error():
    with pytest.raises(TopFileNotFoundError):
        parse_hrs_top_header(str(FIXTURES / "does_not_exist_hrs.txt"))


def test_readable_file_not_found_raises_typed_error():
    with pytest.raises(TopFileNotFoundError):
        parse_readable_top_header(str(FIXTURES / "does_not_exist_readable.txt"))


# ---------------------------------------------------------------------------
# Malformed / non-finite / negative numeric input
# ---------------------------------------------------------------------------
@pytest.mark.parametrize(
    "fixture",
    ["top_hrs_malformed_numeric.txt", "top_hrs_nonfinite.txt", "top_hrs_negative_depth.txt"],
)
def test_hrs_malformed_or_nonfinite_or_negative_depth_rejected(fixture):
    with pytest.raises(TopParsingError):
        read_hrs_top_rows(str(FIXTURES / fixture))


# ---------------------------------------------------------------------------
# Contract resolution - exact filename and SHA-256 validation
# ---------------------------------------------------------------------------
def test_hrs_contract_passes_for_matching_file():
    header = parse_hrs_top_header(str(FIXTURES / "top_hrs_valid.txt"))
    rows = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    contract = _base_hrs_contract("top_hrs_valid.txt")
    issues = resolve_hrs_top_contract(header, rows, contract)
    assert not any(i.severity == "ERROR" for i in issues)
    assert any(i.code == "WELL_IDENTITY_FILENAME_ONLY" for i in issues)


def test_hrs_contract_sha256_mismatch_is_error():
    header = parse_hrs_top_header(str(FIXTURES / "top_hrs_valid.txt"))
    rows = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    contract = _base_hrs_contract("top_hrs_valid.txt", expected_sha256="0" * 64)
    issues = resolve_hrs_top_contract(header, rows, contract)
    assert any(i.code == "SHA256_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_hrs_contract_filename_mismatch_is_error():
    header = parse_hrs_top_header(str(FIXTURES / "top_hrs_valid.txt"))
    rows = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    contract = _base_hrs_contract("a_different_filename.txt", expected_sha256=header.sha256)
    issues = resolve_hrs_top_contract(header, rows, contract)
    assert any(i.code == "FILENAME_MISMATCH" and i.severity == "ERROR" for i in issues)


def test_readable_contract_passes_for_matching_file():
    header = parse_readable_top_header(str(FIXTURES / "top_readable_valid.txt"))
    rows = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    contract = _base_readable_contract("top_readable_valid.txt")
    issues = resolve_readable_top_contract(header, rows, contract)
    assert not any(i.severity == "ERROR" for i in issues)
    assert any(i.code == "WELL_IDENTITY_IN_FILE_COMMENT_PROJECT_SUPPLIED" for i in issues)
    assert any(i.code == "SUPPLIED_TVDSS_NOT_CORRECTED_DEPTH" for i in issues)


def test_readable_contract_well_name_comment_mismatch_is_error():
    header = parse_readable_top_header(str(FIXTURES / "top_readable_valid.txt"))
    rows = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    contract = _base_readable_contract("top_readable_valid.txt", expected_well_name_from_comment="Wrong Well")
    issues = resolve_readable_top_contract(header, rows, contract)
    assert any(i.code == "WELL_NAME_COMMENT_MISMATCH" and i.severity == "ERROR" for i in issues)


# ---------------------------------------------------------------------------
# Embedded (in-file-comment) vs filename-derived well identity
# ---------------------------------------------------------------------------
def test_well_identity_evidence_never_verified_for_either_format():
    hrs_header = parse_hrs_top_header(str(FIXTURES / "top_hrs_valid.txt"))
    readable_header = parse_readable_top_header(str(FIXTURES / "top_readable_valid.txt"))
    assert hrs_header.well_identity_source == "filename_only"
    assert readable_header.well_identity_source == "in_file_comment_project_supplied"
    # Neither source's structural identity evidence is "verified" - that
    # status is a contract-level declaration, never inferred automatically
    # from a successful parse.
    contract = _base_hrs_contract("top_hrs_valid.txt")
    assert contract.well_identity_evidence_status == "inferred_unverified"


# ---------------------------------------------------------------------------
# Marker-name normalization and alias-contract behavior (no fuzzy matching)
# ---------------------------------------------------------------------------
def test_normalize_marker_name_collapses_whitespace_only():
    assert normalize_marker_name("  Marker   A  ") == "Marker A"
    assert normalize_marker_name("Marker A") == "Marker A"
    # Case and punctuation are never altered.
    assert normalize_marker_name("marker a") == "marker a"


def test_reconciliation_matches_whitespace_padded_names_without_alias():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    assert not any(i.severity == "ERROR" for i in issues)
    assert all(e.name_match_status in ("exact", "normalized_match") for e in entries)


def test_alias_contract_resolves_differently_spelled_marker():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    aliased = read_readable_top_rows(str(FIXTURES / "top_readable_aliased_name.txt"))
    # Without the alias, "Marker B (Alt Spelling)" is NOT recognized as
    # the same marker as "Marker B" - no fuzzy matching.
    issues_no_alias, entries_no_alias = reconcile_formation_top_sources("Test_Well_1", hrs, aliased, {})
    names = {e.canonical_marker_name for e in entries_no_alias}
    assert "Marker B" in names and "Marker B (Alt Spelling)" in names
    assert any(e.name_match_status == "missing_in_readable" for e in entries_no_alias)
    assert any(e.name_match_status == "missing_in_hrs" for e in entries_no_alias)

    # With the explicit, human-authored alias, they ARE reconciled as one
    # canonical marker.
    issues_aliased, entries_aliased = reconcile_formation_top_sources(
        "Test_Well_1", hrs, aliased, {"Marker B (Alt Spelling)": "Marker B"}
    )
    b_entry = next(e for e in entries_aliased if e.canonical_marker_name == "Marker B")
    assert b_entry.name_match_status == "aliased_match"
    assert b_entry.present_in_hrs and b_entry.present_in_readable
    assert b_entry.mdrt_status == "MATCHED"


def test_no_silent_fuzzy_matching_of_similar_but_unaliased_names():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    aliased = read_readable_top_rows(str(FIXTURES / "top_readable_aliased_name.txt"))
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, aliased, {})
    # "Marker B" and "Marker B (Alt Spelling)" are textually similar but
    # NOT identical/normalized-identical/aliased - they must be reported
    # as two separate, single-source markers, never silently paired.
    canon_names = [e.canonical_marker_name for e in entries]
    assert canon_names.count("Marker B") == 1
    assert canon_names.count("Marker B (Alt Spelling)") == 1


# ---------------------------------------------------------------------------
# Duplicate marker names / marker-order reversal (reconciliation-time QC)
# ---------------------------------------------------------------------------
def test_duplicate_marker_name_within_one_source_is_error():
    hrs_dup = read_hrs_top_rows(str(FIXTURES / "top_hrs_duplicate_marker.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, _ = reconcile_formation_top_sources("Test_Well_1", hrs_dup, readable, {})
    assert any(i.code == "DUPLICATE_MARKER_NAME" and i.severity == "ERROR" for i in issues)


def test_marker_order_reversal_within_one_source_is_error():
    hrs_rev = read_hrs_top_rows(str(FIXTURES / "top_hrs_order_reversal.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, _ = reconcile_formation_top_sources("Test_Well_1", hrs_rev, readable, {})
    assert any(i.code == "MARKER_ORDER_REVERSAL" and i.severity == "ERROR" for i in issues)


# ---------------------------------------------------------------------------
# Source marker mismatch (missing-in-one-source) / MDRT mismatch
# ---------------------------------------------------------------------------
def test_marker_present_in_only_one_source_is_not_comparable():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    missing = read_readable_top_rows(str(FIXTURES / "top_readable_missing_marker.txt"))
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, missing, {})
    c_entry = next(e for e in entries if e.canonical_marker_name == "Marker C")
    assert c_entry.name_match_status == "missing_in_readable"
    assert c_entry.mdrt_status == "NOT_COMPARABLE"
    assert c_entry.MDRT_source_readable_m is None
    assert c_entry.TVDSS_source_readable_m is None


def test_mdrt_mismatch_beyond_tolerance_is_registered_not_silently_chosen():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    mismatch = read_readable_top_rows(str(FIXTURES / "top_readable_mdrt_mismatch.txt"))
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, mismatch, {})
    assert any(i.code == "MDRT_MISMATCH" and i.severity == "WARNING" for i in issues)
    b_entry = next(e for e in entries if e.canonical_marker_name == "Marker B")
    assert b_entry.mdrt_status == "MISMATCH"
    assert b_entry.MDRT_agreement_readable_minus_hrs_m == pytest.approx(5.0)
    assert abs(b_entry.MDRT_agreement_readable_minus_hrs_m) > MDRT_AGREEMENT_TOLERANCE_M


def test_mdrt_mismatch_marker_excluded_from_mapping_not_silently_picked():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    mismatch = read_readable_top_rows(str(FIXTURES / "top_readable_mdrt_mismatch.txt"))
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, mismatch, {})
    issues, markers = build_formation_top_markers("Test_Well_1", entries, _VERTICAL_WELL, "inferred_unverified")
    b_marker = next(m for m in markers if m.canonical_marker_name == "Marker B")
    assert b_marker.mapping_status == "not_mapped_mdrt_unresolved"
    assert b_marker.MDRT_reconciled_m is None
    assert b_marker.mdrt_authority_basis == "disagreement_unresolved"
    assert any(i.code == "MDRT_UNRESOLVED_NOT_MAPPED" for i in issues)
    # The other two markers, unaffected, still map normally.
    a_marker = next(m for m in markers if m.canonical_marker_name == "Marker A")
    assert a_marker.mapping_status == "mapped_within_coverage"


# ---------------------------------------------------------------------------
# Mapping inside coverage / rejection outside coverage / no extrapolation
# ---------------------------------------------------------------------------
def test_mapping_within_survey_coverage_succeeds():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    issues, markers = build_formation_top_markers("Test_Well_1", entries, _VERTICAL_WELL, "inferred_unverified")
    assert all(m.mapping_status == "mapped_within_coverage" for m in markers)
    assert not any(i.code == "MARKER_OUTSIDE_SURVEY_COVERAGE" for i in issues)
    for m in markers:
        assert m.TVD_survey_m is not None
        assert m.depth_basis_used == DEPTH_BASIS_PETREL_SOURCE


def test_marker_outside_survey_coverage_is_rejected_never_extrapolated():
    # _VERTICAL_WELL's survey MD coverage is [0, 400] m; a marker at
    # MDRT=500 m is genuinely outside coverage.
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_extra_marker.txt"))  # Marker D at 400.0, within range
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_extra_padding.txt"))
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    # Manually construct one marker beyond coverage by reusing the
    # reconciliation machinery's own output shape - simplest robust way is
    # to add a synthetic entry via the same dataclass used elsewhere.
    from p2mem.top_models import TopReconciliationEntry

    beyond_entry = TopReconciliationEntry(
        well_key="Test_Well_1", canonical_marker_name="Marker Beyond Coverage",
        hrs_marker_name_raw="Marker Beyond Coverage", readable_marker_name_raw="Marker Beyond Coverage",
        hrs_row_number=0, readable_row_number=0, present_in_hrs=True, present_in_readable=True,
        name_match_status="exact", MDRT_source_hrs_m=999.0, MDRT_source_readable_m=999.0,
        MDRT_agreement_readable_minus_hrs_m=0.0, mdrt_status="MATCHED",
        TVDSS_source_readable_m=900.0, note_readable="", reconciliation_notes="test",
    )
    issues, markers = build_formation_top_markers("Test_Well_1", (beyond_entry,), _VERTICAL_WELL, "inferred_unverified")
    assert markers[0].mapping_status == "rejected_outside_coverage"
    assert markers[0].TVD_survey_m is None
    assert markers[0].TVDSS_survey_corrected_m is None
    assert any(i.code == "MARKER_OUTSIDE_SURVEY_COVERAGE" and i.severity == "WARNING" for i in issues)


# ---------------------------------------------------------------------------
# Explicit residual sign convention
# ---------------------------------------------------------------------------
def test_residual_sign_convention_is_source_minus_survey():
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    _, markers = build_formation_top_markers("Test_Well_1", entries, _VERTICAL_WELL, "inferred_unverified")
    for m in markers:
        expected = m.TVDSS_source_m - m.TVDSS_survey_corrected_m
        assert m.TVDSS_residual_source_minus_survey_m == pytest.approx(expected)


# ---------------------------------------------------------------------------
# Poseidon-like vertical-assumption defect / Boreas-like survey-consistent
# tops (synthetic analogs of the real Increment 5 regression findings -
# see INCREMENT_05_MANIFEST.md Section 1 for the actual real-data numbers)
# ---------------------------------------------------------------------------
def test_vertical_assumption_defect_grows_with_deviation_poseidon_like():
    """
    A synthetic analog of the confirmed Poseidon 2 finding: when a
    well's supplied TVDSS was generated by assuming zero deviation
    (TVDSS_source = MDRT - datum_elevation_m, i.e. TVD == MD from the
    well datum), the source-minus-survey residual is ~0 at MD == 0 and
    grows monotonically as the well's real surveyed deviation increases
    with depth. Built the same way the real Poseidon 2 defect was
    independently discovered: from the well's own datum elevation, not
    an arbitrary constant.
    """
    from p2mem.top_models import ReadableTopStationData

    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    mdrt = np.array([100.0, 200.0, 300.0])
    vertical_assumption_tvdss = mdrt - _DEVIATED_WELL.header.datum_elevation_m
    readable = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B", "Marker C"),
        MDRT_source_m=mdrt, TVDSS_source_m=vertical_assumption_tvdss,
        NOTE_source=("", "", ""),
    )
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    _, markers = build_formation_top_markers("Test_Well_1", entries, _DEVIATED_WELL, "inferred_unverified")
    residuals = [abs(m.TVDSS_residual_source_minus_survey_m) for m in markers]
    # Monotonically non-decreasing residual as MD (and therefore
    # inclination/deviation) increases - the vertical-assumption defect
    # signature.
    assert residuals == sorted(residuals)
    assert residuals[0] < residuals[-1]


def test_survey_consistent_tops_boreas_like_small_residual():
    """
    A synthetic analog of the confirmed Boreas 1 finding: when a well's
    supplied TVDSS was itself generated from the real surveyed
    trajectory (not a vertical assumption), the source-minus-survey
    residual stays small even for a deviated well.
    """
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    # Build a "survey-consistent" readable fixture in-memory: TVDSS equal
    # to the deviated well's own actual survey-derived TVDSS at each
    # marker (i.e., generated from the real trajectory, not MDRT - const).
    from p2mem.depth_mapping import map_las_md_to_tvd_tvdss
    from p2mem.top_models import ReadableTopStationData

    mdrt = np.array([100.0, 200.0, 300.0])
    mapped = map_las_md_to_tvd_tvdss("Test_Well_1", mdrt, _DEVIATED_WELL)
    readable = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B", "Marker C"),
        MDRT_source_m=mdrt, TVDSS_source_m=mapped.tvdss_mapped_m.copy(),
        NOTE_source=("", "", ""),
    )
    _, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    _, markers = build_formation_top_markers("Test_Well_1", entries, _DEVIATED_WELL, "inferred_unverified")
    max_abs_residual = max(abs(m.TVDSS_residual_source_minus_survey_m) for m in markers)
    assert max_abs_residual < 1e-6  # essentially zero - never "corrected" when already consistent


# ---------------------------------------------------------------------------
# Ambiguous-dtype / malformed input rejection
# ---------------------------------------------------------------------------
def test_reconcile_rejects_boolean_dtype_mdrt_array():
    from p2mem.top_models import HRSTopStationData, ReadableTopStationData

    hrs_bad = HRSTopStationData(Top_Name_source=("Marker A",), MDRT_source_m=np.array([True]))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TypeError):
        reconcile_formation_top_sources("Test_Well_1", hrs_bad, readable, {})


def test_reconcile_rejects_string_dtype_tvdss_array():
    from p2mem.top_models import ReadableTopStationData

    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable_bad = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B", "Marker C"),
        MDRT_source_m=np.array([100.0, 200.0, 300.0]),
        TVDSS_source_m=np.array(["95.0", "190.0", "285.0"]),
        NOTE_source=("", "", ""),
    )
    with pytest.raises(TypeError):
        reconcile_formation_top_sources("Test_Well_1", hrs, readable_bad, {})


def test_reconcile_rejects_mismatched_array_lengths():
    from p2mem.top_models import HRSTopStationData

    hrs_bad = HRSTopStationData(Top_Name_source=("Marker A", "Marker B"), MDRT_source_m=np.array([100.0]))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs_bad, readable, {})


# ---------------------------------------------------------------------------
# End-to-end well loading and contract-definition validation
# ---------------------------------------------------------------------------
def test_load_formation_top_well_end_to_end_succeeds():
    hrs_contract = _base_hrs_contract("top_hrs_valid.txt")
    readable_contract = _base_readable_contract("top_readable_valid.txt")
    result = load_formation_top_well(
        str(FIXTURES / "top_hrs_valid.txt"), str(FIXTURES / "top_readable_valid.txt"),
        hrs_contract, readable_contract, {}, _VERTICAL_WELL,
    )
    assert result.contract_status == "PASSED"
    assert len(result.markers) == 3
    assert all(m.mapping_status == "mapped_within_coverage" for m in result.markers)


def test_load_formation_top_well_raises_contract_error_on_sha_mismatch():
    hrs_contract = _base_hrs_contract("top_hrs_valid.txt", expected_sha256="0" * 64)
    readable_contract = _base_readable_contract("top_readable_valid.txt")
    with pytest.raises(TopContractError):
        load_formation_top_well(
            str(FIXTURES / "top_hrs_valid.txt"), str(FIXTURES / "top_readable_valid.txt"),
            hrs_contract, readable_contract, {}, _VERTICAL_WELL,
        )


def test_load_formation_top_well_raises_reconciliation_error_on_duplicate():
    hrs_contract = _base_hrs_contract(
        "top_hrs_duplicate_marker.txt", expected_marker_count=2, expected_mdrt_min_m=100.0, expected_mdrt_max_m=150.0
    )
    readable_contract = _base_readable_contract("top_readable_valid.txt")
    with pytest.raises(TopSourceReconciliationError):
        load_formation_top_well(
            str(FIXTURES / "top_hrs_duplicate_marker.txt"), str(FIXTURES / "top_readable_valid.txt"),
            hrs_contract, readable_contract, {}, _VERTICAL_WELL,
        )


def test_contract_definition_duplicate_yaml_key_rejected(tmp_path):
    bad_yaml = tmp_path / "bad_formation_top_contracts.yml"
    bad_yaml.write_text(
        "files:\n"
        "  \"dup.txt\":\n"
        "    expected_sha256: \"a\"\n"
        "  \"dup.txt\":\n"
        "    expected_sha256: \"b\"\n"
    )
    with pytest.raises(TopContractDefinitionError):
        load_formation_top_contract_config(str(bad_yaml))


def test_contract_definition_invalid_representation_type_rejected(tmp_path):
    bad_yaml = tmp_path / "bad2.yml"
    bad_yaml.write_text(
        "files:\n"
        "  \"x.txt\":\n"
        "    expected_sha256: \"a\"\n"
        "    project_well_key: \"W\"\n"
        "    representation_type: \"not_a_real_type\"\n"
        "    well_identity_evidence_status: \"inferred_unverified\"\n"
        "    well_identity_evidence_notes: \"n\"\n"
        "    model_use_status: \"primary_model\"\n"
        "    expected_column_header_line: \"h\"\n"
        "    expected_column_order: [\"a\", \"b\"]\n"
        "    expected_marker_count: 1\n"
        "    expected_mdrt_min_m: 0.0\n"
        "    expected_mdrt_max_m: 1.0\n"
        "    numeric_range_tolerance: 0.01\n"
        "    datum_depth_column_interpretation: \"d\"\n"
        "    notes: \"n\"\n"
    )
    with pytest.raises(TopContractDefinitionError):
        load_formation_top_contract_config(str(bad_yaml))


# ---------------------------------------------------------------------------
# Batch failure isolation and absolute-path sanitization
# ---------------------------------------------------------------------------
def test_batch_isolates_one_well_contract_failure_from_another_success(tmp_path):
    hrs_contracts = {
        "top_hrs_valid.txt": _base_hrs_contract("top_hrs_valid.txt"),
        "top_hrs_duplicate_marker.txt": _base_hrs_contract(
            "top_hrs_duplicate_marker.txt", project_well_key="Test_Well_2",
            expected_marker_count=2, expected_mdrt_min_m=100.0, expected_mdrt_max_m=150.0,
        ),
    }
    readable_contracts = {
        "top_readable_valid.txt": _base_readable_contract("top_readable_valid.txt"),
    }
    # Test_Well_2 has no readable contract at all in this deliberately
    # incomplete map - simulate via a missing readable file path instead,
    # reusing the same readable file (well-2's HRS has a duplicate-name
    # defect regardless, which is what this test isolates).
    file_paths = {
        "Test_Well_1": (str(FIXTURES / "top_hrs_valid.txt"), str(FIXTURES / "top_readable_valid.txt")),
        "Test_Well_2": (str(FIXTURES / "top_hrs_duplicate_marker.txt"), str(FIXTURES / "top_readable_valid.txt")),
    }
    readable_contracts["top_readable_valid.txt"] = _base_readable_contract("top_readable_valid.txt")
    results, failures = load_formation_top_surveys(
        file_paths, hrs_contracts, readable_contracts, {},
        {"Test_Well_1": _VERTICAL_WELL, "Test_Well_2": _VERTICAL_WELL},
    )
    assert "Test_Well_1" in results
    assert "Test_Well_2" in failures
    assert failures["Test_Well_2"].error_type == "reconciliation_failure"
    # The failure message must never leak an absolute path.
    assert str(FIXTURES) not in failures["Test_Well_2"].message


def test_batch_missing_contract_raises_contract_definition_error():
    file_paths = {"Test_Well_1": (str(FIXTURES / "top_hrs_valid.txt"), str(FIXTURES / "top_readable_valid.txt"))}
    with pytest.raises(TopContractDefinitionError):
        load_formation_top_surveys(file_paths, {}, {}, {}, {"Test_Well_1": _VERTICAL_WELL})


def test_batch_missing_deviation_result_raises_value_error():
    hrs_contracts = {"top_hrs_valid.txt": _base_hrs_contract("top_hrs_valid.txt")}
    readable_contracts = {"top_readable_valid.txt": _base_readable_contract("top_readable_valid.txt")}
    file_paths = {"Test_Well_1": (str(FIXTURES / "top_hrs_valid.txt"), str(FIXTURES / "top_readable_valid.txt"))}
    with pytest.raises(ValueError):
        load_formation_top_surveys(file_paths, hrs_contracts, readable_contracts, {}, {})


# ===========================================================================
# Increment 5.1 - corrective-patch regression tests
# ===========================================================================
# Finding 1: zero-common-marker logic defect (disjoint non-empty sources
# were silently accepted as one-sided NOT_COMPARABLE entries instead of a
# fatal NO_COMMON_MARKERS ERROR).
# ---------------------------------------------------------------------------
def test_disjoint_nonempty_sources_produce_no_common_markers_error():
    """Two entirely disjoint, non-empty marker sets (HRS={Marker X, Marker
    Y}, readable={Marker A, Marker B, Marker C}) must raise a fatal
    NO_COMMON_MARKERS ERROR - never be silently accepted as five one-sided
    NOT_COMPARABLE entries."""
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_disjoint_markers.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    assert any(i.code == "NO_COMMON_MARKERS" and i.severity == "ERROR" for i in issues)
    # The entries are still constructed (for audit visibility) but every one
    # of them is one-sided - no fuzzy pairing is invented to "fix" this.
    assert len(entries) == 5
    assert all(e.mdrt_status == "NOT_COMPARABLE" for e in entries)


def test_load_formation_top_well_raises_reconciliation_error_for_disjoint_sources():
    """`load_formation_top_well()` must consequently raise
    `TopSourceReconciliationError` for the disjoint-non-empty-sources
    condition - it must never construct/map an apparently valid model from
    two unrelated marker sets."""
    hrs_contract = _base_hrs_contract(
        "top_hrs_disjoint_markers.txt", expected_marker_count=2,
        expected_mdrt_min_m=100.0, expected_mdrt_max_m=200.0,
    )
    readable_contract = _base_readable_contract("top_readable_valid.txt")
    with pytest.raises(TopSourceReconciliationError) as excinfo:
        load_formation_top_well(
            str(FIXTURES / "top_hrs_disjoint_markers.txt"), str(FIXTURES / "top_readable_valid.txt"),
            hrs_contract, readable_contract, {}, _VERTICAL_WELL,
        )
    assert any(i.code == "NO_COMMON_MARKERS" for i in excinfo.value.issues)


def test_common_subset_plus_one_sided_markers_remains_auditable_nonfatal():
    """When at least one canonical marker IS shared between the two
    sources, a legitimate one-sided marker (present in only one source)
    remains a non-fatal, auditable NOT_COMPARABLE entry - the Finding 1 fix
    must not turn this already-correct, already-tested behavior into a
    false-positive NO_COMMON_MARKERS ERROR."""
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))  # Marker A, B, C
    missing = read_readable_top_rows(str(FIXTURES / "top_readable_missing_marker.txt"))  # missing Marker C
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, missing, {})
    assert not any(i.code == "NO_COMMON_MARKERS" for i in issues)
    assert not any(i.severity == "ERROR" for i in issues)
    c_entry = next(e for e in entries if e.canonical_marker_name == "Marker C")
    assert c_entry.mdrt_status == "NOT_COMPARABLE"
    assert c_entry.name_match_status == "missing_in_readable"


# ===========================================================================
# Increment 5.1.1 - corrective-patch regression tests
# ===========================================================================
# Finding 1: one-empty-source zero-common-marker defect (Increment 5.1
# correctly rejected two non-empty disjoint sources, but still silently
# accepted the case where exactly ONE source is entirely empty and the
# other is not - the intersection with an empty set is itself empty, so
# this is still a zero-common-markers condition and must also be fatal).
# ---------------------------------------------------------------------------
def test_no_common_markers_hrs_empty_readable_nonempty():
    """HRS entirely empty, readable contains a marker: must raise a fatal
    NO_COMMON_MARKERS ERROR, never silently accept the marker as a bare
    'missing_in_hrs' NOT_COMPARABLE entry with no ERROR."""
    from p2mem.top_models import HRSTopStationData

    hrs_empty = HRSTopStationData(Top_Name_source=(), MDRT_source_m=np.array([]))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs_empty, readable, {})
    assert any(i.code == "NO_COMMON_MARKERS" and i.severity == "ERROR" for i in issues)
    assert all(e.mdrt_status == "NOT_COMPARABLE" for e in entries)


def test_no_common_markers_hrs_nonempty_readable_empty():
    """HRS contains a marker, readable entirely empty: must raise a fatal
    NO_COMMON_MARKERS ERROR, never silently accept the marker as a bare
    'missing_in_readable' NOT_COMPARABLE entry with no ERROR."""
    from p2mem.top_models import ReadableTopStationData

    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable_empty = ReadableTopStationData(
        TOP_NAME_source=(), MDRT_source_m=np.array([]), TVDSS_source_m=np.array([]), NOTE_source=(),
    )
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable_empty, {})
    assert any(i.code == "NO_COMMON_MARKERS" and i.severity == "ERROR" for i in issues)
    assert all(e.mdrt_status == "NOT_COMPARABLE" for e in entries)


def test_no_common_markers_both_sources_empty():
    """Both sources entirely empty: the pre-existing, already-correct
    behavior (fatal NO_COMMON_MARKERS ERROR) must remain unaffected by the
    Increment 5.1.1 condition rewrite."""
    from p2mem.top_models import HRSTopStationData, ReadableTopStationData

    hrs_empty = HRSTopStationData(Top_Name_source=(), MDRT_source_m=np.array([]))
    readable_empty = ReadableTopStationData(
        TOP_NAME_source=(), MDRT_source_m=np.array([]), TVDSS_source_m=np.array([]), NOTE_source=(),
    )
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs_empty, readable_empty, {})
    assert any(i.code == "NO_COMMON_MARKERS" and i.severity == "ERROR" for i in issues)
    assert entries == ()


def test_no_common_markers_both_nonempty_disjoint_still_rejected():
    """Both sources non-empty but disjoint (the Increment 5.1 case): must
    remain rejected under the Increment 5.1.1 simplified single-condition
    check - this is a non-regression check on the 5.1 fix itself."""
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_disjoint_markers.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, readable, {})
    assert any(i.code == "NO_COMMON_MARKERS" and i.severity == "ERROR" for i in issues)
    assert all(e.mdrt_status == "NOT_COMPARABLE" for e in entries)


def test_load_formation_top_well_raises_reconciliation_error_for_one_empty_source():
    """`load_formation_top_well()` must consequently raise
    `TopSourceReconciliationError` for the one-empty-source condition too -
    never construct/map an apparently valid model when one entire source
    supplied zero markers."""
    from p2mem.top_models import ReadableTopStationData

    hrs_contract = _base_hrs_contract("top_hrs_valid.txt")
    readable_contract = _base_readable_contract(
        "top_readable_valid.txt", expected_marker_count=0, expected_mdrt_min_m=0.0, expected_mdrt_max_m=0.0,
        expected_tvdss_min_m=0.0, expected_tvdss_max_m=0.0,
    )
    # Directly exercise reconcile_formation_top_sources with an empty
    # readable source (constructing a genuinely empty readable FILE would
    # itself be rejected by the file parser as "no data rows found" before
    # reconciliation is ever reached - the in-memory API is what Finding 1
    # of Increment 5.1.1 targets).
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable_empty = ReadableTopStationData(
        TOP_NAME_source=(), MDRT_source_m=np.array([]), TVDSS_source_m=np.array([]), NOTE_source=(),
    )
    issues, _ = reconcile_formation_top_sources("Test_Well_1", hrs, readable_empty, {})
    assert any(i.severity == "ERROR" and i.code == "NO_COMMON_MARKERS" for i in issues)
    # The same ERROR-severity-issue-to-exception conversion already used by
    # load_formation_top_well for every other reconciliation ERROR applies
    # here unchanged - confirmed via the existing duplicate-marker case
    # (test_load_formation_top_well_raises_reconciliation_error_on_duplicate)
    # and the disjoint-sources case above; this test isolates the specific
    # one-empty-source reconciliation-level ERROR that must feed that same
    # conversion.


def test_common_subset_plus_one_sided_markers_still_nonfatal_after_5_1_1():
    """Re-verification (post Increment 5.1.1 condition rewrite) that a
    genuinely shared marker alongside a one-sided marker remains the
    pre-existing, non-fatal NOT_COMPARABLE case - the single-condition
    `not common_markers` check must not be broader than intended."""
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))  # Marker A, B, C
    missing = read_readable_top_rows(str(FIXTURES / "top_readable_missing_marker.txt"))  # missing Marker C
    issues, entries = reconcile_formation_top_sources("Test_Well_1", hrs, missing, {})
    assert not any(i.code == "NO_COMMON_MARKERS" for i in issues)
    assert not any(i.severity == "ERROR" for i in issues)
    matched = [e for e in entries if e.mdrt_status == "MATCHED"]
    assert len(matched) >= 1
    c_entry = next(e for e in entries if e.canonical_marker_name == "Marker C")
    assert c_entry.mdrt_status == "NOT_COMPARABLE"


# ---------------------------------------------------------------------------
# Finding 2: readable-file absolute-path leakage (the batch loader recorded
# only the HRS path for every failure type, regardless of which file/stage
# actually failed).
# ---------------------------------------------------------------------------
def test_batch_missing_readable_file_reports_readable_origin_and_both_paths():
    hrs_contracts = {"top_hrs_valid.txt": _base_hrs_contract("top_hrs_valid.txt")}
    # Contracts are keyed by source filename, not by file existence - a
    # contract entry for the (missing) readable file must exist so the
    # batch loader reaches `load_formation_top_well` and genuinely fails
    # at the file-not-found stage, not at contract lookup.
    readable_contracts = {
        "does_not_exist_readable.txt": _base_readable_contract("top_readable_valid.txt", source_filename="does_not_exist_readable.txt"),
    }
    missing_readable_path = str(FIXTURES / "does_not_exist_readable.txt")
    file_paths = {"Test_Well_1": (str(FIXTURES / "top_hrs_valid.txt"), missing_readable_path)}
    _, failures = load_formation_top_surveys(
        file_paths, hrs_contracts, readable_contracts, {}, {"Test_Well_1": _VERTICAL_WELL}
    )
    f = failures["Test_Well_1"]
    assert f.error_type == "file_not_found"
    assert f.failure_origin == "readable"
    assert f.readable_path == missing_readable_path
    assert f.hrs_path == str(FIXTURES / "top_hrs_valid.txt")
    # The absolute readable path must appear in the raw exception message
    # (it is a genuine, unsanitized internal record) ...
    assert missing_readable_path in f.message
    # ... but must never survive into any exported builder row/manifest field.
    from p2mem.io.tops_inventory import build_formation_top_manifest, build_top_availability_rows, build_top_issues_rows

    issues_rows = build_top_issues_rows({}, failures)
    assert not any(str(FIXTURES) in row["message"] for row in issues_rows)
    assert issues_rows[0]["context"] == "does_not_exist_readable.txt"
    availability_rows = build_top_availability_rows({}, failures, {})
    assert not any(str(FIXTURES) in row["notes"] for row in availability_rows)
    manifest = build_formation_top_manifest({}, failures, {})
    assert str(FIXTURES) not in manifest["failed_wells"]["Test_Well_1"]["message"]


def test_batch_malformed_readable_file_reports_readable_origin():
    hrs_contracts = {"top_hrs_valid.txt": _base_hrs_contract("top_hrs_valid.txt")}
    readable_contracts = {
        "top_readable_malformed_numeric.txt": _base_readable_contract(
            "top_readable_malformed_numeric.txt", expected_marker_count=1,
            expected_mdrt_min_m=0.0, expected_mdrt_max_m=1000.0,
            expected_tvdss_min_m=0.0, expected_tvdss_max_m=1000.0,
        ),
    }
    readable_path = str(FIXTURES / "top_readable_malformed_numeric.txt")
    file_paths = {"Test_Well_1": (str(FIXTURES / "top_hrs_valid.txt"), readable_path)}
    _, failures = load_formation_top_surveys(
        file_paths, hrs_contracts, readable_contracts, {}, {"Test_Well_1": _VERTICAL_WELL}
    )
    f = failures["Test_Well_1"]
    assert f.error_type == "parsing_failure"
    assert f.failure_origin == "readable"
    assert f.readable_path == readable_path

    from p2mem.io.tops_inventory import build_top_availability_rows, build_top_issues_rows

    issues_rows = build_top_issues_rows({}, failures)
    assert not any(str(FIXTURES) in row["message"] for row in issues_rows)
    assert issues_rows[0]["context"] == "top_readable_malformed_numeric.txt"
    availability_rows = build_top_availability_rows({}, failures, {})
    assert not any(str(FIXTURES) in row["notes"] for row in availability_rows)


def test_batch_readable_contract_failure_reports_readable_origin():
    hrs_contracts = {"top_hrs_valid.txt": _base_hrs_contract("top_hrs_valid.txt")}
    # A deliberately wrong expected_sha256 forces a TopContractError whose
    # message embeds the readable file's own absolute path.
    readable_contracts = {"top_readable_valid.txt": _base_readable_contract("top_readable_valid.txt", expected_sha256="0" * 64)}
    readable_path = str(FIXTURES / "top_readable_valid.txt")
    file_paths = {"Test_Well_1": (str(FIXTURES / "top_hrs_valid.txt"), readable_path)}
    _, failures = load_formation_top_surveys(
        file_paths, hrs_contracts, readable_contracts, {}, {"Test_Well_1": _VERTICAL_WELL}
    )
    f = failures["Test_Well_1"]
    assert f.error_type == "contract_failure"
    assert f.failure_origin == "readable"
    assert f.readable_path == readable_path
    assert readable_path in f.message

    from p2mem.io.tops_inventory import build_top_issues_rows

    issues_rows = build_top_issues_rows({}, failures)
    assert not any(str(FIXTURES) in row["message"] for row in issues_rows)
    assert issues_rows[0]["context"] == "top_readable_valid.txt"


def test_batch_hrs_failure_still_reports_hrs_origin_and_sanitizes_correctly():
    """The pre-existing HRS-failure sanitization path must remain correct
    after the Finding 2 restructuring."""
    hrs_contracts = {"top_hrs_valid.txt": _base_hrs_contract("top_hrs_valid.txt", expected_sha256="0" * 64)}
    readable_contracts = {"top_readable_valid.txt": _base_readable_contract("top_readable_valid.txt")}
    hrs_path = str(FIXTURES / "top_hrs_valid.txt")
    file_paths = {"Test_Well_1": (hrs_path, str(FIXTURES / "top_readable_valid.txt"))}
    _, failures = load_formation_top_surveys(
        file_paths, hrs_contracts, readable_contracts, {}, {"Test_Well_1": _VERTICAL_WELL}
    )
    f = failures["Test_Well_1"]
    assert f.error_type == "contract_failure"
    assert f.failure_origin == "hrs"
    assert f.hrs_path == hrs_path

    from p2mem.io.tops_inventory import build_top_issues_rows

    issues_rows = build_top_issues_rows({}, failures)
    assert not any(str(FIXTURES) in row["message"] for row in issues_rows)
    assert issues_rows[0]["context"] == "top_hrs_valid.txt"


# ---------------------------------------------------------------------------
# Finding 3: incomplete caller-supplied numerical validation of
# `reconcile_formation_top_sources()`.
# ---------------------------------------------------------------------------
def test_reconcile_rejects_nan_in_hrs_mdrt():
    from p2mem.top_models import HRSTopStationData

    hrs_bad = HRSTopStationData(Top_Name_source=("Marker A",), MDRT_source_m=np.array([np.nan]))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs_bad, readable, {})


def test_reconcile_rejects_inf_in_readable_mdrt():
    from p2mem.top_models import ReadableTopStationData

    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable_bad = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B", "Marker C"),
        MDRT_source_m=np.array([100.0, np.inf, 300.0]),
        TVDSS_source_m=np.array([95.0, 190.0, 285.0]),
        NOTE_source=("", "", ""),
    )
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs, readable_bad, {})


def test_reconcile_rejects_nan_in_readable_tvdss():
    from p2mem.top_models import ReadableTopStationData

    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable_bad = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B", "Marker C"),
        MDRT_source_m=np.array([100.0, 200.0, 300.0]),
        TVDSS_source_m=np.array([95.0, np.nan, 285.0]),
        NOTE_source=("", "", ""),
    )
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs, readable_bad, {})


def test_reconcile_rejects_negative_hrs_mdrt():
    from p2mem.top_models import HRSTopStationData

    hrs_bad = HRSTopStationData(Top_Name_source=("Marker A",), MDRT_source_m=np.array([-5.0]))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs_bad, readable, {})


def test_reconcile_rejects_non_1d_array():
    from p2mem.top_models import HRSTopStationData

    hrs_bad = HRSTopStationData(
        Top_Name_source=("Marker A", "Marker B"), MDRT_source_m=np.array([[100.0], [200.0]])
    )
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs_bad, readable, {})


def test_reconcile_rejects_shorter_note_source_with_typed_error_not_indexerror():
    """A shorter `NOTE_source` tuple must be rejected with a documented
    `TopParsingError` before any reconciliation - never allowed to reach an
    untyped `IndexError` deep inside the reconciliation loop."""
    from p2mem.top_models import ReadableTopStationData

    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable_bad = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B", "Marker C"),
        MDRT_source_m=np.array([100.0, 200.0, 300.0]),
        TVDSS_source_m=np.array([95.0, 190.0, 285.0]),
        NOTE_source=("only one note",),  # length 1, not 3
    )
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs, readable_bad, {})


@pytest.mark.parametrize("bad_tolerance", [np.nan, np.inf, -np.inf, -0.01])
def test_reconcile_rejects_invalid_tolerance_values(bad_tolerance):
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TopParsingError):
        reconcile_formation_top_sources("Test_Well_1", hrs, readable, {}, mdrt_agreement_tolerance_m=bad_tolerance)


@pytest.mark.parametrize("bad_tolerance", [True, False, "0.5", b"0.5", 1 + 2j])
def test_reconcile_rejects_invalid_tolerance_types(bad_tolerance):
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    with pytest.raises(TypeError):
        reconcile_formation_top_sources("Test_Well_1", hrs, readable, {}, mdrt_agreement_tolerance_m=bad_tolerance)


@pytest.mark.parametrize(
    "good_tolerance",
    [0.5, 1, np.float64(0.5), np.int32(1), np.array(0.5), np.array([0.5])],
)
def test_reconcile_accepts_valid_tolerance_scalars_and_arrays(good_tolerance):
    hrs = read_hrs_top_rows(str(FIXTURES / "top_hrs_valid.txt"))
    readable = read_readable_top_rows(str(FIXTURES / "top_readable_valid.txt"))
    issues, entries = reconcile_formation_top_sources(
        "Test_Well_1", hrs, readable, {}, mdrt_agreement_tolerance_m=good_tolerance
    )
    assert not any(i.severity == "ERROR" for i in issues)
    assert len(entries) == 3


# ---------------------------------------------------------------------------
# Finding 3 (readable-format-equivalent coverage): the readable-file parser
# already enforces malformed/non-finite/negative-MDRT rejection (unlike the
# in-memory reconciliation API's prior gap above), but only HRS-format
# fixtures previously existed for these three conditions - this closes that
# coverage gap for the readable format.
# ---------------------------------------------------------------------------
@pytest.mark.parametrize(
    "fixture",
    ["top_readable_malformed_numeric.txt", "top_readable_nonfinite.txt", "top_readable_negative_depth.txt"],
)
def test_readable_malformed_or_nonfinite_or_negative_depth_rejected(fixture):
    with pytest.raises(TopParsingError):
        read_readable_top_rows(str(FIXTURES / fixture))


In [ ]:
%%writefile tests/test_tops_inventory.py
"""
tests/test_tops_inventory.py - Validation suite for p2mem.io.tops_inventory
(Increment 5: deterministic, metadata-only inventory-table builders for the
formation-top ingestion/reconciliation/survey-mapping layer).

PORTABLE unit tests only: hand-built typed result objects (mirroring
tests/test_checkshot_inventory.py's pattern), never the real project top
files. Verifies (a) correct row content, (b) deterministic/sorted ordering,
(c) environment-independent output (no absolute-path leakage in a failure
row's sanitized message), and (d) Tier C classification and well-status
disclosure in the JSON manifest.
"""

import numpy as np
import pytest

from p2mem.top_models import (
    FormationTopAvailabilityRecord,
    FormationTopWellResult,
    HRSTopFileContract,
    HRSTopHeaderInfo,
    HRSTopStationData,
    ReadableTopFileContract,
    ReadableTopHeaderInfo,
    ReadableTopStationData,
    TopIngestionFailure,
    TopIngestionIssue,
    TopMarkerRecord,
    TopReconciliationEntry,
)
from p2mem.io.tops_inventory import (
    build_formation_top_manifest,
    build_top_availability_rows,
    build_top_corrected_marker_rows,
    build_top_file_inventory_rows,
    build_top_issues_rows,
    build_top_marker_register_rows,
    build_top_reconciliation_rows,
)


def _make_hrs_contract(well_key) -> HRSTopFileContract:
    return HRSTopFileContract(
        source_filename=f"{well_key}_HRS_tops_no_wellname_MDRT.txt",
        expected_sha256="a" * 64,
        project_well_key=well_key,
        representation_type="HRS_MDRT_only",
        well_identity_evidence_status="inferred_unverified",
        well_identity_evidence_notes="filename-only association; never content-verified.",
        model_use_status="primary_model",
        expected_column_header_line="Top_Name\tMDRT_m",
        expected_column_order=("Top_Name", "MDRT_m"),
        expected_column_count=2,
        expected_marker_count=2,
        expected_mdrt_min_m=100.0,
        expected_mdrt_max_m=200.0,
        numeric_range_tolerance=0.005,
        datum_depth_column_interpretation="MDRT referenced to rotary table, increasing downward.",
        notes="test",
    )


def _make_readable_contract(well_key) -> ReadableTopFileContract:
    return ReadableTopFileContract(
        source_filename=f"{well_key}_selected_well_tops.txt",
        expected_sha256="b" * 64,
        project_well_key=well_key,
        representation_type="selected_readable_MDRT_TVDSS",
        well_identity_evidence_status="inferred_unverified",
        well_identity_evidence_notes="project-supplied in-file comment only; never content-verified.",
        model_use_status="primary_model",
        expected_well_name_from_comment=well_key.replace("_", " "),
        expected_column_header_line="TOP_NAME\tMDRT_M\tTVDSS_M\tNOTE",
        expected_column_order=("TOP_NAME", "MDRT_M", "TVDSS_M", "NOTE"),
        expected_column_count=4,
        expected_marker_count=2,
        expected_mdrt_min_m=100.0,
        expected_mdrt_max_m=200.0,
        expected_tvdss_min_m=95.0,
        expected_tvdss_max_m=190.0,
        numeric_range_tolerance=0.005,
        datum_depth_column_interpretation="MDRT referenced to rotary table; TVDSS is the file's own supplied value.",
        notes="test",
    )


def _make_result(well_key, issues=()) -> FormationTopWellResult:
    hrs_contract = _make_hrs_contract(well_key)
    readable_contract = _make_readable_contract(well_key)
    hrs_header = HRSTopHeaderInfo(
        source_filename=hrs_contract.source_filename, sha256="a" * 64,
        column_header_line="Top_Name\tMDRT_m", column_names=("Top_Name", "MDRT_m"),
        well_identity_source="filename_only", line_ending_convention="LF",
        header_line_count=1, data_line_offset=1,
    )
    readable_header = ReadableTopHeaderInfo(
        source_filename=readable_contract.source_filename, sha256="b" * 64,
        comment_lines=(f"# Selected readable well tops for {well_key.replace('_', ' ')}",),
        well_name_from_comment=well_key.replace("_", " "),
        column_header_line="TOP_NAME\tMDRT_M\tTVDSS_M\tNOTE",
        column_names=("TOP_NAME", "MDRT_M", "TVDSS_M", "NOTE"),
        separator_line_raw="-" * 40, well_identity_source="in_file_comment_project_supplied",
        line_ending_convention="LF", header_line_count=4, data_line_offset=4,
    )
    hrs_raw = HRSTopStationData(
        Top_Name_source=("Marker A", "Marker B"), MDRT_source_m=np.array([100.0, 200.0]),
    )
    readable_raw = ReadableTopStationData(
        TOP_NAME_source=("Marker A", "Marker B"), MDRT_source_m=np.array([100.0, 200.0]),
        TVDSS_source_m=np.array([95.0, 190.0]), NOTE_source=("", ""),
    )
    reconciliation = (
        TopReconciliationEntry(
            well_key=well_key, canonical_marker_name="Marker A",
            hrs_marker_name_raw="Marker A", readable_marker_name_raw="Marker A",
            hrs_row_number=0, readable_row_number=0,
            present_in_hrs=True, present_in_readable=True, name_match_status="exact",
            MDRT_source_hrs_m=100.0, MDRT_source_readable_m=100.0,
            MDRT_agreement_readable_minus_hrs_m=0.0, mdrt_status="MATCHED",
            TVDSS_source_readable_m=95.0, note_readable="",
            reconciliation_notes="Present in both sources; MDRT agrees within tolerance.",
        ),
        TopReconciliationEntry(
            well_key=well_key, canonical_marker_name="Marker B",
            hrs_marker_name_raw="Marker B", readable_marker_name_raw="Marker B",
            hrs_row_number=1, readable_row_number=1,
            present_in_hrs=True, present_in_readable=True, name_match_status="exact",
            MDRT_source_hrs_m=200.0, MDRT_source_readable_m=200.0,
            MDRT_agreement_readable_minus_hrs_m=0.0, mdrt_status="MATCHED",
            TVDSS_source_readable_m=190.0, note_readable="",
            reconciliation_notes="Present in both sources; MDRT agrees within tolerance.",
        ),
    )
    markers = (
        TopMarkerRecord(
            well_key=well_key, canonical_marker_name="Marker A",
            MDRT_source_hrs_m=100.0, MDRT_source_readable_m=100.0, MDRT_reconciled_m=100.0,
            mdrt_authority_basis="hrs_and_readable_agree", TVDSS_source_m=95.0,
            depth_basis_used="petrel_source_trace", interpolation_method="piecewise_linear_station_interpolation",
            TVD_survey_m=99.5, TVDSS_survey_corrected_m=79.5,
            TVDSS_residual_source_minus_survey_m=15.5, mapping_status="mapped_within_coverage",
            well_identity_evidence_status="inferred_unverified", notes="mapped",
        ),
        TopMarkerRecord(
            well_key=well_key, canonical_marker_name="Marker B",
            MDRT_source_hrs_m=200.0, MDRT_source_readable_m=200.0, MDRT_reconciled_m=200.0,
            mdrt_authority_basis="hrs_and_readable_agree", TVDSS_source_m=190.0,
            depth_basis_used="petrel_source_trace", interpolation_method="piecewise_linear_station_interpolation",
            TVD_survey_m=194.6, TVDSS_survey_corrected_m=174.6,
            TVDSS_residual_source_minus_survey_m=15.4, mapping_status="mapped_within_coverage",
            well_identity_evidence_status="inferred_unverified", notes="mapped",
        ),
    )
    return FormationTopWellResult(
        well_key=well_key, hrs_header=hrs_header, hrs_contract=hrs_contract, hrs_raw=hrs_raw,
        readable_header=readable_header, readable_contract=readable_contract, readable_raw=readable_raw,
        reconciliation=reconciliation, markers=markers, issues=issues, contract_status="PASSED",
    )


# ---------------------------------------------------------------------------
# File inventory rows
# ---------------------------------------------------------------------------
def test_file_inventory_rows_two_rows_per_well_plus_failures_and_availability():
    results = {"Boreas_1": _make_result("Boreas_1")}
    failures = {
        "Broken_Well": TopIngestionFailure(
            well_key="Broken_Well", source_path="/abs/path/to/Broken_Well_HRS_tops_no_wellname_MDRT.txt",
            error_type="parsing_failure", message="/abs/path/to/Broken_Well_HRS_tops_no_wellname_MDRT.txt: bad row",
            exception=ValueError("bad row"),
        )
    }
    availability = {
        "Poseidon_North_1": FormationTopAvailabilityRecord(
            "Poseidon_North_1", "NOT_AVAILABLE", "No approved formation-top file exists."
        )
    }
    rows = build_top_file_inventory_rows(results, failures, availability)
    well_keys = [r["well_key"] for r in rows]
    # Two rows for Boreas_1 (HRS + readable), one each for the failure and
    # the NOT_AVAILABLE well.
    assert well_keys.count("Boreas_1") == 2
    assert well_keys.count("Broken_Well") == 1
    assert well_keys.count("Poseidon_North_1") == 1
    assert well_keys == sorted(well_keys)

    boreas_rows = [r for r in rows if r["well_key"] == "Boreas_1"]
    hrs_row = next(r for r in boreas_rows if r["representation_type"] == "HRS_MDRT_only")
    readable_row = next(r for r in boreas_rows if r["representation_type"] == "selected_readable_MDRT_TVDSS")
    assert hrs_row["tvdss_min_m"] == ""  # HRS format has no TVDSS column
    assert readable_row["tvdss_min_m"] == pytest.approx(95.0)
    assert hrs_row["n_markers"] == 2
    assert hrs_row["contract_status"] == "PASSED"

    broken_row = next(r for r in rows if r["well_key"] == "Broken_Well")
    assert broken_row["contract_status"] == "FAILED"

    na_row = next(r for r in rows if r["well_key"] == "Poseidon_North_1")
    assert na_row["contract_status"] == "NOT_AVAILABLE"


def test_file_inventory_rows_count_errors_and_warnings_per_source_file():
    issues = (
        TopIngestionIssue("WARNING", "MDRT_MISMATCH", "msg", "Poseidon_2_HRS_tops_no_wellname_MDRT.txt"),
        TopIngestionIssue("ERROR", "DUPLICATE_MARKER_NAME", "msg", "Poseidon_2_selected_well_tops.txt"),
    )
    results = {"Poseidon_2": _make_result("Poseidon_2", issues=issues)}
    rows = build_top_file_inventory_rows(results, {}, {})
    hrs_row = next(r for r in rows if r["representation_type"] == "HRS_MDRT_only")
    readable_row = next(r for r in rows if r["representation_type"] == "selected_readable_MDRT_TVDSS")
    assert hrs_row["n_warnings"] == 1 and hrs_row["n_errors"] == 0
    assert readable_row["n_errors"] == 1 and readable_row["n_warnings"] == 0


# ---------------------------------------------------------------------------
# Marker / source register rows
# ---------------------------------------------------------------------------
def test_marker_register_rows_preserve_raw_source_values_per_file():
    results = {"Poseidon_2": _make_result("Poseidon_2")}
    rows = build_top_marker_register_rows(results)
    # 2 HRS rows + 2 readable rows = 4 total for this one well.
    assert len(rows) == 4
    hrs_rows = [r for r in rows if r["source_representation"] == "HRS_MDRT_only"]
    readable_rows = [r for r in rows if r["source_representation"] == "selected_readable_MDRT_TVDSS"]
    assert len(hrs_rows) == 2 and len(readable_rows) == 2
    assert hrs_rows[0]["TVDSS_source_m"] == ""  # HRS carries no TVDSS at all
    assert readable_rows[0]["TVDSS_source_m"] == pytest.approx(95.0)
    assert hrs_rows[0]["well_identity_source"] == "filename_only"
    assert readable_rows[0]["well_identity_source"] == "in_file_comment_project_supplied"
    # Row numbers trace back to original (0-indexed) file order.
    assert [r["source_row_number"] for r in hrs_rows] == [0, 1]


def test_marker_register_rows_sorted_by_well_key():
    results = {"Poseidon_2": _make_result("Poseidon_2"), "Boreas_1": _make_result("Boreas_1")}
    rows = build_top_marker_register_rows(results)
    well_keys = [r["well_key"] for r in rows]
    assert well_keys == sorted(well_keys)
    assert well_keys[0] == "Boreas_1"


# ---------------------------------------------------------------------------
# HRS-versus-readable reconciliation rows
# ---------------------------------------------------------------------------
def test_reconciliation_rows_content_and_none_handling():
    results = {"Poseidon_2": _make_result("Poseidon_2")}
    rows = build_top_reconciliation_rows(results)
    assert len(rows) == 2
    row_a = next(r for r in rows if r["canonical_marker_name"] == "Marker A")
    assert row_a["mdrt_status"] == "MATCHED"
    assert row_a["present_in_hrs"] is True and row_a["present_in_readable"] is True
    assert row_a["MDRT_source_hrs_m"] == pytest.approx(100.0)
    assert row_a["TVDSS_source_readable_m"] == pytest.approx(95.0)


def test_reconciliation_rows_none_fields_become_empty_string():
    entry = TopReconciliationEntry(
        well_key="Test_Well", canonical_marker_name="Marker Z",
        hrs_marker_name_raw=None, readable_marker_name_raw="Marker Z",
        hrs_row_number=None, readable_row_number=0,
        present_in_hrs=False, present_in_readable=True, name_match_status="missing_in_hrs",
        MDRT_source_hrs_m=None, MDRT_source_readable_m=300.0,
        MDRT_agreement_readable_minus_hrs_m=None, mdrt_status="NOT_COMPARABLE",
        TVDSS_source_readable_m=285.0, note_readable="", reconciliation_notes="Present only in readable source.",
    )
    from dataclasses import replace
    result = replace(_make_result("Test_Well"), reconciliation=(entry,))
    rows = build_top_reconciliation_rows({"Test_Well": result})
    assert len(rows) == 1
    assert rows[0]["hrs_marker_name_raw"] == ""
    assert rows[0]["hrs_row_number"] == ""
    assert rows[0]["MDRT_source_hrs_m"] == ""
    assert rows[0]["mdrt_status"] == "NOT_COMPARABLE"


# ---------------------------------------------------------------------------
# Corrected, survey-mapped marker rows
# ---------------------------------------------------------------------------
def test_corrected_marker_rows_carry_explicit_residual_field():
    results = {"Poseidon_2": _make_result("Poseidon_2")}
    rows = build_top_corrected_marker_rows(results)
    assert len(rows) == 2
    row_a = next(r for r in rows if r["canonical_marker_name"] == "Marker A")
    assert row_a["TVDSS_residual_source_minus_survey_m"] == pytest.approx(15.5)
    assert row_a["mapping_status"] == "mapped_within_coverage"
    assert row_a["mdrt_authority_basis"] == "hrs_and_readable_agree"


def test_corrected_marker_rows_none_fields_become_empty_string_when_not_mapped():
    marker = TopMarkerRecord(
        well_key="Test_Well", canonical_marker_name="Marker Z",
        MDRT_source_hrs_m=100.0, MDRT_source_readable_m=999.0, MDRT_reconciled_m=None,
        mdrt_authority_basis="disagreement_unresolved", TVDSS_source_m=None,
        depth_basis_used=None, interpolation_method=None, TVD_survey_m=None,
        TVDSS_survey_corrected_m=None, TVDSS_residual_source_minus_survey_m=None,
        mapping_status="not_mapped_mdrt_unresolved", well_identity_evidence_status="inferred_unverified",
        notes="MDRT disagreement beyond tolerance; not silently resolved.",
    )
    from dataclasses import replace
    result = replace(_make_result("Test_Well"), markers=(marker,))
    rows = build_top_corrected_marker_rows({"Test_Well": result})
    assert len(rows) == 1
    assert rows[0]["MDRT_reconciled_m"] == ""
    assert rows[0]["TVDSS_survey_corrected_m"] == ""
    assert rows[0]["TVDSS_residual_source_minus_survey_m"] == ""
    assert rows[0]["mapping_status"] == "not_mapped_mdrt_unresolved"


# ---------------------------------------------------------------------------
# Issues rows
# ---------------------------------------------------------------------------
def test_issues_rows_include_well_issues_and_sanitize_failure_paths():
    issues = (TopIngestionIssue("WARNING", "MDRT_MISMATCH", "msg", "Poseidon_2_HRS_tops_no_wellname_MDRT.txt"),)
    results = {"Poseidon_2": _make_result("Poseidon_2", issues=issues)}
    failures = {
        "Well_X": TopIngestionFailure(
            well_key="Well_X", source_path="/home/user/secret_build_dir/Well_X_HRS_tops_no_wellname_MDRT.txt",
            error_type="file_not_found",
            message="/home/user/secret_build_dir/Well_X_HRS_tops_no_wellname_MDRT.txt not found",
            exception=FileNotFoundError(),
        )
    }
    rows = build_top_issues_rows(results, failures)
    assert len(rows) == 2
    well_row = next(r for r in rows if r["well_key"] == "Poseidon_2")
    assert well_row["code"] == "MDRT_MISMATCH" and well_row["severity"] == "WARNING"
    failure_row = next(r for r in rows if r["well_key"] == "Well_X")
    assert "/home/user/secret_build_dir/" not in failure_row["message"]
    assert failure_row["context"] == "Well_X_HRS_tops_no_wellname_MDRT.txt"


# ---------------------------------------------------------------------------
# Availability rows
# ---------------------------------------------------------------------------
def test_availability_rows_cover_available_failed_and_not_available_wells():
    results = {"Boreas_1": _make_result("Boreas_1")}
    failures = {
        "Well_X": TopIngestionFailure(
            well_key="Well_X", source_path="/abs/Well_X_HRS_tops_no_wellname_MDRT.txt",
            error_type="parsing_failure", message="/abs/Well_X_HRS_tops_no_wellname_MDRT.txt: bad row",
            exception=ValueError("bad row"),
        )
    }
    availability = {
        "Poseidon_North_1": FormationTopAvailabilityRecord(
            "Poseidon_North_1", "NOT_AVAILABLE", "No approved formation-top file exists."
        )
    }
    rows = build_top_availability_rows(results, failures, availability)
    well_keys = [r["well_key"] for r in rows]
    assert well_keys == sorted(well_keys)
    boreas = next(r for r in rows if r["well_key"] == "Boreas_1")
    assert boreas["formation_top_availability"] == "AVAILABLE"
    well_x = next(r for r in rows if r["well_key"] == "Well_X")
    assert well_x["formation_top_availability"] == "INGESTION_FAILED"
    assert "/abs/" not in well_x["notes"]
    na = next(r for r in rows if r["well_key"] == "Poseidon_North_1")
    assert na["formation_top_availability"] == "NOT_AVAILABLE"


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
def test_manifest_carries_tier_c_and_well_statuses():
    results = {"Poseidon_2": _make_result("Poseidon_2"), "Boreas_1": _make_result("Boreas_1")}
    availability = {
        "Poseidon_North_1": FormationTopAvailabilityRecord(
            "Poseidon_North_1", "NOT_AVAILABLE", "No approved formation-top file exists."
        ),
        "Proteus_1ST2": FormationTopAvailabilityRecord(
            "Proteus_1ST2", "NOT_AVAILABLE", "No approved formation-top file exists."
        ),
    }
    manifest = build_formation_top_manifest(results, {}, availability)
    assert manifest["tier_classification"].startswith("Tier C")
    assert manifest["increment"] == "5"
    assert manifest["wells"]["Poseidon_2"]["formation_top_availability"] == "AVAILABLE"
    assert manifest["wells"]["Poseidon_2"]["well_identity_evidence_status"] == "inferred_unverified"
    assert manifest["wells"]["Poseidon_2"]["n_canonical_markers"] == 2
    assert manifest["wells"]["Poseidon_2"]["n_markers_matched"] == 2
    assert manifest["wells"]["Poseidon_2"]["n_markers_mapped_within_coverage"] == 2
    assert manifest["wells"]["Poseidon_2"]["max_abs_tvdss_residual_source_minus_survey_m"] == pytest.approx(15.5)
    assert manifest["wells"]["Poseidon_North_1"]["formation_top_availability"] == "NOT_AVAILABLE"
    assert manifest["wells"]["Proteus_1ST2"]["formation_top_availability"] == "NOT_AVAILABLE"
    assert manifest["n_wells_formation_top_available"] == 2
    assert manifest["n_wells_formation_top_not_available"] == 2


def test_manifest_never_upgrades_to_verified_when_only_one_representation_is_verified():
    from dataclasses import replace

    result = _make_result("Poseidon_2")
    result = replace(result, hrs_contract=replace(result.hrs_contract, well_identity_evidence_status="verified"))
    # readable_contract remains "inferred_unverified" -> combined status must
    # NOT silently upgrade to "verified".
    manifest = build_formation_top_manifest({"Poseidon_2": result}, {}, {})
    assert manifest["wells"]["Poseidon_2"]["well_identity_evidence_status"] == "inferred_unverified"


def test_manifest_excludes_unmapped_markers_from_residual_statistics():
    from dataclasses import replace

    unresolved_marker = TopMarkerRecord(
        well_key="Poseidon_2", canonical_marker_name="Marker Z",
        MDRT_source_hrs_m=100.0, MDRT_source_readable_m=999.0, MDRT_reconciled_m=None,
        mdrt_authority_basis="disagreement_unresolved", TVDSS_source_m=None,
        depth_basis_used=None, interpolation_method=None, TVD_survey_m=None,
        TVDSS_survey_corrected_m=None, TVDSS_residual_source_minus_survey_m=None,
        mapping_status="not_mapped_mdrt_unresolved", well_identity_evidence_status="inferred_unverified",
        notes="MDRT disagreement beyond tolerance.",
    )
    base = _make_result("Poseidon_2")
    result = replace(base, markers=base.markers + (unresolved_marker,))
    manifest = build_formation_top_manifest({"Poseidon_2": result}, {}, {})
    well = manifest["wells"]["Poseidon_2"]
    assert well["n_markers_mapped_within_coverage"] == 2
    assert well["n_markers_not_mapped_mdrt_unresolved"] == 1
    # Residual max must be computed only over the two mapped markers (15.5,
    # 15.4), never inflated/deflated by the unresolved marker's None residual.
    assert well["max_abs_tvdss_residual_source_minus_survey_m"] == pytest.approx(15.5)


def test_manifest_is_json_serializable_and_deterministic():
    import json

    results = {"Poseidon_2": _make_result("Poseidon_2")}
    m1 = build_formation_top_manifest(results, {}, {})
    m2 = build_formation_top_manifest(results, {}, {})
    assert json.dumps(m1, sort_keys=True) == json.dumps(m2, sort_keys=True)


def test_manifest_sanitizes_failed_well_paths():
    failures = {
        "Well_X": TopIngestionFailure(
            well_key="Well_X", source_path="/home/user/build/Well_X_HRS_tops_no_wellname_MDRT.txt",
            error_type="file_not_found", message="/home/user/build/Well_X_HRS_tops_no_wellname_MDRT.txt missing",
            exception=FileNotFoundError(),
        )
    }
    manifest = build_formation_top_manifest({}, failures, {})
    assert "/home/user/build/" not in manifest["failed_wells"]["Well_X"]["message"]
    assert manifest["n_wells_failed"] == 1


# ===========================================================================
# Increment 5.1 - Finding 2 regression tests (readable-file absolute-path
# leakage): a failure originating from the READABLE file must have its own
# absolute path sanitized in every exported field - never only the (wrong)
# HRS path.
# ===========================================================================
def _readable_origin_failure(readable_abs_path: str, hrs_abs_path: str = "/build/Well_X_HRS_tops_no_wellname_MDRT.txt") -> TopIngestionFailure:
    return TopIngestionFailure(
        well_key="Well_X",
        source_path=readable_abs_path,  # correctly identifies the readable file as primary, post-fix
        error_type="parsing_failure",
        message=f"{readable_abs_path}: data row 1 contains a non-numeric MDRT/TVDSS token",
        exception=ValueError("bad row"),
        failure_origin="readable",
        hrs_path=hrs_abs_path,
        readable_path=readable_abs_path,
    )


def test_issues_rows_sanitize_readable_origin_failure_path():
    readable_path = "/home/private_build/Poseidon_2_selected_well_tops.txt"
    failures = {"Well_X": _readable_origin_failure(readable_path)}
    rows = build_top_issues_rows({}, failures)
    assert len(rows) == 1
    assert readable_path not in rows[0]["message"]
    assert "/home/private_build/" not in rows[0]["message"]
    # The context must identify the READABLE file, not the (unrelated) HRS path.
    assert rows[0]["context"] == "Poseidon_2_selected_well_tops.txt"


def test_availability_rows_sanitize_readable_origin_failure_path():
    readable_path = "/home/private_build/Poseidon_2_selected_well_tops.txt"
    failures = {"Well_X": _readable_origin_failure(readable_path)}
    rows = build_top_availability_rows({}, failures, {})
    row = next(r for r in rows if r["well_key"] == "Well_X")
    assert readable_path not in row["notes"]
    assert "/home/private_build/" not in row["notes"]


def test_manifest_sanitizes_readable_origin_failure_path():
    readable_path = "/home/private_build/Poseidon_2_selected_well_tops.txt"
    failures = {"Well_X": _readable_origin_failure(readable_path)}
    manifest = build_formation_top_manifest({}, failures, {})
    assert readable_path not in manifest["failed_wells"]["Well_X"]["message"]
    assert "/home/private_build/" not in manifest["failed_wells"]["Well_X"]["message"]


def test_file_inventory_rows_sanitize_readable_origin_failure_context():
    readable_path = "/home/private_build/Poseidon_2_selected_well_tops.txt"
    failures = {"Well_X": _readable_origin_failure(readable_path)}
    rows = build_top_file_inventory_rows({}, failures, {})
    row = next(r for r in rows if r["well_key"] == "Well_X")
    assert row["source_filename"] == "Poseidon_2_selected_well_tops.txt"
    assert "/home/private_build/" not in row["source_filename"]


def test_hrs_origin_failure_still_sanitizes_and_identifies_hrs_context():
    """The pre-existing HRS-failure sanitization behavior must remain
    correct after the Finding 2 restructuring (both explicit `hrs_path`
    now present, `failure_origin='hrs'`)."""
    hrs_path = "/home/user/secret_build_dir/Well_X_HRS_tops_no_wellname_MDRT.txt"
    failures = {
        "Well_X": TopIngestionFailure(
            well_key="Well_X", source_path=hrs_path, error_type="file_not_found",
            message=f"Formation-top (HRS) file not found: {hrs_path}", exception=FileNotFoundError(),
            failure_origin="hrs", hrs_path=hrs_path, readable_path="/home/user/secret_build_dir/Well_X_selected_well_tops.txt",
        )
    }
    issues_rows = build_top_issues_rows({}, failures)
    assert "/home/user/secret_build_dir/" not in issues_rows[0]["message"]
    assert issues_rows[0]["context"] == "Well_X_HRS_tops_no_wellname_MDRT.txt"
    availability_rows = build_top_availability_rows({}, failures, {})
    assert "/home/user/secret_build_dir/" not in availability_rows[0]["notes"]


def test_reconciliation_origin_failure_context_combines_both_basenames():
    """A "reconciliation"-origin failure has BOTH files successfully
    parsed - neither is individually "the" failing file, so the exported
    context retains both basenames (never silently drops one)."""
    hrs_path = "/build/Boreas_1_HRS_tops_no_wellname_MDRT.txt"
    readable_path = "/build/Boreas_1_selected_well_tops.txt"
    failures = {
        "Boreas_1": TopIngestionFailure(
            well_key="Boreas_1", source_path=hrs_path, error_type="reconciliation_failure",
            message="Boreas_1: 1 source-reconciliation ERROR(s): [NO_COMMON_MARKERS] ...",
            exception=RuntimeError("no common markers"),
            failure_origin="reconciliation", hrs_path=hrs_path, readable_path=readable_path,
        )
    }
    rows = build_top_issues_rows({}, failures)
    assert rows[0]["context"] == "Boreas_1_HRS_tops_no_wellname_MDRT.txt+Boreas_1_selected_well_tops.txt"
    assert "/build/" not in rows[0]["message"]


#### Step 8 — Install the package in editable mode

**Technical objective:** (re)install `p2mem` from the just-written source tree so the notebook's Python kernel imports the exact code just written.

In [ ]:
%cd /content/drive/MyDrive/Poseidon_1D_MEM
!pip install -q -e .


#### Step 9 — Run the complete unit-test suite (Increments 1.1 through 5 together)

**Technical objective:** confirm every existing test still passes and every new Increment 5 test passes, in one combined run. The actual reported pass count is read from this cell's own output — never assumed.

**Truthful completion gate (Increment 4.1.2 lesson applied from the start):** this cell runs pytest via `subprocess.run` with the ACTIVE interpreter (`sys.executable` — never a bare `pytest` that could silently resolve to a different environment), explicitly checks `returncode`, sets a boolean `FULL_TEST_SUITE_PASSED` used by the completion gate below, and raises `RuntimeError` (stopping the notebook) if the suite did not pass with zero failures. This notebook never runs pytest as a bare `!pytest` shell-escape line, which Jupyter/Colab would execute without inspecting its exit code — the exact defect independently found and corrected by Increment 4.1.2 in the prior increment's notebook.

**Failure behavior:** raises `RuntimeError` containing the actual subprocess return code if `returncode != 0`; `FULL_TEST_SUITE_PASSED` is left `False` unless this cell completes with `returncode == 0`.

In [ ]:
import subprocess
import sys

FULL_TEST_SUITE_PASSED = False  # default to NOT passed; only set True below on returncode == 0

result = subprocess.run([sys.executable, "-m", "pytest", "-v"])
if result.returncode == 0:
    FULL_TEST_SUITE_PASSED = True
    print(f"\npytest returncode = {result.returncode}: full test suite passed with zero failures.")
else:
    raise RuntimeError(
        f"pytest returncode = {result.returncode} (non-zero): the combined test suite did NOT "
        f"pass with zero failures. FULL_TEST_SUITE_PASSED remains False. Stopping here rather "
        f"than continuing past a failing test suite - see the pytest output above for the "
        f"specific failure(s)."
    )


### Real Formation-Top Integration

#### Step 10 — Load the locked survey trajectories (Poseidon 2, Boreas 1 only)

**Technical objective:** load the LOCKED Increment 3/3.1.1 `petrel_source_trace` deviation-survey trajectories for exactly the two wells with an approved formation-top file — needed for MDRT-to-TVD/TVDSS mapping. This reuses the locked `p2mem.io.deviation` loader unmodified; no survey is recomputed or replaced.

In [ ]:
from p2mem.io.deviation import load_deviation_contract_config, load_deviation_surveys

DEV_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "deviation")
DEV_TOP_FILES = {
    "Poseidon_2": "Poseidon 2_dev.txt",
    "Boreas_1": "Boreas 1_dev.txt",
}
dev_contracts = load_deviation_contract_config(os.path.join(PROJECT_ROOT, "config", "deviation_survey_contracts.yml"))
dev_paths = {k: os.path.join(DEV_DIR, fn) for k, fn in DEV_TOP_FILES.items()}
dev_results, dev_failures = load_deviation_surveys(dev_paths, dev_contracts)
print(f"Locked survey trajectories loaded: {sorted(dev_results)}  failed: {sorted(dev_failures)}")


#### Step 11 — Load the four approved formation-top files (full ingestion + reconciliation + survey mapping)

**Technical objective:** ingest all four approved formation-top files against `config/formation_top_contracts.yml`, reconciling each well's HRS-versus-readable sources and mapping reconciled MDRT through the locked survey trajectory. Poseidon North 1 and Proteus 1ST2 are recorded separately as `NOT_AVAILABLE` — never substituted.

In [ ]:
from p2mem.io.tops import load_formation_top_contract_config, load_formation_top_surveys
from p2mem.top_models import FormationTopAvailabilityRecord

hrs_contracts, readable_contracts, marker_name_aliases = load_formation_top_contract_config(
    os.path.join(PROJECT_ROOT, "config", "formation_top_contracts.yml")
)
top_paths = {
    key: (os.path.join(TOPS_DIR, hrs_fn), os.path.join(TOPS_DIR, readable_fn))
    for key, (hrs_fn, readable_fn) in TOP_FILES.items()
}
top_results, top_failures = load_formation_top_surveys(
    top_paths, hrs_contracts, readable_contracts, marker_name_aliases, dev_results
)
print(f"Formation-top wells loaded: {sorted(top_results)}  failed: {sorted(top_failures)}")

availability = {
    "Poseidon_North_1": FormationTopAvailabilityRecord(
        well_key="Poseidon_North_1", formation_top_availability="NOT_AVAILABLE",
        notes=(
            "No approved formation-top file exists for Poseidon North 1. This is a "
            "factual data gap, not an ingestion failure; no other well's formation "
            "tops are substituted, and no depth-only correlation is performed."
        ),
    ),
    "Proteus_1ST2": FormationTopAvailabilityRecord(
        well_key="Proteus_1ST2", formation_top_availability="NOT_AVAILABLE",
        notes=(
            "No approved formation-top file exists for Proteus 1ST2. This is a "
            "factual data gap, not an ingestion failure; no other well's formation "
            "tops are substituted, and no depth-only correlation is performed."
        ),
    ),
}

for key in TOP_FILES:
    r = top_results[key]
    n_mapped = sum(1 for m in r.markers if m.mapping_status == "mapped_within_coverage")
    n_rejected = sum(1 for m in r.markers if m.mapping_status == "rejected_outside_coverage")
    n_unresolved = sum(1 for m in r.markers if m.mapping_status == "not_mapped_mdrt_unresolved")
    print(f"\n{key}: n_canonical_markers={len(r.reconciliation)}  n_mapped={n_mapped}  "
          f"n_rejected_outside_coverage={n_rejected}  n_mdrt_unresolved={n_unresolved}")
    for issue in r.issues:
        print(f"  [{issue.severity}] {issue.code}")


#### Step 12 — Independently recompute both required regression findings (never hardcoded)

**Technical objective:** independently recompute, against the real approved files and the code actually run in this notebook, the two regression findings this increment's design anticipated: (1) Poseidon 2's supplied TVDSS follows a vertical-well-assumption formula (`TVDSS_source_m = MDRT_m - 21.8 m`, the well's own rotary-table elevation); (2) Boreas 1's supplied TVDSS is survey-consistent, with a maximum absolute residual below the approved 0.05 m tolerance.

**Failure behavior:** raises `AssertionError` if either finding does not reproduce as expected.

In [ ]:
p2 = top_results["Poseidon_2"]
boreas = top_results["Boreas_1"]

print("=== Regression finding 1: Poseidon 2 vertical-well-assumption defect ===")
p2_exact_defect = all(
    abs((mdrt - 21.8) - tvdss) < 1e-9
    for mdrt, tvdss in zip(p2.readable_raw.MDRT_source_m, p2.readable_raw.TVDSS_source_m)
)
print(f"Every one of {len(p2.readable_raw.TOP_NAME_source)} Poseidon 2 supplied TVDSS values "
      f"equals MDRT_M - 21.8 m EXACTLY: {p2_exact_defect}")
assert p2_exact_defect, "REGRESSION: Poseidon 2 vertical-assumption defect did not reproduce exactly"

p2_markers_by_name = {m.canonical_marker_name: m for m in p2.markers}
for name, expected_residual in [("Sea Bed", 0.0), ("Plover Fm (Top Reservoir)", 1.68), ("TD", 2.49)]:
    m = p2_markers_by_name[name]
    resid = m.TVDSS_residual_source_minus_survey_m
    print(f"  {name}: TVDSS_residual_source_minus_survey_m = {resid:.4f} m (expected ~= {expected_residual})")
    assert abs(resid - expected_residual) < 0.01, f"REGRESSION: {name} residual does not match expectation"

print("\n=== Regression finding 2: Boreas 1 survey-consistent tops ===")
boreas_residuals = [
    abs(m.TVDSS_residual_source_minus_survey_m) for m in boreas.markers
    if m.TVDSS_residual_source_minus_survey_m is not None
]
boreas_max_abs_residual = max(boreas_residuals)
print(f"Boreas 1 maximum absolute TVDSS residual across {len(boreas_residuals)} mapped markers: "
      f"{boreas_max_abs_residual:.6f} m (approved tolerance: < 0.05 m)")
assert boreas_max_abs_residual < 0.05, "REGRESSION: Boreas 1 max abs residual exceeds the approved tolerance"

boreas_follows_p2_pattern = all(
    abs((mdrt - 21.8) - tvdss) < 1e-6
    for mdrt, tvdss in zip(boreas.readable_raw.MDRT_source_m, boreas.readable_raw.TVDSS_source_m)
)
print(f"Boreas 1 follows the Poseidon-2-like MDRT-21.8 pattern (must be False): {boreas_follows_p2_pattern}")
assert not boreas_follows_p2_pattern, "REGRESSION: Boreas 1 unexpectedly follows the Poseidon 2 vertical-assumption pattern"

print("\nBoth regression findings independently confirmed.")
regression_findings_confirmed = True


### Quality-Control Results

#### Step 13 — Generate the deterministic inventory outputs

**Technical objective:** write the six deterministic CSV tables plus the JSON manifest under `outputs/05_formation_tops/` — never raw per-marker arrays beyond a single scalar per field, and never a full environment-dependent build path, only a basename.

In [ ]:
import csv
import json

from p2mem.io.tops_inventory import (
    build_formation_top_manifest,
    build_top_availability_rows,
    build_top_corrected_marker_rows,
    build_top_file_inventory_rows,
    build_top_issues_rows,
    build_top_marker_register_rows,
    build_top_reconciliation_rows,
)

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "05_formation_tops")

def write_csv(path, rows):
    fieldnames = list(rows[0].keys()) if rows else []
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(os.path.join(OUT_DIR, "top_file_inventory.csv"), build_top_file_inventory_rows(top_results, top_failures, availability))
write_csv(os.path.join(OUT_DIR, "top_marker_register.csv"), build_top_marker_register_rows(top_results))
write_csv(os.path.join(OUT_DIR, "top_hrs_vs_readable_reconciliation.csv"), build_top_reconciliation_rows(top_results))
write_csv(os.path.join(OUT_DIR, "top_survey_corrected_markers.csv"), build_top_corrected_marker_rows(top_results))
write_csv(os.path.join(OUT_DIR, "top_ingestion_issues.csv"), build_top_issues_rows(top_results, top_failures))
write_csv(os.path.join(OUT_DIR, "top_availability.csv"), build_top_availability_rows(top_results, top_failures, availability))

manifest = build_formation_top_manifest(top_results, top_failures, availability)
with open(os.path.join(OUT_DIR, "formation_top_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
    f.write("\n")

print("Outputs written to:", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, fn)
    if os.path.isfile(fp):
        print(" -", fn, os.path.getsize(fp), "bytes")


#### Step 14 — Display concise summary tables

**Technical objective:** display the formation-top file inventory and survey-corrected marker table directly in the notebook.

In [ ]:
import pandas as pd

df_inventory = pd.read_csv(os.path.join(OUT_DIR, "top_file_inventory.csv"))
display(df_inventory[["well_key", "representation_type", "source_filename", "well_identity_evidence_status",
                       "model_use_status", "n_markers", "contract_status"]])

df_corrected = pd.read_csv(os.path.join(OUT_DIR, "top_survey_corrected_markers.csv"))
display(df_corrected[["well_key", "canonical_marker_name", "MDRT_reconciled_m", "TVDSS_source_m",
                       "TVDSS_survey_corrected_m", "TVDSS_residual_source_minus_survey_m", "mapping_status"]])

df_availability = pd.read_csv(os.path.join(OUT_DIR, "top_availability.csv"))
display(df_availability)


#### Step 15 — Generate professional QC figures

**Technical objective:** produce three portfolio-quality QC figures (Poseidon 2 TVDSS residual by marker, Boreas 1 equivalent, and a Poseidon 2 – Boreas 1 marker-depth panel), each with units, legends, well names, evidence status, and a Tier C footer. The third figure is explicitly labelled as a marker-depth comparison only — never a geological correlation or lithology interpretation.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path as _Path

FIG_DIR = _Path(OUT_DIR) / "figures"
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

# fig01: Poseidon 2 supplied-vs-survey-corrected TVDSS residual by marker/depth
fig, ax = plt.subplots(figsize=(8, 6))
mapped_p2 = [m for m in p2.markers if m.mapping_status == "mapped_within_coverage"]
mapped_p2 = sorted(mapped_p2, key=lambda m: m.MDRT_reconciled_m)
names_p2 = [m.canonical_marker_name for m in mapped_p2]
residuals_p2 = [m.TVDSS_residual_source_minus_survey_m for m in mapped_p2]
depths_p2 = [m.TVDSS_survey_corrected_m for m in mapped_p2]
ax.plot(residuals_p2, depths_p2, "o-", ms=6, color="tab:blue", label="TVDSS_residual_source_minus_survey_m")
for name, resid, depth in zip(names_p2, residuals_p2, depths_p2):
    ax.annotate(name, (resid, depth), textcoords="offset points", xytext=(6, 0), fontsize=7)
ax.axvline(0.0, color="k", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("TVDSS_residual_source_minus_survey_m = TVDSS_source_m - TVDSS_survey_corrected_m")
ax.set_ylabel("TVDSS_survey_corrected_m (m, survey-corrected depth basis)")
ax.set_title(
    "Fig 1. Poseidon 2: supplied vs. survey-corrected TVDSS residual by marker\n"
    "Well identity evidence: inferred_unverified — vertical-well-assumption defect "
    "(TVDSS_source = MDRT - 21.8 m)", fontsize=9,
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER} — depth comparison only, not a lithology/geological interpretation",
          ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(FIG_DIR / "fig01_poseidon2_tvdss_residual_by_marker.png", dpi=150)
plt.close(fig)

# fig02: Boreas 1 equivalent residual comparison
fig, ax = plt.subplots(figsize=(8, 6))
mapped_b1 = [m for m in boreas.markers if m.mapping_status == "mapped_within_coverage"]
mapped_b1 = sorted(mapped_b1, key=lambda m: m.MDRT_reconciled_m)
names_b1 = [m.canonical_marker_name for m in mapped_b1]
residuals_b1 = [m.TVDSS_residual_source_minus_survey_m for m in mapped_b1]
depths_b1 = [m.TVDSS_survey_corrected_m for m in mapped_b1]
ax.plot(residuals_b1, depths_b1, "o-", ms=6, color="tab:orange", label="TVDSS_residual_source_minus_survey_m")
for name, resid, depth in zip(names_b1, residuals_b1, depths_b1):
    ax.annotate(name, (resid, depth), textcoords="offset points", xytext=(6, 0), fontsize=7)
ax.axvline(0.0, color="k", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("TVDSS_residual_source_minus_survey_m = TVDSS_source_m - TVDSS_survey_corrected_m")
ax.set_ylabel("TVDSS_survey_corrected_m (m, survey-corrected depth basis)")
ax.set_title(
    f"Fig 2. Boreas 1: supplied vs. survey-corrected TVDSS residual by marker\n"
    f"Identity evidence: inferred_unverified — max|residual| = {boreas_max_abs_residual:.4f} m "
    f"(< 0.05 m tolerance,\nsurvey-consistent, NOT corrected)", fontsize=9,
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER} — depth comparison only, not a lithology/geological interpretation",
          ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(FIG_DIR / "fig02_boreas1_tvdss_residual_by_marker.png", dpi=150)
plt.close(fig)

# fig03: Poseidon 2 - Boreas 1 stratigraphic-marker panel using survey-corrected
# TVDSS - explicitly a marker-DEPTH comparison, never a geological correlation.
fig, axes = plt.subplots(1, 2, figsize=(10, 8), sharey=False)
axes[0].scatter([0] * len(depths_p2), depths_p2, color="tab:blue", s=40, zorder=3)
for name, depth in zip(names_p2, depths_p2):
    axes[0].annotate(f"{name}  ({depth:.1f} m)", (0, depth), textcoords="offset points",
                      xytext=(8, 0), fontsize=7, va="center")
axes[0].set_xlim(-0.5, 3.0)
axes[0].set_xticks([])
axes[0].invert_yaxis()
axes[0].set_ylabel("TVDSS_survey_corrected_m (m)")
axes[0].set_title("Poseidon 2", fontsize=10)
axes[0].grid(axis="y", alpha=0.3)

axes[1].scatter([0] * len(depths_b1), depths_b1, color="tab:orange", s=40, zorder=3)
for name, depth in zip(names_b1, depths_b1):
    axes[1].annotate(f"{name}  ({depth:.1f} m)", (0, depth), textcoords="offset points",
                      xytext=(8, 0), fontsize=7, va="center")
axes[1].set_xlim(-0.5, 3.0)
axes[1].set_xticks([])
axes[1].invert_yaxis()
axes[1].set_title("Boreas 1", fontsize=10)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle(
    "Fig 3. Poseidon 2 - Boreas 1 marker-DEPTH comparison panel (survey-corrected TVDSS)\n"
    "NOTE: marker-depth comparison only - NOT a geological correlation or lithology\n"
    "interpretation. No marker is inferred to be genetically related across wells.",
    fontsize=9,
)
fig.text(0.5, 0.01, f"{ASSURANCE_TIER} — both wells' identity evidence: inferred_unverified",
          ha="center", fontsize=7)
fig.tight_layout(rect=[0, 0.03, 1, 0.90])
fig.savefig(FIG_DIR / "fig03_poseidon2_boreas1_marker_depth_panel.png", dpi=150)
plt.close(fig)


#### Step 16 — Integration regression checks

**Technical objective:** independently re-verify, against the code and data actually run in this notebook, every regression this increment was designed against — never assumed, always recomputed here.

In [ ]:
print("=== INTEGRATION REGRESSION CHECKS (Increment 5) ===")

EXPECTED_MARKER_COUNTS = {"Poseidon_2": 9, "Boreas_1": 10}
for key, expected in EXPECTED_MARKER_COUNTS.items():
    actual = len(top_results[key].reconciliation)
    print(f"{key}: {actual} canonical markers (expected {expected}) -> {'MATCH' if actual == expected else 'MISMATCH'}")

print(f"Contracts passed: {sum(1 for k in TOP_FILES if top_results[k].contract_status == 'PASSED')}/2")
print(f"Poseidon 2 markers mapped (must be 9, 0 rejected/unresolved): "
      f"{sum(1 for m in p2.markers if m.mapping_status == 'mapped_within_coverage')}")
print(f"Boreas 1 markers mapped (must be 10, 0 rejected/unresolved): "
      f"{sum(1 for m in boreas.markers if m.mapping_status == 'mapped_within_coverage')}")
print(f"Poseidon North 1 / Proteus 1ST2 formation_top_availability: "
      f"{availability['Poseidon_North_1'].formation_top_availability} / "
      f"{availability['Proteus_1ST2'].formation_top_availability}")
print(f"Regression findings independently confirmed: {regression_findings_confirmed}")

_expected_outputs = [
    "top_file_inventory.csv", "top_marker_register.csv", "top_hrs_vs_readable_reconciliation.csv",
    "top_survey_corrected_markers.csv", "top_ingestion_issues.csv", "top_availability.csv",
    "formation_top_manifest.json",
]
print("All 7 deterministic outputs present:", all(os.path.exists(os.path.join(OUT_DIR, fn)) for fn in _expected_outputs))

_expected_figures = [
    "fig01_poseidon2_tvdss_residual_by_marker.png", "fig02_boreas1_tvdss_residual_by_marker.png",
    "fig03_poseidon2_boreas1_marker_depth_panel.png",
]
print("All 3 QC figures present:", all(os.path.exists(os.path.join(FIG_DIR, fn)) for fn in _expected_figures))


### Interpretation

Poseidon 2's "selected readable" formation-top file was independently confirmed to have been generated under a vertical-well assumption: every one of its 9 supplied TVDSS values equals its MDRT minus 21.8 m (the well's own rotary-table elevation) EXACTLY, and the survey-corrected residual grows from ~0 m at Sea Bed to +2.49 m at TD — a real depth-reference defect, corrected here in `TVDSS_survey_corrected_m` while `TVDSS_source_m` itself is preserved unmodified for audit. Boreas 1 shows no such pattern: its maximum absolute source-versus-survey residual (0.0416 m, across 10 markers) is below the approved 0.05 m tolerance and does not follow the Poseidon-2-like formula, supporting the interpretation that it was generated from the well's real surveyed trajectory — it is therefore NOT "corrected" the way Poseidon 2 is. This evidence is applied strictly per file, per well; Boreas 1's apparent consistency was never assumed from Poseidon 2's defect, or vice versa.

### Limitations

- This increment's formation-top ingestion is, like the rest of this project, screening-level and uncalibrated — no independent seismic or well-tie calibration exists to adjudicate which of a well's two source representations is closer to true geological reality where they might disagree (none of the 19 reconciled markers across both wells showed an MDRT disagreement beyond tolerance in the real data, but the mechanism exists and is tested).
- Both formation-top file representations for both wells are `well_identity_evidence_status: inferred_unverified` — the HRS files carry no well name in their body at all (filename-only association), and the readable files' well name appears only in a project-supplied `#` comment, not independently verified file content. This is a deliberately MORE conservative classification than Increment 4's checkshot files.
- Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as a factual data gap (`formation_top_availability: NOT_AVAILABLE`) — never substituted with another well's tops or correlated by depth alone.
- The Poseidon 2 – Boreas 1 marker-depth panel (Fig 3) is a depth comparison only. No genetic, stratigraphic, or lithological correlation between the two wells' markers is asserted or implied by this increment.
- Downstream phases (petrophysics, pore pressure, elastic properties, rock strength, stresses, wellbore stability) all remain explicitly out of scope and unimplemented.

### Completion Gate

> **PHASE COMPLETION GATE:** Increment 5 / 5.1 / 5.1.1 (Formation-Top Ingestion, Source Reconciliation, and Survey-Corrected Stratigraphic Depth Framework, as corrected by the Increment 5.1 and Increment 5.1.1 narrowly scoped corrective patches) is complete when: (0) the complete combined test suite (Step 9) passed with ZERO failures, checked by actual subprocess return code, not assumed or left uninspected; (1) all locked Increment 1–4.1.2 tests and all Increment 5/5.1/5.1.1 tests pass in the combined suite; (2) all four approved formation-top files load with zero contract-resolution ERRORs; (3) HRS-versus-readable reconciliation is complete for both wells with zero silently-resolved MDRT disagreements, and all zero-common-canonical-marker configurations (Increment 5.1.1, Finding 1) are fatal; (4) every raw marker name and raw source depth is preserved, with every corrected depth under a separately named field; (5) zero marker is extrapolated; (6) the Poseidon 2 vertical-assumption defect and the Boreas 1 survey-consistency finding are independently reproduced (not hardcoded) and reported; (7) Poseidon North 1 and Proteus 1ST2 are recorded as `NOT_AVAILABLE`; (8) all 7 deterministic outputs and 3 QC figures are generated. All conditions are verified programmatically below, not asserted.

In [ ]:
print("=" * 78)
print("INCREMENT 5 / 5.1 / 5.1.1 COMPLETION GATE")
print("=" * 78)

gate_checks = {
    # Read via globals().get(..., False), NOT a bare FULL_TEST_SUITE_PASSED
    # name reference, so that if Step 9 were ever skipped entirely (the
    # variable never defined at all) this check fails CLEANLY as False -
    # printed as an ordinary [FAIL] line below - rather than crashing the
    # whole gate cell with an uncaught NameError. A clean [FAIL] here
    # cannot result in this gate printing overall completion (`is True`
    # only matches the literal boolean True, set only by Step 9 on a
    # confirmed returncode == 0).
    "Complete combined test suite passed with zero failures": (
        globals().get("FULL_TEST_SUITE_PASSED", False) is True
    ),
    "All 2 formation-top wells loaded (0 failures)": len(top_failures) == 0,
    "Both contracts PASSED": all(top_results[k].contract_status == "PASSED" for k in TOP_FILES),
    "Poseidon 2: 9 markers, all mapped, 0 rejected/unresolved": (
        len(p2.reconciliation) == 9
        and sum(1 for m in p2.markers if m.mapping_status == "mapped_within_coverage") == 9
    ),
    "Boreas 1: 10 markers, all mapped, 0 rejected/unresolved": (
        len(boreas.reconciliation) == 10
        and sum(1 for m in boreas.markers if m.mapping_status == "mapped_within_coverage") == 10
    ),
    "Poseidon North 1 and Proteus 1ST2 recorded as NOT_AVAILABLE": (
        availability["Poseidon_North_1"].formation_top_availability == "NOT_AVAILABLE"
        and availability["Proteus_1ST2"].formation_top_availability == "NOT_AVAILABLE"
    ),
    "Both regression findings independently confirmed": (
        globals().get("regression_findings_confirmed", False) is True
    ),
    "7 deterministic output files present": all(
        os.path.exists(os.path.join(OUT_DIR, fn)) for fn in _expected_outputs
    ),
    "3 QC figures present": all(
        os.path.exists(os.path.join(FIG_DIR, fn)) for fn in _expected_figures
    ),
}
for label, passed in gate_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")

if all(gate_checks.values()):
    print("\nIncrement 5.1.1 (further narrowly scoped corrective patch to Increment 5.1, itself a")
    print("corrective patch to Increment 5) is complete. Stopping here per the approved scope.")
    print("Not implemented (explicitly out of scope): gamma-ray normalization, shale-volume")
    print("calculation, named lithology classification, petrophysical interpretation, method-")
    print("eligibility masks, shallow-density modelling, overburden-stress integration, NCT")
    print("fitting, pore-pressure prediction, elastic properties, rock strength, horizontal")
    print("stresses, wellbore-stability analysis. Increment 6 has NOT been started.")
else:
    raise RuntimeError("Increment 5.1.1 completion gate FAILED - see failed check(s) above.")
